# What Actually Limits a Small-Model SWE Agent? Reproduction Notebook

Companion to the paper-track writeup on the Gemma 4 Developer Agent harness. It recomputes the paper's tables and Figure 1 from the competition files and from one published table of run metrics.

Writeup: https://www.kaggle.com/competitions/gemma-4-developer-agent-paper/writeups/what-actually-limits-a-small-model-swe-agent-a-ge. Code and reports: https://github.com/bsw0610/gemma4-swe-harness-audit.

**TL;DR**
- **C1.** In `swegemma` 0.2.7 the documented short-name resolver did not exist. Release 0.2.11 added it, and reach from the issue text rose, but BM25 still leads every shipped graph policy at k = 50. The tools search the whole graph and BM25 ranks spans outside tests and docs, so this compares the tools as shipped, not ranking methods on one corpus.
- **C2.** The offline skill puts an in-graph gold symbol in its default top-15 list for about half of the tasks. Two undocumented harness rules break skills until they are handled.
- **C3.** With `max_output_tokens` 16,384, about one run in five ends in a context-window error. Halving it removes these errors, but the gain in resolved tasks is small and cannot be told apart from run-to-run variation.


## Definitions and boundaries

| Item | Choice | Why |
|---|---|---|
| Harness versions | C1 audit, C2 and C3 runs on `swegemma` 0.2.7; Table 1 also on 0.2.11 | 0.2.7 was current when we began; 0.2.11 was published on 2026-10-12 |
| Gold symbols | Innermost `def` or `class` enclosing each changed base-commit line, by `ast` | A text-matching heuristic agrees on too few hunks (shown below) |
| Reach sample | Tasks with at least one gold symbol in the graph | The graph tools cannot reach a symbol that is not in the graph |
| Candidate list | Resolved seeds first, then round-robin over tool results, at most k entries | Crediting seeds favours the tools |
| Trace events | A file read or edit counts only if the tool call succeeded | Failed calls are not access |
| C3 tasks | Positions 6-125 of a seed-0 random order of the 125 tasks with an existing gold code file | The first 5 were the pilot and informed the design |
| C3 arms | `base`: reference agent without LoRA adapter and sub-agent, `max_output_tokens` 16,384. `out8k`: the same with 8,192 | One value changed |
| C3 budgets | 100 tool calls, 300 s per command, 500 turns, 30 minutes, compaction interval 5 | Interval as in the README at the time; the current README lists 15 |
| Statistics | Unit = task; 95% CI = task bootstrap, 10,000 draws, seed 0; McNemar exact for paired binary outcomes | Tasks are the sampling unit |

**What this notebook does not recompute.** The C3 GPU runs (one A100 in Colab, 320 runs): their metrics are read from `results/c3_run_rows.csv`, embedded below, which holds no issue text, code or patches. Also the pilot's counts (0 of 8 runs called the skill; traces are not published, see `analysis/c2_pilot_summary.py`), the skill's per-task run time, and the mock-model test of the overflow path, which needs the harness and ADK wheels (`analysis/test_overflow_salvage.py` in the repository).


In [ ]:
# Locate the inputs. On Kaggle: the competition files and the wheelhouse dataset. Locally: data/raw (read-only) plus cached
# kernel outputs under work/, because the snapshots are only on Kaggle.
import hashlib, json, os, re, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
ON_KAGGLE = KAGGLE_INPUT.exists()
if ON_KAGGLE:
    INPUT = sorted(KAGGLE_INPUT.glob("**/gemma-4-developer-agent/tasks.jsonl"))[0].parent
    WHEEL = sorted(KAGGLE_INPUT.glob("**/swegemma-0.2.11-py3-none-any.whl"))[0]
    PROJECT = None
else:
    PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
    INPUT = PROJECT / "data/raw"
    WHEEL = PROJECT / "work/harness_0_2_11/wheels/swegemma-0.2.11-py3-none-any.whl"

# Intermediate files stay outside /kaggle/working; aggregate results remain in cell outputs.
W = str(Path(tempfile.gettempdir()) / "g4swe")
WORK = Path(W)
shutil.rmtree(WORK, ignore_errors=True)
for d in ("analysis", "kaggle/gold_ast", "kaggle/bm25", "kaggle/gold_mutation", "kaggle/skill_eval", "results", "reports", "data", "work/c1", "work/c2"):
    (WORK / d).mkdir(parents=True)
(WORK / "data/raw").symlink_to(INPUT, target_is_directory=True)
FINAL = Path("/kaggle/working") if ON_KAGGLE else WORK / "final"
FINAL.mkdir(exist_ok=True)
print("environment:", "Kaggle" if ON_KAGGLE else "local dry run with cached kernel outputs")
print("graphs:", len(list(INPUT.glob("graphs/*.json"))), "embeddings:", len(list(INPUT.glob("embeddings/*.npz"))))


def run(cmd, tail=None, drop=None, ret=False):
    """Run a script from WORK; print its stdout (last `tail` lines), without lines matching `drop` (timings)."""
    p = subprocess.run([sys.executable, *cmd], cwd=WORK, capture_output=True, text=True)
    if p.returncode:
        print(p.stdout[-3000:], p.stderr[-3000:])
        raise RuntimeError(f"{cmd} exited with {p.returncode}")
    lines = [ln for ln in p.stdout.splitlines() if not (drop and re.search(drop, ln))]
    print("\n".join(lines if tail is None else lines[-tail:] if tail else []))
    return p.stdout if ret else None


def kernel(script, rel_dir, files, tail=1, drop=None):
    """Run a snapshot script on Kaggle and move its outputs out of /kaggle/working; locally copy the cached outputs."""
    (WORK / rel_dir).mkdir(parents=True, exist_ok=True)
    if ON_KAGGLE:
        run([f"kaggle/{script}"], tail=tail, drop=drop)
    for f in files:
        src = Path("/kaggle/working") / f if ON_KAGGLE else PROJECT / rel_dir / f
        (shutil.move if ON_KAGGLE else shutil.copy)(str(src), str(WORK / rel_dir / f))


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


CHECKS = []


def check(name, got, want):
    """Stop the notebook if a key count differs from its expected value."""
    assert got == want, f"{name}: got {got!r}, expected {want!r}"
    CHECKS.append(name)
    print(f"ok  {name} = {got}")

## Scripts

The next cells write the repository's scripts and the C3 table to a scratch folder outside `/kaggle/working`. Their inputs are collapsed; the digests after them identify each file.


In [ ]:
%%writefile {W}/analysis/gold_core.py
"""Gold-location core shared by the Kaggle kernel (kaggle/gold_ast) and local regression tests.

attribute(patch_hunks, src) maps every changed old-file line of a code file to the innermost
enclosing def/class of the base snapshot (ast).

Rules (v2, after Codex audit A01/A11):
- A hunk is split into change blocks: maximal runs of '-'/'+' lines between context lines.
- Replacement block (has '-' lines): attributed to the scopes of its removed old lines only.
  Its '+' lines are NOT treated as insertions (A01 fix).
- Pure insertion block (only '+'), inserted between old lines a and a+1 (v3, audit 2 N01/N02):
  * decorator-only block (only '@' lines and continuation, no def/class) placed right before an existing scope T
    whose header column equals the decorator indent -> owner = T (the existing definition gains a decorator);
  * otherwise owner = innermost scope S that lexically encloses the gap and whose body column the new code sits in:
    S.start <= a' <= S.end, where a' = last non-blank old line <= a (trailing blank lines after a scope still belong to it),
    and indent(first non-blank '+' line) > S.col. A new method -> its class; a new top-level def/class -> '<module>'.
- '\\ No newline at end of file' lines are ignored; '+++ /dev/null' (deleted files) are skipped.
- Whitespace-only blocks (every removed old line and every added line is blank) are formatting, not gold: skipped,
  unless they lie inside a multi-line string literal (tokenize), where blank lines are part of the value (N06).
"""
import ast
import io
import re
import tokenize

HUNK_RE = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")
TEST_RE = re.compile(r"(^|/)tests?/|(^|/)test_[^/]*$|_test\.py$")


def code_file(path):
    return path.endswith(".py") and not TEST_RE.search(path) and not path.startswith(("docs/", "docs_src/"))


def parse_patch(patch):
    """{path: [hunk]}; hunk = {'old_start', 'header', 'blocks': [{'removed': [old line nos], 'added': [text],
    'after': old line no. preceding the block}]}"""
    files, cur, hunk, old, block = {}, None, None, 0, None
    for line in patch.splitlines():
        if line.startswith("--- "):
            continue
        if line.startswith("+++ "):
            p = line[4:].strip()
            cur = None if p == "/dev/null" else (p[2:] if p.startswith("b/") else p)
            if cur is not None:
                files.setdefault(cur, [])
            hunk = None
            continue
        if cur is None:
            continue
        m = HUNK_RE.match(line)
        if m:
            old = int(m.group(1)) if (m.group(2) is None or m.group(2) != "0") else int(m.group(1)) + 1
            hunk = {"old_start": int(m.group(1)), "header": line, "blocks": []}
            files[cur].append(hunk)
            block = None
            continue
        if hunk is None or line.startswith("\\"):
            continue
        if line.startswith("-") or line.startswith("+"):
            if block is None:
                block = {"removed": [], "added": [], "after": old - 1}
                hunk["blocks"].append(block)
            if line.startswith("-"):
                block["removed"].append(old)
                old += 1
            else:
                block["added"].append(line[1:])
        else:
            block = None
            old += 1
    return files


def scopes(src):
    """[(start, end, qualname, kind, col)] for every def/class; start includes decorators."""
    tree = ast.parse(src)
    out = []

    def walk(node, prefix):
        for ch in ast.iter_child_nodes(node):
            if isinstance(ch, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                q = f"{prefix}.{ch.name}" if prefix else ch.name
                start = min([ch.lineno] + [d.lineno for d in ch.decorator_list])
                kind = {ast.FunctionDef: "def", ast.AsyncFunctionDef: "async_def", ast.ClassDef: "class"}[type(ch)]
                out.append((start, ch.end_lineno, q, kind, ch.col_offset))
                walk(ch, q)
            else:
                walk(ch, prefix)
    walk(tree, "")
    return out


def innermost(sc, line):
    best = None
    for s in sc:
        if s[0] <= line <= s[1] and (best is None or (s[1] - s[0]) < (best[1] - best[0])):
            best = s
    return best


def _indent(text):
    return len(text) - len(text.lstrip(" \t"))


DEF_RE = re.compile(r"\s*(async\s+def|def|class)\b")


def insertion_owner(sc, a, added, lines):
    nonblank = [t for t in added if t.strip()]
    if not nonblank:
        return None
    ind = _indent(nonblank[0])
    top = [t for t in nonblank if _indent(t) == ind]
    decorator_only = top[0].lstrip().startswith("@") and not any(DEF_RE.match(t) for t in top)
    if decorator_only:
        nxt = [x for x in sc if x[0] == a + 1 and x[4] == ind]
        if nxt:
            return min(nxt, key=lambda x: x[1] - x[0])
    a2 = a
    while a2 > 0 and a2 <= len(lines) and not lines[a2 - 1].strip():
        a2 -= 1
    cands = [x for x in sc if x[0] <= a2 <= x[1] and ind > x[4]]
    return min(cands, key=lambda x: x[1] - x[0]) if cands else None


def string_continuation_lines(src):
    """Old line numbers that lie inside a multi-line string literal (after its first line)."""
    out = set()
    try:
        for tok in tokenize.generate_tokens(io.StringIO(src).readline):
            if tok.type == tokenize.STRING and tok.end[0] > tok.start[0]:
                out.update(range(tok.start[0] + 1, tok.end[0] + 1))
    except (tokenize.TokenError, IndentationError, SyntaxError):
        pass
    return out


def attribute(hunks, src):
    """Yield (hunk_index, qualname | '<module>', kind, line, block_type)."""
    sc = scopes(src)
    lines = src.splitlines()
    in_string = string_continuation_lines(src)
    for hi, h in enumerate(hunks):
        seen = set()
        for b in h["blocks"]:
            removed_text = [lines[ln - 1] if 0 < ln <= len(lines) else "x" for ln in b["removed"]]
            if all(not t.strip() for t in removed_text + b["added"]):
                inside = any(ln in in_string for ln in b["removed"]) or (not b["removed"] and (b["after"] + 1) in in_string)
                if not inside:
                    continue
                s_ = innermost(sc, b["removed"][0] if b["removed"] else b["after"] + 1)
                key = s_[2] if s_ else "<module>"
                if key not in seen:
                    seen.add(key)
                    yield hi, key, (s_[3] if s_ else "module"), (b["removed"] or [b["after"]])[0], "string_ws"
                continue
            if b["removed"]:
                for ln in b["removed"]:
                    s = innermost(sc, ln)
                    key = s[2] if s else "<module>"
                    if key not in seen:
                        seen.add(key)
                        yield hi, key, (s[3] if s else "module"), ln, "replace"
            else:
                s = insertion_owner(sc, b["after"], b["added"], lines)
                key = s[2] if s else "<module>"
                if key not in seen:
                    seen.add(key)
                    yield hi, key, (s[3] if s else "module"), b["after"], "insert"


In [ ]:
%%writefile {W}/analysis/c1_report.py
"""Regenerate every number in reports/c1_results.md §3-§4 from raw data + Kaggle kernel outputs.

Inputs (read-only):
  data/raw/tasks.jsonl, data/raw/graphs/*.json
  work/c1/gold_ast_v4/gold_ast.csv      (kaggle/gold_ast, kernel version 4)
  work/c1/bm25_v3/bm25_ranks.csv        (kaggle/bm25, kernel version 3)
  work/c1/gold_coverage/gold.csv        (deprecated text-matching heuristic; only for the agreement check)
Outputs: work/c1/report/*.csv and stdout.

Statistics: unit = task; 95% CI = percentile interval of a task-resampling bootstrap (10,000 draws, seed 0,
tasks sorted by id). The CI is conditional on this task set (4 repositories, fastapi + rich = 89%);
it does not cover repository-to-repository variation. Leave-one-repo-out rows are reported for that reason.
Run: uv run python analysis/c1_report.py
"""
import collections
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
RAW, WORK = ROOT / "data/raw", ROOT / "work/c1"
OUT = WORK / "report"
OUT.mkdir(parents=True, exist_ok=True)
GOLD_CSV = WORK / "gold_ast_v4/gold_ast.csv"
BM25_CSV = WORK / "bm25_v3/bm25_ranks.csv"
KERNEL_VERSIONS = {"bsw0610/g4swe-gold-ast": 4, "bsw0610/g4swe-bm25": 3}
PKG = {"fastapi/fastapi": "fastapi", "Textualize/rich": "rich", "psf/requests": "requests", "encode/httpx": "httpx"}
KS = (10, 25, 50, 100)
B, SEED = 10_000, 0

TASKS = {json.loads(l)["instance_id"]: json.loads(l) for l in open(RAW / "tasks.jsonl")}


def module_of(path):
    p = path[4:] if path.startswith("src/") else path
    m = p[:-3].replace("/", ".")
    return m[: -len(".__init__")] if m.endswith(".__init__") else m


_graph_cache = {}


def graph(tid):
    t = TASKS[tid]
    f = RAW / "graphs" / f"{t['repo'].split('/')[1].lower()}_{t['base_commit']}.json"
    if f not in _graph_cache:
        g = json.load(open(f))
        adj = collections.defaultdict(set)
        for e in g["edges"]:
            if e["source"] != e["target"]:
                adj[e["source"]].add(e["target"])
                adj[e["target"]].add(e["source"])
        _graph_cache.clear()  # keep one graph in memory
        _graph_cache[f] = ({n["id"]: (n.get("text") or "") for n in g["nodes"]}, adj)
    return _graph_cache[f]


def boot(x, idx):
    x = np.asarray(x, float)
    bs = x[idx].mean(axis=1)
    return x.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)


def fmt(x, idx):
    m, lo, hi = boot(x, idx)
    return f"{int(round(m * len(x)))}/{len(x)} = {m * 100:.1f}% [{lo * 100:.1f}, {hi * 100:.1f}]"


def fmt_diff(d, idx):
    m, lo, hi = boot(d, idx)
    return f"{m * 100:+.1f} pp [{lo * 100:+.1f}, {hi * 100:+.1f}]"


# ---------------------------------------------------------------- §3 gold and coverage
g = pd.read_csv(GOLD_CSV)
g["node"] = [f"{module_of(f)}.{q}" if q != "<module>" else None for f, q in zip(g.file, g.qualname)]
g["in_graph"] = [bool(n in graph(t)[0]) if n else None for n, t in zip(g.node, g.instance_id)]
g.to_csv(OUT / "gold_mapped.csv", index=False)
sym = g[g.kind != "module"].drop_duplicates(["instance_id", "node"])
print("## §3 Gold (AST, kernel v4)")
print(f"tasks 129 -> with symbol gold {sym.instance_id.nunique()} -> with in-graph gold {sym[sym.in_graph == True].instance_id.nunique()}")
print(f"unique (task, symbol): {len(sym)}; in graph {int(sym.in_graph.sum())} ({sym.in_graph.mean() * 100:.1f}%)")
print(sym.groupby("kind").in_graph.agg(n="size", in_graph="sum").to_string())
miss_tasks = sym.groupby("instance_id").in_graph.apply(lambda s: (~s.astype(bool)).any())
print(f"tasks with >=1 gold symbol missing from graph: {int(miss_tasks.sum())}/{len(miss_tasks)}")
print("block types:", g.block_type.value_counts().to_dict())

# heuristic vs AST agreement (A05): unit = (task, file, hunk) with >=1 in-graph symbol under EITHER method
h = pd.read_csv(WORK / "gold_coverage/gold.csv")
h = h[(h.level == "symbol") & (h.in_graph == True)]
H = h.groupby(["instance_id", "file", "hunk_index"]).symbol.agg(set)
# hunk-level sets come from the undeduplicated rows (audit 2 N03): dedup only within (task, file, hunk, node)
gh = g[(g.kind != "module") & (g.in_graph == True)].drop_duplicates(["instance_id", "file", "hunk_index", "node"])
A = gh.groupby(["instance_id", "file", "hunk_index"]).node.agg(set)
keys = H.index.union(A.index)
exact = sum(H.get(k, set()) == A.get(k, set()) for k in keys)
overlap = sum(bool(H.get(k, set()) & A.get(k, set())) for k in keys)
pd.DataFrame([dict(instance_id=k[0], file=k[1], hunk_index=k[2], heuristic=sorted(H.get(k, set())), ast=sorted(A.get(k, set())))
              for k in keys]).to_csv(OUT / "hunk_agreement.csv", index=False)
print(f"heuristic vs AST, hunks with an in-graph symbol under either method: {len(keys)}; "
      f"exact {exact} ({exact / len(keys) * 100:.1f}%), any overlap {overlap} ({overlap / len(keys) * 100:.1f}%)")

# ---------------------------------------------------------------- §4 reach: BM25
gold = sym[sym.in_graph == True].groupby("instance_id").node.agg(set)
tasks = sorted(gold.index)
n = len(tasks)
rng = np.random.default_rng(SEED)
IDX = rng.integers(0, n, size=(B, n))
repo_of = {t: TASKS[t]["repo"] for t in tasks}
gold_files = g[g.kind != "module"].groupby("instance_id").file.agg(set)

b = pd.read_csv(BM25_CSV)
bs = b[b.granularity == "symbol"].sort_values(["instance_id", "rank"])
bl = b[b.granularity == "symbol_lib"].sort_values(["instance_id", "rank"])
bf = b[b.granularity == "file"].sort_values(["instance_id", "rank"])
rank_sym = bs.groupby("instance_id").unit.agg(list)
rank_file = bf.groupby("instance_id").file.agg(list)
rank_lib = bl.groupby("instance_id").unit.agg(list)
hits = {}
for k in KS:
    hits[("bm25_snapshot", k)] = [bool(set(rank_sym.get(t, [])[:k]) & gold[t]) for t in tasks]
    hits[("bm25_file", k)] = [bool(set(rank_file.get(t, [])[:k]) & gold_files.get(t, set())) for t in tasks]

# graph-text BM25 on library-package nodes only (A04): corpus = graph nodes whose id starts with '<pkg>.'
TOK = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
STOP = set("the a an and or of to in for is are be this that with on as it by not if from at we can should when use using".split())


def toks(s):
    out = []
    for w in TOK.findall(s):
        out += [p.lower() for p in [w] + [p for p in re.split(r"_|(?<=[a-z])(?=[A-Z])", w) if p] if len(p) > 1]
    return [x for x in out if x not in STOP]


def bm25_order(query, ids, texts, k1=1.2, bb=0.75):
    tf = [collections.Counter(toks(x)) for x in texts]
    N = len(tf)
    avg = sum(sum(c.values()) for c in tf) / N
    df = collections.Counter(w for c in tf for w in c)
    q = set(toks(query))
    sc = []
    for c in tf:
        dl = sum(c.values())
        sc.append(sum(math.log(1 + (N - df[w] + .5) / (df[w] + .5)) * c[w] * (k1 + 1) / (c[w] + k1 * (1 - bb + bb * dl / avg))
                      for w in q if w in c))
    return [ids[i] for i in sorted(range(N), key=lambda i: (-sc[i], ids[i]))]


graph_rank = {}
for t in tasks:
    texts, _ = graph(t)
    pkg = PKG[repo_of[t]] + "."
    ids = sorted(i for i in texts if i.startswith(pkg))
    graph_rank[t] = bm25_order(TASKS[t]["problem_statement"], ids, [texts[i] for i in ids])
for k in KS:
    hits[("bm25_graph_library", k)] = [bool(set(graph_rank[t][:k]) & gold[t]) for t in tasks]

# ---------------------------------------------------------------- gate (A07): equal unique-id budget, package filter
def expand(t, R, m, k, order):
    _, adj = graph(t)
    pkg = PKG[repo_of[t]] + "."
    cand = list(dict.fromkeys(R[:m]))
    seen = set(cand)
    nbrs = [sorted(v for v in adj.get(s, ()) if v.startswith(pkg)) for s in R[:m]]
    if order == "sequential":
        stream = [v for lst in nbrs for v in lst]
    else:  # round-robin over seeds
        stream = [lst[i] for i in range(max((len(x) for x in nbrs), default=0)) for lst in nbrs if i < len(lst)]
    for v in stream:
        if v not in seen:
            cand.append(v)
            seen.add(v)
    for u in R[m:]:
        if len(cand) >= k:
            break
        if u not in seen:
            cand.append(u)
            seen.add(u)
    return cand[:k]


gate_rows = []
for t in tasks:
    R = list(dict.fromkeys(rank_lib.get(t, [])))
    assert all(u.startswith(PKG[repo_of[t]] + ".") for u in R)
    for k in (25, 50, 100):
        assert len(R) >= k, (t, k, len(R))  # equal budget needs k library units (audit 2 N04)
        base = bool(set(R[:k]) & gold[t])
        for m in (5, 10, 25):
            if m >= k:
                continue
            for order in ("sequential", "round_robin"):
                cand = expand(t, R, m, k, order)
                assert len(cand) == k == len(set(cand)), (t, k, m, order, len(cand))
                gate_rows.append(dict(task=t, k=k, m=m, order=order, base=base, hit=bool(set(cand) & gold[t])))
gate = pd.DataFrame(gate_rows)
gate.to_csv(OUT / "gate.csv", index=False)

# ---------------------------------------------------------------- print §4
print("\n## §4 Reach (n = %d tasks with in-graph gold; repo counts %s)" % (n, dict(collections.Counter(repo_of.values()))))
for name in ("bm25_snapshot", "bm25_graph_library", "bm25_file"):
    print(name, " | ".join(f"@{k} {fmt(hits[(name, k)], IDX)}" for k in KS))
print("\nGate: BM25 (library units ranked by the kernel after filtering, unique ids, every list exactly k) top-m seeds + 1-hop calls neighbours (library only), filled to k with BM25")
for (k, m, order), d in gate.groupby(["k", "m", "order"]):
    d = d.set_index("task").reindex(tasks)
    diff = d.hit.astype(float).values - d.base.astype(float).values
    print(f"k={k:3d} m={m:2d} {order:11s} base {d.base.mean() * 100:5.1f}%  expanded {d.hit.mean() * 100:5.1f}%  diff {fmt_diff(diff, IDX)}  "
          f"(+{int((diff > 0).sum())} / -{int((diff < 0).sum())} tasks)")

print("\nLeave-one-repo-out, BM25 snapshot @50 and gate k=50 m=5 sequential diff:")
for r in sorted(set(repo_of.values())):
    keep = [i for i, t in enumerate(tasks) if repo_of[t] != r]
    x = np.array(hits[("bm25_snapshot", 50)], float)[keep]
    d = gate[(gate.k == 50) & (gate.m == 5) & (gate.order == "sequential")].set_index("task").reindex(tasks).iloc[keep]
    print(f"  without {r:16s} n={len(keep):3d}  BM25@50 {x.mean() * 100:.1f}%  gate diff {(d.hit.astype(float) - d.base.astype(float)).mean() * 100:+.1f} pp")
print("Per repo BM25 snapshot @50:", {r: f"{sum(h for h, t in zip(hits[('bm25_snapshot', 50)], tasks) if repo_of[t] == r)}/{sum(repo_of[t] == r for t in tasks)}"
                                      for r in sorted(set(repo_of.values()))})

verb = [any(re.search(r"\b" + re.escape(x.split(".")[-1]) + r"\b", TASKS[t]["problem_statement"]) for x in gold[t]) for t in tasks]
h50 = hits[("bm25_snapshot", 50)]
for v in (True, False):
    sel = [hh for hh, vv in zip(h50, verb) if vv == v]
    print(f"BM25@50, gold symbol's last name {'appears' if v else 'absent'} in problem_statement: {sum(sel)}/{len(sel)} = {np.mean(sel) * 100:.1f}%")
pd.DataFrame({"task": tasks, **{f"{a}@{k}": hits[(a, k)] for a, k in hits}}).to_csv(OUT / "hits.csv", index=False)

# ---------------------------------------------------------------- provenance manifest (A12)
import hashlib


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


manifest = {"kernel_versions": KERNEL_VERSIONS,
            "inputs": {str(p.relative_to(ROOT)): sha(p) for p in [RAW / "tasks.jsonl", GOLD_CSV, BM25_CSV,
                                                                 WORK / "gold_coverage/gold.csv",
                                                                 ROOT / "kaggle/gold_ast/gold_ast.py", ROOT / "kaggle/bm25/bm25.py"]},
            "graphs_sha256_of_sorted_hashes": hashlib.sha256("".join(sha(p) for p in sorted((RAW / "graphs").glob("*.json"))).encode()).hexdigest()}
(ROOT / "reports/c1_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print("manifest -> reports/c1_manifest.json")

# ---------------------------------------------------------------- PR-description objection: does the text carry code?
fence = [("```" in TASKS[t]["problem_statement"]) for t in tasks]
tick = [("`" in TASKS[t]["problem_statement"]) for t in tasks]
for name, flag in (("code fence (```)", fence), ("any backtick", tick)):
    for v in (True, False):
        sel = [hh for hh, ff in zip(h50, flag) if ff == v]
        print(f"BM25@50, problem_statement {'has' if v else 'has no'} {name}: {sum(sel)}/{len(sel)} = {np.mean(sel) * 100:.1f}%")


In [ ]:
%%writefile {W}/analysis/gold_heuristic.py
"""DEPRECATED text-matching gold heuristic (kept only to reproduce the heuristic-vs-AST agreement in reports/c1_results.md §3).
Writes work/c1/gold_coverage/. Run: uv run python analysis/gold_heuristic.py"""
import json,glob,re,os,collections,random,sys
import numpy as np, pandas as pd
from pathlib import Path
ROOT = str(Path(__file__).resolve().parents[1])
OUT=ROOT+'/work/c1/gold_coverage'
os.makedirs(OUT, exist_ok=True)
tasks=[json.loads(l) for l in open(ROOT+'/data/raw/tasks.jsonl')]
TRIV={'',')',']','}','pass','"""',"'''"}
def trivial(s):
    return s in TRIV or s.startswith('#') or s in ('return','...','else:','try:','finally:') 
def is_test(p):
    b=os.path.basename(p)
    return bool(re.search(r'(^|/)tests?/',p)) or b.startswith('test_') or b.endswith('_test.py')
def is_docs(p): return p.startswith('docs/') or p.startswith('docs_src/')
def module_of(path):
    p=path[:-3]
    if p.startswith('src/'): p=p[4:]   # src-layout (requests, httpx)
    m=p.replace('/','.')
    if m.endswith('.__init__'): m=m[:-9]
    return m
def parse_patch(patch):
    files=[];cur=None
    for line in patch.split('\n'):
        if line.startswith('--- '):
            old=line[4:]; cur=None; continue
        if line.startswith('+++ '):
            new=line[4:]; path=new[2:] if new.startswith('b/') else new
            if new=='/dev/null': path=old[2:]
            cur={'path':path,'new_file':old=='/dev/null','hunks':[]}; files.append(cur); continue
        if cur is None: continue
        m=re.match(r'@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@ ?(.*)',line)
        if m:
            cur['hunks'].append({'old_start':int(m.group(1)),'header':m.group(5),'pre':[],'minus':[],'plus':[]}); continue
        if not cur['hunks']: continue
        h=cur['hunks'][-1]
        if line.startswith('-'): h['pre'].append(('-',line[1:].strip())); h['minus'].append(line[1:].strip())
        elif line.startswith('+'): h['plus'].append(line[1:].strip())
        elif line.startswith(' ') : h['pre'].append((' ',line[1:].strip()))
    return files
# ---------- load graphs
graphs={}
def gpath(t):
    short=t['repo'].split('/')[1].lower()
    return f"{ROOT}/data/raw/graphs/{short}_{t['base_commit']}.json"
def load(t):
    p=gpath(t)
    if p not in graphs:
        if not os.path.exists(p): graphs[p]=None
        else:
            d=json.load(open(p)); nodes={}
            for n in d['nodes']: nodes.setdefault(n['id'],n)
            bymod=collections.defaultdict(list)
            for nid,n in nodes.items(): bymod[nid]  # placeholder
            graphs[p]={'d':d,'nodes':nodes}
    return graphs[p]
def mod_nodes(g,mod):
    if 'idx' not in g:
        # index by every dotted prefix
        idx=collections.defaultdict(list)
        for nid in g['nodes']:
            parts=nid.split('.')
            for i in range(1,len(parts)): idx['.'.join(parts[:i])].append(nid)
        g['idx']=idx
    return [n for n in g['idx'].get(mod,[]) if g['nodes'][n]['text']]
def kind_of(s):
    if re.match(r'async def ',s): return 'async_def'
    if s.startswith('def '): return 'def'
    if s.startswith('class '): return 'class'
    return 'unknown'
defre=re.compile(r'^(async def|def|class)\s+([A-Za-z_]\w*)')
def split_lines(txt): return [l.strip() for l in txt.split('\n') if l.strip()]
rows=[];summ=[]
nmiss_graph=0
for t in tasks:
    iid=t['instance_id'];repo=t['repo'].split('/')[1]
    files=parse_patch(t['patch'])
    g=load(t)
    code=[f for f in files if f['path'].endswith('.py') and not is_test(f['path']) and not is_docs(f['path'])]
    docs=[f for f in files if is_docs(f['path'])]
    tests=[f for f in files if is_test(f['path']) and not is_docs(f['path'])]
    nsym_in=0;nsym_out=0;newonly=0;nofilenode=0;nhunks=0
    for f in code:
        mod=module_of(f['path']); mn=mod_nodes(g,mod) if g else []
        has_node=len(mn)>0
        if g and not has_node: nofilenode+=1
        rows.append(dict(instance_id=iid,repo=repo,file=f['path'],module=mod,symbol='',level='file',in_graph=has_node,kind='file',match_method='',hunk_index=-1,n_tied=0,new_file=f['new_file']))
        if not mn or True:
            tl={n:split_lines(g['nodes'][n]['text']) for n in mn} if g else {}
            tj={n:'\n'.join(v) for n,v in tl.items()}
            name_set={n.split('.')[-1] for n in mn}
        for hi,h in enumerate(f['hunks']):
            nhunks+=1
            if f['new_file'] or h['old_start']==0 and not h['pre']:
                newonly+=1; rows.append(dict(instance_id=iid,repo=repo,file=f['path'],module=mod,symbol='',level='symbol',in_graph=False,kind='new_only',match_method='none',hunk_index=hi,n_tied=0,new_file=f['new_file'])); continue
            n_rows_before=len(rows)
            found={}  # node -> (method,tied)
            minus=[s for s in dict.fromkeys(h['minus']) if not trivial(s)]
            for s in minus:
                c=[n for n in mn if s in tl[n]]
                if c:
                    mlen=min(len(g['nodes'][n]['text']) for n in c)
                    best=[n for n in c if len(g['nodes'][n]['text'])==mlen]
                    for n in best: found.setdefault(n,('removed_line',len(best)))
            if not found:
                ctxl=[s for k,s in h['pre'] if k==' ']
                if not h['minus']:
                    pairs=[]
                    # consecutive context lines (in the diff pre-image order), nontrivial both
                    for i in range(len(h['pre'])-1):
                        a,b=h['pre'][i],h['pre'][i+1]
                        if a[0]==' ' and b[0]==' ' and not trivial(a[1]) and not trivial(b[1]): pairs.append((a[1],b[1]))
                    for a,b in pairs:
                        c=[n for n in mn if any(tl[n][i]==a and tl[n][i+1]==b for i in range(len(tl[n])-1))]
                        if c:
                            mlen=min(len(g['nodes'][n]['text']) for n in c)
                            best=[n for n in c if len(g['nodes'][n]['text'])==mlen]
                            for n in best: found.setdefault(n,('context',len(best)))
            # missing defs from '-' lines
            for s in h['minus']:
                m=defre.match(s)
                if m:
                    nm=m.group(2)
                    if nm not in name_set:
                        rows.append(dict(instance_id=iid,repo=repo,file=f['path'],module=mod,symbol=nm,level='symbol',in_graph=False,kind=kind_of(s),match_method='removed_line',hunk_index=hi,n_tied=0,new_file=False))
                        nsym_out+=1
            # header
            hm=defre.match(h['header'].strip()) if h['header'] else None
            if not found and hm:
                nm=hm.group(2); cands=[n for n in mn if n.split('.')[-1]==nm]
                if cands:
                    best=min(cands,key=lambda n:len(g['nodes'][n]['text'])) 
                    found[best]=('header',len(cands))
                else:
                    rows.append(dict(instance_id=iid,repo=repo,file=f['path'],module=mod,symbol=nm,level='symbol',in_graph=False,kind=kind_of(h['header'].strip()),match_method='header',hunk_index=hi,n_tied=0,new_file=False)); nsym_out+=1
            for n,(meth,tied) in found.items():
                rows.append(dict(instance_id=iid,repo=repo,file=f['path'],module=mod,symbol=n,level='symbol',in_graph=True,kind=kind_of(next((l.strip() for l in g['nodes'][n]['text'].split('\n') if l.strip() and not l.strip().startswith('@')),'')),match_method=meth,hunk_index=hi,n_tied=tied,new_file=False))
            if len(rows)==n_rows_before:
                rows.append(dict(instance_id=iid,repo=repo,file=f['path'],module=mod,symbol='',level='symbol',in_graph=False,kind=('module_level' if not h['header'].strip() else 'unmatched'),match_method='none',hunk_index=hi,n_tied=0,new_file=False))
    gdf_task=[r for r in rows if r['instance_id']==iid and r['level']=='symbol' and r['symbol']!='']
    nin=len({r['symbol'] for r in gdf_task if r['in_graph']}); nout=len({(r['file'],r['symbol']) for r in gdf_task if not r['in_graph']})
    summ.append(dict(instance_id=iid,repo=repo,graph_exists=g is not None,n_code_files=len(code),n_docs_files=len(docs),n_test_files=len(tests),
        n_hunks_code=nhunks,n_gold_in_graph=nin,n_gold_missing=nout,n_newonly_hunks=newonly,n_code_files_no_node=nofilenode,
        docs_only=bool(files) and len(docs)==len(files),test_only=bool(files) and len(tests)==len(files),no_code_file=len(code)==0))
gold=pd.DataFrame(rows);ts=pd.DataFrame(summ)
gold.to_csv(OUT+'/gold.csv',index=False);ts.to_csv(OUT+'/task_summary.csv',index=False)

# ================= B. census
cen=[];typ=collections.Counter();asyncnodes=0;nnodes_total=0
for p in sorted(glob.glob(ROOT+'/data/raw/graphs/*.json')):
    d=json.load(open(p));ns=d['nodes'];es=d['edges'];ids=[n['id'] for n in ns]
    idset=set(ids);ec=collections.Counter(e['type'] for e in es);typ.update(ec)
    deg=collections.Counter()
    for e in es: deg[e['source']]+=1;deg[e['target']]+=1
    selfl=sum(e['source']==e['target'] for e in es)
    dangling=sum((e['source'] not in idset) or (e['target'] not in idset) for e in es)
    iso=sum(1 for i in ids if deg[i]==0)
    cont=sum(1 for e in es if e['target'].startswith(e['source']+'.'))
    modlevel=sum(1 for i in ids if i.count('.')==0)
    asy=sum(1 for n in ns if re.search(r'(^|\n)\s*async def ',n['text'] or ''))
    asy_first=sum(1 for n in ns if re.match(r'\s*(@.*\n\s*)*async def ',n['text'] or ''))
    cen.append(dict(file=os.path.basename(p),n_nodes=len(ns),n_edges=len(es),types=dict(ec),self_loops=selfl,dangling_edges=dangling,isolated=iso,
        id_ne_name=sum(n['id']!=n['name'] for n in ns),dup_ids=len(ids)-len(idset),parallel_edges=sum(1 for e in es if e.get('key',0)>0),
        containment_edges=cont,single_segment_ids=modlevel,nodes_text_async_anywhere=asy,nodes_text_starts_async=asy_first,
        empty_text=sum(1 for n in ns if not n['text'])))
cdf=pd.DataFrame(cen);cdf['types']=cdf['types'].astype(str);cdf.to_csv(OUT+'/graph_census.csv',index=False)
# containment direction check: class->method calls edge exists?
# ================= C. coverage with bootstrap
rng=np.random.default_rng(0)
def boot(vals,B=10000):
    v=np.asarray(vals,float);n=len(v)
    if n==0: return (np.nan,np.nan,np.nan,0)
    idx=rng.integers(0,n,(B,n));m=v[idx].mean(1)
    return (v.mean(),*np.percentile(m,[2.5,97.5]),n)
def boot_ratio(num,den,B=10000):
    num=np.asarray(num,float);den=np.asarray(den,float);n=len(num)
    if den.sum()==0: return (np.nan,np.nan,np.nan,0)
    idx=rng.integers(0,n,(B,n));r=num[idx].sum(1)/np.maximum(den[idx].sum(1),1e-9)
    return (num.sum()/den.sum(),*np.percentile(r,[2.5,97.5]),int(den.sum()))
sym=gold[(gold.level=='symbol')&(gold.symbol!='')]
ts['has_code']=ts.n_code_files>0
lines=[]
def fmt(name,r): return f"| {name} | {r[3]} | {r[0]:.3f} | [{r[1]:.3f}, {r[2]:.3f}] |"
res=[]
def block(sub,label):
    c=sub[sub.has_code]
    res.append(fmt(f"{label}: tasks with >=1 code file (n tasks)",boot([1]*len(c))))
    res.append(fmt(f"{label}: share tasks with >=1 in-graph gold symbol",boot((c.n_gold_in_graph>0).astype(int))))
    res.append(fmt(f"{label}: share tasks with >=1 gold symbol (in or missing)",boot(((c.n_gold_in_graph+c.n_gold_missing)>0).astype(int))))
    res.append(fmt(f"{label}: share tasks with all code files having a node",boot((c.n_code_files_no_node==0).astype(int))))
    res.append(fmt(f"{label}: share tasks with >=1 code file with NO graph node",boot((c.n_code_files_no_node>0).astype(int))))
    res.append(fmt(f"{label}: share of gold symbols missing (pooled)",boot_ratio(c.n_gold_missing,c.n_gold_in_graph+c.n_gold_missing)))
block(ts,'ALL')
for r in ['fastapi','rich','requests','httpx']:
    if (ts[ts.repo==r].has_code).any(): block(ts[ts.repo==r],r)
# missing by kind
mk=sym[~sym.in_graph].drop_duplicates(['instance_id','file','symbol','kind']).groupby('kind').size()
tot_in=sym[sym.in_graph].drop_duplicates(['instance_id','symbol']).shape[0]
tot_out=sym[~sym.in_graph].drop_duplicates(['instance_id','file','symbol']).shape[0]
# async check
asy_pre=[];asy_hdr_missing=[]
for t in tasks:
    g=load(t)
    for f in parse_patch(t['patch']):
        if not f['path'].endswith('.py') or is_test(f['path']) or is_docs(f['path']): continue
        mod=module_of(f['path']);names={n.split('.')[-1] for n in mod_nodes(g,mod)} if g else set()
        for h in f['hunks']:
            for k,s in h['pre']:
                m=re.match(r'async def (\w+)',s)
                if m: asy_pre.append((t['instance_id'],f['path'],m.group(1),k,m.group(1) in names))
            m=re.match(r'async def (\w+)',h['header'].strip())
            if m: asy_hdr_missing.append((t['instance_id'],f['path'],m.group(1),m.group(1) in names))
apdf=pd.DataFrame(asy_pre,columns=['iid','file','name','pre_kind','node_exists'])
ahdf=pd.DataFrame(asy_hdr_missing,columns=['iid','file','name','node_exists'])
apdf.to_csv(OUT+'/async_pre_image.csv',index=False)
# repo-level gold-edge: do any async-def names from patch pre-images exist as nodes
mc=ts[ts.has_code]
# ================= D. eyeball
rs=random.Random(0);elig=[t for t in tasks if (ts.set_index('instance_id').loc[t['instance_id'],'n_gold_in_graph']>0)]
pick=rs.sample(elig,5);E=[]
for t in pick:
    E.append(f"## {t['instance_id']}\n")
    sg=sym[sym.instance_id==t['instance_id']]
    for f in parse_patch(t['patch']):
        if not f['path'].endswith('.py') or is_test(f['path']) or is_docs(f['path']): continue
        E.append(f"### {f['path']} (module `{module_of(f['path'])}`)\n")
        for hi,h in enumerate(f['hunks'][:6]):
            gs=sg[(sg.file==f['path'])&(sg.hunk_index==hi)]
            E.append(f"- hunk {hi}: header `@@ {h['header']}`; first removed: `{(h['minus'] or ['(none: addition-only)'])[0][:100]}`; first added: `{(h['plus'] or ['-'])[0][:100]}`")
            E.append("  - gold: "+("; ".join(f"`{r.symbol}` [{r.match_method}, in_graph={r.in_graph}, tied={r.n_tied}]" for r in gs.itertuples()) or "(none: module-level / new-only)"))
    E.append('')
open(OUT+'/eyeball.md','w').write("# Eyeball check: 5 random tasks (seed 0) with >=1 in-graph gold symbol\n\n"+"\n".join(E))
# ================= summary
S=[]
S.append(f"# Gold coverage summary\nGraph files: {len(cdf)} of 127 expected; tasks {len(tasks)}; graph_exists for {int(ts.graph_exists.sum())} tasks.\n")
S.append("## Coverage (task-level bootstrap, 10000 resamples, seed 0)\n| metric | n | estimate | 95% CI |\n|---|---|---|---|\n"+"\n".join(res))
S.append(f"\n## Gold symbols\nIn-graph unique (task,symbol): {tot_in}; missing unique (task,file,symbol): {tot_out}; missing by kind: {mk.to_dict()}")
S.append("\nmatch_method among in-graph symbol rows: "+str(sym[sym.in_graph].match_method.value_counts().to_dict())+"; tied-candidate rows (n_tied>1): "+str(int((sym[sym.in_graph].n_tied>1).sum())))
S.append("\nHunks without symbol gold: "+str(gold[gold.kind.isin(['module_level','new_only','unmatched'])].kind.value_counts().to_dict()))
S.append(f"\nTask flags: {ts[['docs_only','test_only','no_code_file']].sum().to_dict()}")
S.append(f"Code files (non-test, non-docs) total: {int(ts.n_code_files.sum())}; with NO node in graph: {int(ts.n_code_files_no_node.sum())} ({ts.n_code_files_no_node.sum()/ts.n_code_files.sum():.3f})")
nn=gold[(gold.level=='file')&(~gold.in_graph)]
S.append("No-node code files by repo: "+str(nn.groupby('repo').size().to_dict())+"\nBy file (top): "+str(nn.file.value_counts().head(12).to_dict()))
S.append("\n## Census\n"+f"graphs {len(cdf)}; nodes {cdf.n_nodes.sum()}; edges {cdf.n_edges.sum()}; edge type counts {dict(typ)}; self loops {cdf.self_loops.sum()}; dangling {cdf.dangling_edges.sum()}; isolated nodes {cdf.isolated.sum()} ({cdf.isolated.sum()/cdf.n_nodes.sum():.3f}); id!=name {cdf.id_ne_name.sum()}; dup ids {cdf.dup_ids.sum()}; parallel edges(key>0) {cdf.parallel_edges.sum()}; class->member containment edges {cdf.containment_edges.sum()}; single-segment (module-level) ids {cdf.single_segment_ids.sum()}; empty text {cdf.empty_text.sum()}; nodes with 'async def' line anywhere in text {cdf.nodes_text_async_anywhere.sum()}; nodes whose text starts (after decorators) with async def {cdf.nodes_text_starts_async.sum()}")
S.append("README documents CALLS / DEFINED_IN / IMPORTS; real types: "+str(dict(typ)))
S.append("\n## async def check\n"+f"async def pre-image lines in code-file hunks (patch, '-' or ' '): {len(apdf)} (unique task/file/name: {apdf.drop_duplicates(['iid','file','name']).shape[0]}); of which a node with that bare name exists in the module: {int(apdf.drop_duplicates(['iid','file','name']).node_exists.sum())}. Header-context async defs: {len(ahdf)}, with node: {int(ahdf.node_exists.sum())}.")
open(OUT+'/summary.md','w').write("\n".join(S))
print("\n".join(S))


In [ ]:
%%writefile {W}/analysis/embed_geometry.py
"""embed_geometry v2 (audit A10 fixes). Run: uv run python analysis/embed_geometry.py [n_files]
Fixes vs v1: AST node kind (decorated classes), exact mean pairwise cosine,
explicit aggregation units, display-tie vs true float32 tie at rank 10/11.
Output: work/c1/embed_geometry_v2/{per_file.csv,summary.md}
"""
import json, glob, os, re, sys, ast, textwrap
import numpy as np, pandas as pd
ROOT = str(__import__('pathlib').Path(__file__).resolve().parents[1])
OUT = ROOT + '/work/c1/embed_geometry_v2'
K = 10

def unit(X):
    n = np.linalg.norm(X, axis=1, keepdims=True); return X / np.maximum(n, 1e-12)

def exact_mean_pair_cos(U):
    n = len(U); s = U.sum(0)
    return float((s @ s - n) / (n * (n - 1)))   # mean over i!=j (rows unit-norm)

def node_kind(text):
    try:
        t = ast.parse(textwrap.dedent(text))
    except (SyntaxError, ValueError):
        return 'unknown'
    if not t.body: return 'unknown'
    s = t.body[0]
    if isinstance(s, ast.ClassDef): return 'class'
    if isinstance(s, ast.AsyncFunctionDef): return 'asyncfunc'
    if isinstance(s, ast.FunctionDef): return 'func'
    return 'unknown'

def is_docs(i): return any(re.fullmatch(r'tutorial\d+.*', p) for p in i.split('.'))
def is_test(i): return any(p.startswith('test') for p in i.split('.'))

def run_file(gf):
    base = os.path.basename(gf)[:-5]; repo = base.split('_')[0]
    g = json.load(open(gf)); z = np.load(f'{ROOT}/data/raw/embeddings/{base}.npz')
    nodes = sorted(g['nodes'], key=lambda nd: nd['id'])   # name order => stable sort reproduces tool's (-sim, name) tiebreak
    ids = [nd['id'] for nd in nodes]; n = len(ids); idx = {x: k for k, x in enumerate(ids)}
    X32 = np.stack([z[i] for i in ids]); dtype = str(X32.dtype)
    X32 = X32.astype(np.float32); X = X32.astype(np.float64)
    r = {'file': base, 'repo': repo, 'n': n, 'emb_dtype': dtype}
    nr = np.linalg.norm(X, axis=1)
    r.update(zero_frac=(nr == 0).mean(), nan_any=int(np.isnan(X).any()),
             dup_vec_frac=1 - len(np.unique(X32, axis=0)) / n)
    U = unit(X); mu = X.mean(0); Xc = X - mu; Uc = unit(Xc)
    r['raw_mean_pair_cos'] = exact_mean_pair_cos(U)
    r['cen_mean_pair_cos'] = exact_mean_pair_cos(Uc)
    r['cos_to_mean'] = float((U @ unit(mu[None])[0]).mean())
    ev = np.clip(np.linalg.eigvalsh(np.cov(Xc.T))[::-1], 0, None); p = ev / ev.sum()
    r['eff_rank'] = float(np.exp(-(p[p > 0] * np.log(p[p > 0])).sum()))
    r['pc1_share'] = float(p[0])
    # node kinds
    kinds = np.array([node_kind(nd['text']) for nd in nodes])
    r['n_unknown_kind'] = int((kinds == 'unknown').sum()); r['n_class'] = int((kinds == 'class').sum())
    r['n_func'] = int((kinds == 'func').sum()); r['n_asyncfunc'] = int((kinds == 'asyncfunc').sum())
    r['n_class_startswith_old'] = int(sum(nd['text'].lstrip().startswith('class ') for nd in nodes))
    cls = {i for i, k in zip(ids, kinds) if k == 'class'}
    par = lambda i: i.rsplit('.', 1)[0] if '.' in i else ''
    cpar = []; mod = []
    for i in ids:
        p_ = par(i); cpar.append(p_ if p_ in cls else ''); m = p_
        while m in cls: m = par(m)
        mod.append(m)
    mod = np.array(mod); cpar = np.array(cpar)
    A = np.zeros((n, n), bool)
    for e in g['edges']:
        a, b = idx.get(e['source']), idx.get(e['target'])
        if a is not None and b is not None and a != b: A[a, b] = A[b, a] = True
    docs = np.array([is_docs(i) for i in ids]); tst = np.array([is_test(i) for i in ids])
    r.update(frac_docs=docs.mean(), frac_test=tst.mean(), frac_with_class_parent=(cpar != '').mean())
    # queries: seed 0, up to 2000 (index space = name-sorted order; seed fixed)
    qo = np.random.default_rng(0).choice(n, min(2000, n), replace=False)   # sampled in ORIGINAL graph-node order (as v1)
    q = np.array([idx[g['nodes'][k]['id']] for k in qo]); nq = len(q)
    r['n_query'] = nq
    modc = pd.Series(mod).map(pd.Series(mod).value_counts()).values
    clsc = pd.Series(cpar).map(pd.Series(cpar).value_counts()).values
    # tool-faithful ties: float32 normalized, float32 dot, full sort
    nn32 = np.linalg.norm(X32, axis=1, keepdims=True); nn32[nn32 == 0] = 1e-12
    V32 = X32 / nn32
    S = (V32[q] @ V32.T).astype(np.float32); S[np.arange(nq), q] = -np.inf
    srt = -np.sort(-S, axis=1)                      # descending float32
    a, b = srt[:, K - 1], srt[:, K]
    ok = n - 1 > K; r['tie_eligible_q'] = nq if ok else 0
    if ok:
        r['n_disp_tie'] = int((np.round(a.astype(np.float64), 4) == np.round(b.astype(np.float64), 4)).sum())
        r['n_true_tie'] = int((a == b).sum())
    else: r['n_disp_tie'] = r['n_true_tie'] = 0
    for tag, V in (('raw', U), ('cen', Uc)):
        Sv = S if tag == 'raw' else (V[q] @ V.T)
        if tag == 'cen': Sv[np.arange(nq), q] = -np.inf
        top = np.argsort(-Sv, axis=1, kind='stable')[:, :min(K, n - 1)]   # name order tiebreak
        sims = np.take_along_axis(Sv, top, 1); k_ = top.shape[1]
        r[tag + '_nn_slots'] = nq * k_
        r[tag + '_nn_adj_num'] = int(A[q[:, None], top].sum())
        r[tag + '_nn_mod_num'] = int((mod[top] == mod[q][:, None]).sum())
        hc = cpar[q] != ''
        r[tag + '_cls_q'] = int(hc.sum()); r[tag + '_cls_slots'] = int(hc.sum()) * k_
        r[tag + '_nn_cls_num'] = int((cpar[top][hc] == cpar[q][hc][:, None]).sum())
        r[tag + '_nn_docs_num'] = int(docs[top].sum()); r[tag + '_nn_test_num'] = int(tst[top].sum())
        dq = docs[q]; tq = tst[q]
        r[tag + '_docsq_slots'] = int(dq.sum()) * k_; r[tag + '_nn_docs_given_docsq_num'] = int(docs[top][dq].sum())
        r[tag + '_testq_slots'] = int(tq.sum()) * k_; r[tag + '_nn_test_given_testq_num'] = int(tst[top][tq].sum())
        r[tag + '_dupvec_num'] = int((np.round(sims, 6) >= 1.0).sum()) if tag == 'raw' else 0
    deg = A.sum(1)[q]
    r['base_adj'] = float((deg / (n - 1)).mean()); r['base_mod'] = float(((modc[q] - 1) / (n - 1)).mean())
    hc = cpar[q] != ''
    r['base_cls'] = float((((clsc[q] - 1) / (n - 1))[hc]).mean()) if hc.any() else np.nan
    return r

def med(s): return float(np.median(s))

def summarize(d):
    L = []; w = L.append
    G = len(d)
    w('# embed_geometry v2 summary\n')
    w(f'Denominators: {G} graphs (files); queries = seed 0, up to 2000 per graph; K={K}. '
      f'Total queries = {int(d.n_query.sum()):,}; total nodes = {int(d.n.sum()):,}.\n')
    w('## 1. Geometry (unit: per-graph statistic, then MEDIAN over graphs; "repo median" = median of per-graph values within repo)\n')
    cols = [('eff_rank', 'effective rank (exp entropy of PC variance share)'), ('pc1_share', 'PC1 variance share'),
            ('raw_mean_pair_cos', 'mean pairwise cosine, exact, i!=j, raw L2-normalized'),
            ('cen_mean_pair_cos', 'mean pairwise cosine, exact, mean-centered'),
            ('cos_to_mean', 'cosine to mean direction (mean over nodes)'), ('dup_vec_frac', 'duplicate-vector share (1 - unique/n)')]
    w(f'| statistic | median over {G} graphs | min | max | ' + ' | '.join(sorted(d.repo.unique())) + ' |')
    w('|' + '---|' * (4 + d.repo.nunique()))
    for c, nm in cols:
        rm = d.groupby('repo')[c].median()
        w(f'| {nm} | {med(d[c]):.4f} | {d[c].min():.4f} | {d[c].max():.4f} | ' + ' | '.join(f'{rm[r]:.4f}' for r in sorted(d.repo.unique())) + ' |')
    w(f'\nPooled duplicate-vector share (sum dup nodes / sum nodes) not computed; the per-graph median above is the unit. Mean per-graph (not median) exact pairwise cosine = {d.raw_mean_pair_cos.mean():.4f}; node-count-weighted = {np.average(d.raw_mean_pair_cos, weights=d.n):.4f}.\n')
    w('## 2. Node kind (AST on node text)\n')
    w(f'Nodes: {int(d.n.sum()):,}; class {int(d.n_class.sum()):,}; func {int(d.n_func.sum()):,}; asyncfunc {int(d.n_asyncfunc.sum()):,}; **unknown (SyntaxError/empty) {int(d.n_unknown_kind.sum()):,}**. '
      f'Old startswith("class ") would count {int(d.n_class_startswith_old.sum()):,} classes (missed {int(d.n_class.sum()-d.n_class_startswith_old.sum()):,}).\n')
    w('## 3. Rank 10/11 ties (sort key = float32 cosine; display = round 4)\n')
    E = d.tie_eligible_q.sum()
    d = d.assign(disp_share=d.n_disp_tie / d.tie_eligible_q.replace(0, np.nan), true_share=d.n_true_tie / d.tie_eligible_q.replace(0, np.nan))
    w('| measure | pooled over queries (num/denom) | per-graph median share |')
    w('|---|---|---|')
    w(f'| (a) display tie: rank-10 == rank-11 at 4 decimals | {int(d.n_disp_tie.sum()):,}/{int(E):,} = {d.n_disp_tie.sum()/E:.4%} | {med(d.disp_share.dropna()):.4%} |')
    w(f'| (b) true tie: float32 rank-10 == rank-11 exactly | {int(d.n_true_tie.sum()):,}/{int(E):,} = {d.n_true_tie.sum()/E:.4%} | {med(d.true_share.dropna()):.4%} |')
    w(f'\nDenominator = queries in graphs with n-1 > {K} ({int((d.tie_eligible_q>0).sum())} graphs).\n')
    w('## 4. Neighbor structure vs random baseline (top-10 by cosine, name tiebreak)\n')
    w('Each share = hits / neighbor slots. Overall = pooled over slots; "per-graph median" = median of per-graph shares. Baseline = per-graph mean over queries of (candidates-in-group)/(n-1).\n')
    def share(num, den): return d[num].sum() / d[den].sum(), med(d[num] / d[den].replace(0, np.nan))
    for tag in ('raw', 'cen'):
        w(f'### {tag} vectors\n')
        w('| measure | denominator (slots) | pooled | per-graph median | baseline per-graph median |')
        w('|---|---|---|---|---|')
        rows = [('neighbor adjacent (edge)', 'nn_adj_num', 'nn_slots', 'base_adj'), ('same module', 'nn_mod_num', 'nn_slots', 'base_mod'),
                ('same class parent (queries with class parent)', 'nn_cls_num', 'cls_slots', 'base_cls'),
                ('neighbor is docs_src', 'nn_docs_num', 'nn_slots', None), ('neighbor is test', 'nn_test_num', 'nn_slots', None),
                ('docs neighbor | docs query', 'nn_docs_given_docsq_num', 'docsq_slots', None), ('test neighbor | test query', 'nn_test_given_testq_num', 'testq_slots', None)]
        for nm, nu, de, bs in rows:
            a, b = share(f'{tag}_{nu}', f'{tag}_{de}')
            w(f'| {nm} | {int(d[f"{tag}_{de}"].sum()):,} | {a:.4f} | {b:.4f} | ' + (f'{med(d[bs].dropna()):.4f}' if bs else '-') + ' |')
        if tag == 'raw':
            a, b = share('raw_dupvec_num', 'raw_nn_slots'); w(f'| neighbor is exact duplicate vector (cos>=1 at 6dp) | {int(d.raw_nn_slots.sum()):,} | {a:.4f} | {b:.4f} | - |')
        w('')
    w('## 5. Same-class share per repo (raw vectors; slots = queries with class parent x 10)\n')
    w('| repo | graphs | slots | pooled | per-graph median | baseline median |')
    w('|---|---|---|---|---|---|')
    for r, s in d.groupby('repo'):
        w(f'| {r} | {len(s)} | {int(s.raw_cls_slots.sum()):,} | {s.raw_nn_cls_num.sum()/s.raw_cls_slots.sum():.4f} | {med(s.raw_nn_cls_num/s.raw_cls_slots.replace(0,np.nan)):.4f} | {med(s.base_cls.dropna()):.4f} |')
    w(f'| ALL | {G} | {int(d.raw_cls_slots.sum()):,} | {d.raw_nn_cls_num.sum()/d.raw_cls_slots.sum():.4f} | {med(d.raw_nn_cls_num/d.raw_cls_slots.replace(0,np.nan)):.4f} | {med(d.base_cls.dropna()):.4f} |')
    w('\n## 6. Confound shares (per-graph median of node shares)\n')
    w(f'docs_src node share {med(d.frac_docs):.4f}; test node share {med(d.frac_test):.4f}; nodes with class parent {med(d.frac_with_class_parent):.4f}.')
    return '\n'.join(L)

if __name__ == '__main__':
    os.makedirs(OUT, exist_ok=True)
    fs = sorted(glob.glob(ROOT + '/data/raw/graphs/*.json'))
    if len(sys.argv) > 1: fs = fs[:int(sys.argv[1])]
    rows = []
    for f in fs:
        if not os.path.exists(f'{ROOT}/data/raw/embeddings/{os.path.basename(f)[:-5]}.npz'): continue
        rows.append(run_file(f)); print(rows[-1]['file'], rows[-1]['n'], flush=True)
    d = pd.DataFrame(rows)
    suf = '' if len(sys.argv) == 1 else '_dev'
    d.to_csv(f'{OUT}/per_file{suf}.csv', index=False)
    open(f'{OUT}/summary{suf}.md', 'w').write(summarize(d))


In [ ]:
%%writefile {W}/analysis/reach.py
"""C1 reach analysis v2 (post Codex audit A02/A06/A10): issue text -> gold symbols via graph/embedding tools.
Run from project root:  uv run python analysis/reach.py
Only stdlib/numpy/pandas/scipy. Reads data/raw (read-only); writes work/c1/reach/.
"""
import json, re, sys, collections, math
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
OUT = ROOT / "work/c1/reach_v3"
OUT.mkdir(parents=True, exist_ok=True)
RAW = ROOT / "data/raw"
KS = [10, 25, 50, 100]
CAP = 50
PKG = {"fastapi", "rich", "requests", "httpx"}

# ---------------------------------------------------------------- seeds
STOP = set("""if else elif for while try except finally with as import from return yield raise pass break continue lambda
class def not and or in is del global nonlocal assert await async None True False self cls print len str int float bool list
dict set tuple type range open super isinstance getattr setattr hasattr object bytes sorted enumerate zip map filter any all min max sum
repr id e.g i.e etc Exception ValueError TypeError KeyError RuntimeError AttributeError NotImplementedError OSError
IndexError StopIteration ImportError""".split())
TOK = re.compile(r"[A-Za-z_]\w*")
DOT = re.compile(r"(?<![\w.])[A-Za-z_]\w*(?:\.[A-Za-z_]\w*)+")
CALL = re.compile(r"\b([A-Za-z_]\w*)\s*\(")
BT = re.compile(r"`([^`\n]+)`")
BTOK = re.compile(r"^[\w.]+(\(\))?$")


def is_camel(t):
    return len(t) > 2 and t[0].isupper() and any(c.islower() for c in t) and any(c.isupper() for c in t[1:])


def is_snake(t):
    return "_" in t.strip("_") and any(c.isalpha() for c in t) and not t.isupper()


def clean(c):
    c = c.strip().rstrip(".,;:")
    if c.endswith("()"):
        c = c[:-2]
    return c.strip(".")


def extract_candidates(text):
    """ordered list of (candidate, kind) by first appearance; kinds joined."""
    text0 = re.sub(r"```[\s\S]*?```", lambda m: m.group(0).replace("`", " "), text)  # keep code text, drop fences
    found = {}  # cand -> (pos, set(kinds))

    def add(c, pos, kind):
        c = clean(c)
        if len(c) < 2 or c in STOP:
            return
        if c not in found:
            found[c] = [pos, {kind}]
        else:
            found[c][0] = min(found[c][0], pos)
            found[c][1].add(kind)

    for m in BT.finditer(text0):
        s = m.group(1).strip()
        if BTOK.match(s):
            add(s, m.start(), "backtick")
    for m in DOT.finditer(text0):
        add(m.group(0), m.start(), "dotted")
    for m in TOK.finditer(text0):
        t = m.group(0)
        if is_camel(t):
            add(t, m.start(), "camel")
        if is_snake(t):
            add(t, m.start(), "snake")
    for m in CALL.finditer(text0):
        add(m.group(1), m.start(), "call")
    return sorted(((c, v[0], sorted(v[1])) for c, v in found.items()), key=lambda x: (x[1], x[0]))


# ---------------------------------------------------------------- graph
class G:
    pass


def load_graph(repo, commit):
    short = repo.split("/")[1].lower()
    name = f"{short}_{commit}"
    d = json.load(open(RAW / f"graphs/{name}.json"))
    ids = sorted({n["id"] for n in d["nodes"]})
    idx = {s: i for i, s in enumerate(ids)}  # index == alphabetical rank
    N = len(ids)
    outs = [set() for _ in range(N)]
    ins = [set() for _ in range(N)]
    for e in d["edges"]:
        u, v = idx[e["source"]], idx[e["target"]]
        outs[u].add(v)
        ins[v].add(u)
    g = G()
    g.name, g.ids, g.idx, g.N = name, ids, idx, N
    g.adj = [sorted(outs[i] | ins[i]) for i in range(N)]  # tool: out+in merged, dedup, alphabetical (self loops kept as in tool)
    g.und = [[j for j in a if j != i] for i, a in enumerate(g.adj)]
    g.lower = [s.lower() for s in ids]
    pkg = repo.split("/")[1].lower()
    cls = []
    for s in ids:
        comps = s.split(".")
        if comps[0] == pkg:
            cls.append("lib")
        elif comps[0] in ("tests", "test") or any(c.startswith("test_") or c == "conftest" for c in comps):
            cls.append("test")
        else:
            cls.append("other")
    g.cls = cls
    # WCC
    comp = [-1] * N
    c = 0
    for s in range(N):
        if comp[s] >= 0:
            continue
        comp[s] = c
        st = [s]
        while st:
            x = st.pop()
            for y in g.und[x]:
                if comp[y] < 0:
                    comp[y] = c
                    st.append(y)
        c += 1
    g.comp = comp
    emb = np.load(RAW / f"embeddings/{name}.npz")
    M = np.zeros((N, 256), dtype=np.float32)
    miss = 0
    for s, i in idx.items():
        if s in emb.files:
            M[i] = emb[s]
        else:
            miss += 1
    g.M, g.emb_missing = M, miss
    g.M32 = M.astype(np.float32)
    g.zero_vec = int((np.linalg.norm(M, axis=1) == 0).sum())
    return g


def bfs_dist(g, sources, maxd=50):
    dist = {s: 0 for s in sources}
    fr = list(dist)
    d = 0
    while fr and d < maxd:
        d += 1
        nf = []
        for x in fr:
            for y in g.und[x]:
                if y not in dist:
                    dist[y] = d
                    nf.append(y)
        fr = nf
    return dist


# ---------------------------------------------------------------- resolvers
def resolve_impl(g, cands):
    """-> list of (cand, [node idx]) ; exact id only."""
    out = []
    for c, pos, kinds in cands:
        if c in g.idx:
            out.append((c, [g.idx[c]], "exact"))
    return out


def resolve_doc(g, cands):
    """tiers: exact -> suffix on '.name' boundary -> case-insens (equal or suffix) -> substring (ci).
    Tiers 2-4 need len>=4. returns list of (cand, [idx sorted shortest id first], tier)."""
    out = []
    for c, pos, kinds in cands:
        if c in g.idx:
            out.append((c, [g.idx[c]], "exact"))
            continue
        if len(c) < 4:
            continue
        suf = "." + c
        m = [i for i, s in enumerate(g.ids) if s.endswith(suf)]
        tier = "suffix"
        if not m:
            cl = c.lower()
            sufl = "." + cl
            m = [i for i, s in enumerate(g.lower) if s == cl or s.endswith(sufl)]
            tier = "caseins"
        if not m:
            cl = c.lower()
            m = [i for i, s in enumerate(g.lower) if cl in s]
            tier = "substring"
        if m:
            m = sorted(m, key=lambda i: (len(g.ids[i]), g.ids[i]))
            out.append((c, m, tier))
    return out


def seeds_from(res, maxper=None):
    seeds, seen = [], set()
    for c, m, tier in res:
        for i in (m if maxper is None else m[:maxper]):
            if i not in seen:
                seen.add(i)
                seeds.append(i)
    return seeds


# ---------------------------------------------------------------- candidate generation
def rr(lists, exclude=()):
    """round-robin merge, dedup, drop excluded."""
    seen = set(exclude)
    out = []
    mx = max((len(l) for l in lists), default=0)
    for i in range(mx):
        for l in lists:
            if i < len(l):
                x = l[i]
                if x not in seen:
                    seen.add(x)
                    out.append(x)
    return out


def hop1(g, seeds, cap):
    return rr([g.adj[s][:cap] if cap else g.adj[s] for s in seeds], exclude=seeds)


def hop2(g, seeds, cap):
    r1 = hop1(g, seeds, cap)
    seen = set(seeds) | set(r1)
    sec = rr([g.adj[p][:cap] if cap else g.adj[p] for p in r1], exclude=seen)
    return r1 + sec, len(seeds) + len(r1)


def emb_lists(g, seeds, K=200):
    """Exactly as retrieval_utils.py:819-833: float32 normalise, float32 dot, sort by (-float32 sim, name) on UNROUNDED value."""
    E = g.M32
    norms = np.linalg.norm(E, axis=1, keepdims=True)
    norms[norms == 0] = 1e-12
    NE = E / norms
    ar = np.arange(g.N)  # g.ids is sorted -> index order == name order
    out = []
    for s in seeds:
        t = E[s]
        tn = np.linalg.norm(t)
        if tn == 0:
            tn = 1e-12
        nt = t / tn
        sims = np.dot(NE, nt.T).flatten()
        assert sims.dtype == np.float32
        sims[s] = -np.inf
        order = np.lexsort((ar, -sims))
        out.append([int(x) for x in order if x != s][:K])
    return out


# ---------------------------------------------------------------- stats
def boot_idx(n, B=10000, seed=0):
    return np.random.default_rng(seed).integers(0, n, size=(B, n))


_BI = {}


def bi(n):
    if n not in _BI:
        _BI[n] = boot_idx(n)
    return _BI[n]


def ci(x):
    x = np.asarray(x, float)
    n = len(x)
    if n == 0:
        return (np.nan, np.nan, np.nan)
    if n < 2:
        return (x.mean(), np.nan, np.nan)
    m = x[bi(n)].mean(axis=1)
    return (x.mean(), *np.percentile(m, [2.5, 97.5]))


def cid(x, y):
    d = np.asarray(x, float) - np.asarray(y, float)
    n = len(d)
    m = d[bi(n)].mean(axis=1)
    return (d.mean(), *np.percentile(m, [2.5, 97.5]))


def fmt(t, pct=True):
    m, lo, hi = t
    if np.isnan(lo):
        return f"{m*100:.1f}% (n=1, no CI)" if pct else f"{m:.2f}"
    return f"{m*100:.1f}% [{lo*100:.1f}, {hi*100:.1f}]" if pct else f"{m:.2f} [{lo:.2f}, {hi:.2f}]"


def fmtd(t):
    return f"{t[0]*100:+.1f}pp [{t[1]*100:+.1f}, {t[2]*100:+.1f}]"


# ================================================================= main
VIEWS = ("exp", "e2e")


def table(headers, rows):
    s = "| " + " | ".join(headers) + " |\n|" + "|".join(["---"] * len(headers)) + "|\n"
    for r in rows:
        s += "| " + " | ".join(str(x) for x in r) + " |\n"
    return s


def main():
    tasks = {}
    for l in open(RAW / "tasks.jsonl"):
        t = json.loads(l)
        tasks[t["instance_id"]] = t
    gold = pd.read_csv(ROOT / "work/c1/report/gold_mapped.csv")
    ing = gold[(gold.kind != "module") & (gold.in_graph == True)].drop_duplicates(["instance_id", "node"])  # noqa: E712
    eval_ids = sorted(ing.instance_id.unique())
    n = len(eval_ids)
    rc = {tasks[i]["repo"].split("/")[1]: 0 for i in eval_ids}
    for i in eval_ids:
        rc[tasks[i]["repo"].split("/")[1]] += 1
    assert n == 115 and rc == {"fastapi": 59, "rich": 43, "requests": 12, "httpx": 1}, (n, rc)  # gold v4

    rows, seed_rows, trunc = [], [], collections.Counter()
    gcache = {}
    for iid in eval_ids:
        t = tasks[iid]
        repo = t["repo"]
        key = (repo, t["base_commit"])
        if key not in gcache:
            gcache.clear()
            gcache[key] = load_graph(*key)
        g = gcache[key]
        gold_idx = sorted(g.idx[x] for x in ing[ing.instance_id == iid].node)
        goldset = set(gold_idx)
        cands = extract_candidates(t["problem_statement"])
        r_impl, r_doc = resolve_impl(g, cands), resolve_doc(g, cands)
        S = {"impl": seeds_from(r_impl), "doc5": seeds_from(r_doc, maxper=5)}
        seed_rows.append(dict(instance_id=iid, repo=repo, n_cands=len(cands), n_impl_seeds=len(S["impl"]), n_doc5_seeds=len(S["doc5"]),
                              impl_trivial=bool(set(S["impl"]) & goldset), doc5_trivial=bool(set(S["doc5"]) & goldset), n_gold=len(gold_idx)))
        strat = {}
        for lab, sd in S.items():
            for s in sd:
                trunc[(lab, "1hop_calls")] += 1
                trunc[(lab, "1hop_truncated")] += int(len(g.adj[s]) > CAP)
            if not sd:
                for nm in ("nb1", "nb1u", "nb2", "nb2u", "emb"):
                    strat[f"{nm}_{lab}"] = (sd, [])
                continue
            strat[f"nb1_{lab}"] = (sd, hop1(g, sd, CAP))
            strat[f"nb1u_{lab}"] = (sd, hop1(g, sd, None))
            strat[f"nb2_{lab}"] = (sd, hop2(g, sd, CAP)[0])
            strat[f"nb2u_{lab}"] = (sd, hop2(g, sd, None)[0])
            L = emb_lists(g, sd)
            strat[f"emb_{lab}"] = (sd, rr(L, exclude=sd))
        for nm, (sd, cl) in strat.items():
            triv = bool(set(sd) & goldset)
            for k in KS:
                for view in VIEWS:
                    top = cl[:k] if view == "exp" else (sd + cl)[:k]
                    rows.append(dict(instance_id=iid, repo=repo, strategy=nm, k=k, view=view, n_seeds=len(sd), n_cands=len(top),
                                     trivial=triv, hit=int(bool(set(top) & goldset))))
        print(iid, len(S["impl"]), len(S["doc5"]), file=sys.stderr)
    PT = pd.DataFrame(rows)
    PT.to_csv(OUT / "per_task.csv", index=False)
    SR = pd.DataFrame(seed_rows)
    SR.to_csv(OUT / "seeds_per_task.csv", index=False)
    report(PT, SR, trunc, n, rc)


def report(PT, SR, trunc, n, rc):
    md = []
    A = md.append

    def H(s, k, v):
        return PT[(PT.strategy == s) & (PT.k == k) & (PT.view == v)].sort_values("instance_id").hit.values

    def cell(x):
        m, lo, hi = ci(x)
        return f"{int(x.sum())}/{n} = {m*100:.1f}% [{lo*100:.1f}, {hi*100:.1f}]"

    A("# C1 reach v2 (AST gold v3; float32 embedding order; end-to-end view)\n")
    A(f"Unit = task, n={n} ({rc}). Gold = `work/c1/report/gold_mapped.csv`, kind != module, in_graph, dedup (instance_id, node). "
      "Bootstrap as analysis/c1_report.py: tasks sorted by id, rng=default_rng(0), idx=integers(0,n,(10000,n)), percentile 2.5/97.5; "
      "task-resampling only, repository clustering not reflected (fastapi+rich = 102/115).\n")
    A("## Policies (stated)\n")
    A("""- **IMPL**: candidate must equal a node id exactly (what the tools do).
- **DOC-5** (README counterfactual, NOT specified by the docs): tier 1 exact id; tiers 2-4 (suffix on `.name` boundary, case-insensitive equal/suffix, case-insensitive substring) require **min length 4**; the first non-empty tier wins; **at most 5 matches per candidate, shortest ids first** (then alphabetical). Seeds = union in candidate first-appearance order.
- **Neighbours**: out+in merged, dedup, alphabetical, `max_neighbors=50` cap (default of the tool, not a fixed limit); `u` = no cap. Per-seed lists merged round-robin, deduped. 2-hop = 1-hop list followed by round-robin expansion of all 1-hop nodes.
- **Embedding (tool-faithful, A02)**: float32 normalisation and float32 dot as `retrieval_utils.py:819-833`, sort by (-sim, name) on the UNROUNDED float32 value, no rank-time rounding; per-seed top-200, round-robin merged.
- **Expansion-only (exp)**: candidates exclude seeds. **End-to-end (e2e)**: candidate list = resolved seeds first (resolver order) then the expansion, whole list cut to the same k.
""")
    A("## 1. Seed gating\n")
    rows = []
    for lab, c in (("IMPL (exact id)", "n_impl_seeds"), ("DOC-5", "n_doc5_seeds")):
        x = (SR[c] > 0).astype(int).values
        rows.append((lab, cell(x)))
    for lab, c in (("IMPL seed is already gold (trivial)", "impl_trivial"), ("DOC-5 seed is already gold (trivial)", "doc5_trivial")):
        x = SR[c].astype(int).values
        rows.append((lab, cell(x)))
    A(table(["measure", "tasks"], rows))
    A("Seed counts per task, median [min, max]: " + "; ".join(f"{c} {int(SR[c].median())} [{SR[c].min()}, {SR[c].max()}]" for c in ["n_impl_seeds", "n_doc5_seeds"]) + "\n")
    names = {"nb1_impl": "IMPL 1-hop (cap50)", "nb2_impl": "IMPL 2-hop (cap50)", "emb_impl": "IMPL embedding (float32 order)",
             "nb1_doc5": "DOC-5 1-hop (cap50)", "nb2_doc5": "DOC-5 2-hop (cap50)", "emb_doc5": "DOC-5 embedding (float32 order)"}
    for v, title in (("exp", "2a. Hit@k, expansion-only (seeds excluded)"), ("e2e", "2b. Hit@k, END-TO-END (seeds first, same budget k)")):
        A(f"## {title}\n")
        A(table(["strategy (policy)"] + [f"@{k}" for k in KS], [[f"{names[s]}"] + [cell(H(s, k, v)) for k in KS] for s in names]))
    A("DOC-5 policy: min length 4 (tiers 2-4), <=5 matches per candidate, shortest ids first. IMPL = exact id only.\n")
    A("## 3. Paired difference e2e minus exp\n")
    rows = []
    for s in names:
        rows.append([names[s]] + [fmtd(cid(H(s, k, "e2e"), H(s, k, "exp"))) for k in KS])
    A(table(["strategy"] + [f"@{k}" for k in KS], rows))
    A("## 4. Effect of removing the alphabetical max_neighbors=50 cap (task-level hit changes in the evaluated settings only)\n")
    A("Counts of tasks whose hit status changes when the cap is removed: gained (uncapped hits, capped misses) / lost (capped hits, uncapped misses). Not a claim about other settings.\n")
    rows = []
    for a, b, lab in (("nb1_impl", "nb1u_impl", "IMPL 1-hop"), ("nb2_impl", "nb2u_impl", "IMPL 2-hop"), ("nb1_doc5", "nb1u_doc5", "DOC-5 1-hop"), ("nb2_doc5", "nb2u_doc5", "DOC-5 2-hop")):
        for v in VIEWS:
            r = [lab, v]
            for k in KS:
                x, y = H(a, k, v), H(b, k, v)
                r.append(f"+{int(((y==1)&(x==0)).sum())} / -{int(((x==1)&(y==0)).sum())} (hits {int(x.sum())}->{int(y.sum())})")
            rows.append(r)
    A(table(["setting", "view"] + [f"@{k}" for k in KS], rows))
    A("Share of 1-hop seed calls whose neighbour list was actually truncated by the cap (merged degree incl. self-loop > 50):\n")
    rows = []
    for lab in ("impl", "doc5"):
        c, tr = trunc[(lab, "1hop_calls")], trunc[(lab, "1hop_truncated")]
        rows.append((lab, c, tr, f"{tr/c*100:.1f}%" if c else "n/a"))
    A(table(["seeds", "1-hop seed calls", "truncated", "share"], rows))
    A("## 5. Per-repo hit@50 (e2e / exp), hits/n\n")
    rows = []
    for s in names:
        r = [names[s]]
        for repo in ("fastapi/fastapi", "Textualize/rich", "psf/requests", "encode/httpx"):
            r.append("/".join(f"{int(PT[(PT.strategy==s)&(PT.k==50)&(PT.view==v)&(PT.repo==repo)].hit.sum())}" for v in ("e2e", "exp")) + f" of {int(((PT.strategy==s)&(PT.k==50)&(PT.view=='exp')&(PT.repo==repo)).sum())}")
        rows.append(r)
    A(table(["strategy", "fastapi", "rich", "requests", "httpx"], rows))
    (OUT / "summary.md").write_text("\n".join(md))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {W}/analysis/reach_0211.py
"""Table 1 reach, measured by calling the shipped swegemma graph functions instead of a counterfactual resolver.

Same 115 tasks, same gold, same identifier extraction and round-robin merge as analysis/reach.py. Differences:
the seeds and candidate lists come from the wheel's own resolve_node_name / get_neighbor / get_similar_nodes.
Seeds are credited end-to-end (seeds first, then expansion) as in Table 1, which favours the graph tools:
the tools do not report which node a short name resolved to.

Needs a throwaway env with networkx, numpy, cachetools, pandas (not project dependencies):
  python analysis/reach_0211.py --wheel-src work/harness_0_2_11/src --out work/c1/reach_0211   # shipped 0.2.11
  python analysis/reach_0211.py --wheel-src work/pkgsrc --out work/c1/reach_0207               # control: must give 2 / 2 / 2
"""
import argparse, collections, json, logging, sys, types
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "analysis"))
from reach import extract_candidates, rr  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--wheel-src", required=True)
ap.add_argument("--out", required=True)
a = ap.parse_args()

# swegemma/__init__.py pulls in the whole harness; only the graph subpackage is needed here.
pkg = types.ModuleType("swegemma")
pkg.__path__ = [str(Path(a.wheel_src) / "swegemma")]
sys.modules["swegemma"] = pkg
import swegemma.graph as sg  # noqa: E402

logging.disable(logging.WARNING)
RAW = ROOT / "data/raw"
OUT = ROOT / a.out
OUT.mkdir(parents=True, exist_ok=True)
GD, ED = str(RAW / "graphs"), str(RAW / "embeddings")
KS = (10, 25, 50, 100)
CAP, SIM_K = 50, 200  # reach.py: 50 neighbours per call, 200 embedding results per seed
HAS_RESOLVER = hasattr(sg, "resolve_node_name")

tasks = {t["instance_id"]: t for t in map(json.loads, open(RAW / "tasks.jsonl"))}
gold = pd.read_csv(ROOT / "work/c1/report/gold_mapped.csv")
ing = gold[(gold.kind != "module") & (gold.in_graph == True)].drop_duplicates(["instance_id", "node"])  # noqa: E712
ids = sorted(ing.instance_id.unique())
rc = collections.Counter(tasks[i]["repo"].split("/")[1] for i in ids)
assert len(ids) == 115 and rc == {"fastapi": 59, "rich": 43, "requests": 12, "httpx": 1}, rc


def bare(s):
    return s.split(";", 1)[1] if ";" in s else s


rows, seed_rows = [], []
for iid in ids:
    t = tasks[iid]
    repo, commit = t["repo"], t["base_commit"]
    kw = dict(use_local_storage=True, repo_name=repo, graph_dir=GD, embeddings_dir=ED, base_commit=commit)
    g = sg.get_graph(repo, use_local_storage=True, graph_dir=GD, embeddings_dir=ED, base_commit=commit)
    assert g is not None and g.number_of_nodes() > 0, iid
    goldset = set(ing[ing.instance_id == iid].node)
    assert goldset <= set(g.nodes()), iid

    seeds, nb_lists, sim_lists = [], [], []
    n_resolved = n_lexical = 0
    cands = [c for c, _, _ in extract_candidates(t["problem_statement"])]
    for c in cands:
        if HAS_RESOLVER:
            r = sg.resolve_node_name(c, g)
        else:
            r = c if g.has_node(c) else None
        if r is not None:
            n_resolved += 1
            if r not in seeds:
                seeds.append(r)
            nb = sg.get_neighbor(node=f"{repo};{c}", max_neighbors=CAP, **kw)
            nb_lists.append([bare(x) for x in nb])
        sim = sg.get_similar_nodes(node=c, k=SIM_K, repo_name=repo, use_local_storage=True, graph_dir=GD,
                                   embeddings_dir=ED, base_commit=commit)
        if sim:
            n_lexical += int(r is None)
            sim_lists.append([bare(x["node_name"]) for x in sim])
    issue = sg.get_similar_nodes(node=t["problem_statement"], k=max(KS), repo_name=repo, use_local_storage=True,
                                 graph_dir=GD, embeddings_dir=ED, base_commit=commit)
    strat = {
        "neighbors_e2e": seeds + rr(nb_lists, exclude=seeds),
        "similar_e2e": seeds + rr(sim_lists, exclude=seeds),
        "issue_text_query": [bare(x["node_name"]) for x in issue],
    }
    seed_rows.append(dict(task=iid, repo=repo, n_cands=len(cands), n_resolved=n_resolved, n_seeds=len(seeds),
                          n_lexical=n_lexical, seed_is_gold=bool(set(seeds) & goldset), n_issue=len(issue)))
    for nm, lst in strat.items():
        row = dict(task=iid, strategy=nm)
        for k in KS:
            row[f"hit{k}"] = int(bool(set(lst[:k]) & goldset))
        rows.append(row)
    print(iid, len(cands), n_resolved, len(seeds), n_lexical, len(issue), file=sys.stderr, flush=True)

PT = pd.DataFrame(rows)
PT.to_csv(OUT / "per_task.csv", index=False)
SR = pd.DataFrame(seed_rows)
SR.to_csv(OUT / "seeds_per_task.csv", index=False)

rng = np.random.default_rng(0)
B = rng.integers(0, len(ids), size=(10000, len(ids)))
hits = pd.read_csv(ROOT / "work/c1/report/hits.csv").set_index("task").loc[ids]


def ci(x):
    d = np.asarray(x, float)[B].mean(axis=1) * 100
    return np.percentile(d, 2.5), np.percentile(d, 97.5)


lines = [f"wheel source: {a.wheel_src}; resolver present: {HAS_RESOLVER}; n = {len(ids)}", ""]
lines += ["| strategy | " + " | ".join(f"hit@{k}" for k in KS) + " | hit@50 95% CI | hit@50 minus BM25@50 [95% CI] | only strategy, only BM25 |",
          "|---|" + "---|" * (len(KS) + 3)]
bm = hits["bm25_snapshot@50"].astype(int).to_numpy()
for nm, df in PT.groupby("strategy", sort=False):
    df = df.set_index("task").loc[ids]
    x = df["hit50"].to_numpy()
    lo, hi = ci(x)
    dl, dh = ci(x - bm)
    lines.append(f"| {nm} | " + " | ".join(str(int(df[f'hit{k}'].sum())) for k in KS)
                 + f" | {x.mean()*100:.1f}% [{lo:.1f}, {hi:.1f}] | {(x-bm).mean()*100:+.1f} pp [{dl:+.1f}, {dh:+.1f}] | {int(((x==1)&(bm==0)).sum())}, {int(((x==0)&(bm==1)).sum())} |")
lines += ["", f"BM25 (work/c1/report/hits.csv): @10 {int(hits['bm25_snapshot@10'].sum())}, @25 {int(hits['bm25_snapshot@25'].sum())}, "
          f"@50 {int(bm.sum())}, @100 {int(hits['bm25_snapshot@100'].sum())}", "",
          f"seeds: median candidates {SR.n_cands.median():.0f}, median resolved {SR.n_resolved.median():.0f}, median distinct seeds {SR.n_seeds.median():.0f}, "
          f"tasks with a gold seed {int(SR.seed_is_gold.sum())}, candidates that reached the lexical fallback (total) {int(SR.n_lexical.sum())}, "
          f"tasks where the whole-issue query returned nothing {int((SR.n_issue == 0).sum())}"]
(OUT / "summary.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))


In [ ]:
%%writefile {W}/analysis/skill_eval.py
"""Score the localize-code skill's offline rankings (Kaggle kernel kaggle/skill_eval) against AST gold v4.

Same definitions and bootstrap as analysis/c1_report.py: n = tasks with >=1 in-graph gold symbol (115),
hit@k = any gold symbol among the first k ranked symbols; symbol id = module path (src/ stripped) + qualname.
Also reports hit on ALL gold symbols (including async/nested ones the graph lacks) and file hit@k.
Run: uv run python analysis/skill_eval.py
"""
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
RANKS = ROOT / "work/c2/skill_eval/skill_ranks.csv"
TIMING = ROOT / "work/c2/skill_eval/skill_timing.csv"
g = pd.read_csv(ROOT / "work/c1/report/gold_mapped.csv")
sym = g[g.kind != "module"].drop_duplicates(["instance_id", "node"])
gold_in = sym[sym.in_graph == True].groupby("instance_id").node.agg(set)
gold_all = sym.groupby("instance_id").node.agg(set)
gold_files = g[g.kind != "module"].groupby("instance_id").file.agg(set)
tasks = sorted(gold_in.index)
n = len(tasks)
IDX = np.random.default_rng(0).integers(0, n, size=(10_000, n))


def module_of(path):
    p = path[4:] if path.startswith("src/") else path
    m = p[:-3].replace("/", ".")
    return m[: -len(".__init__")] if m.endswith(".__init__") else m


r = pd.read_csv(RANKS, keep_default_na=False)
rs = r[r.granularity == "symbol"].sort_values(["instance_id", "rank"])
rs = rs.assign(node=[f"{module_of(p)}.{q}" for p, q in zip(rs.path, rs.qualname)])
rank_sym = rs.groupby("instance_id").node.agg(list)
rank_file = r[r.granularity == "file"].sort_values(["instance_id", "rank"]).groupby("instance_id").path.agg(list)


def fmt(x):
    x = np.asarray(x, float)
    bs = x[IDX].mean(axis=1)
    return f"{int(x.sum())}/{len(x)} = {x.mean() * 100:.1f}% [{np.percentile(bs, 2.5) * 100:.1f}, {np.percentile(bs, 97.5) * 100:.1f}]"


print(f"n = {n} tasks (in-graph gold)")
for k in (5, 10, 15, 25, 50, 100):
    hin = [bool(set(rank_sym.get(t, [])[:k]) & gold_in[t]) for t in tasks]
    hall = [bool(set(rank_sym.get(t, [])[:k]) & gold_all[t]) for t in tasks]
    print(f"symbol hit@{k:3d}: in-graph gold {fmt(hin)} | any gold {fmt(hall)}")
for k in (1, 3, 5, 10):
    print(f"file hit@{k:2d}: {fmt([bool(set(rank_file.get(t, [])[:k]) & gold_files.get(t, set())) for t in tasks])}")
tm = pd.read_csv(TIMING)
print(f"runtime per task: median {tm.seconds.median():.2f}s, max {tm.seconds.max():.2f}s (Kaggle CPU); output chars at --top 100: max {tm.output_chars.max()}")


In [ ]:
%%writefile {W}/analysis/trace_metrics.py
"""Intermediate outcomes of one agent run, measured from the harness trace (C3).

Input: results/traces/trace_<id>.json written by swegemma (adk_eval_core SessionTrace.save, ATIF v1.7) and the task's
gold `patch` from tasks.jsonl. Standard library only; gold is recomputed from the patch with gold_core.

Definitions (step = 1-based ordinal of the tool call among all tool calls in the trace, in emission order):
- gold files: code files (gold_core.code_file) changed by the gold patch that exist in the base snapshot
  (a file whose only hunk is "@@ -0,0" is new and cannot be opened; it is excluded).
- gold lines: per gold file, the base-snapshot lines the patch removes, plus both neighbours of every pure insertion.
- mention: a gold file path appears in a tool response (grep/find output, graph tools, skill output, ...).
- open: a successful read_file on a gold file that returned content, or a successful read-like run_command (READ_CMD) whose
  command contains a gold path and that printed something. Shell output is not parsed further, so this measures access, not understanding.
- see lines: a read_file response on a gold file whose displayed range (start_line .. start_line + shown lines - 1)
  contains a gold line. Ranges refer to the file as it was read; reads after the agent edited that file are skipped.
- edit: a successful edit_file / write_file on a gold file. It may be undone later, so it does not imply a non-empty final patch.
- skill: run_skill_script with skill_name == "localize-code"; skill_hit = its first response mentions a gold file.
- skill_any: any skill tool (run_skill_script, load_skill, list_skills, load_skill_resource); ADK's own skill instruction
  tells the model to load_skill first (google/adk/tools/skill_toolset.py:61-88), so this counts engagement without a run.
- *_work_step: the same event counted in tool calls other than the skill tools (run_skill_script, load_skill, list_skills,
  load_skill_resource). The skill arm is told to call the skill first, so its raw step is >= 2 by construction; compare
  arms on *_work_step or on seconds (*_s), and use the raw step only within an arm.
Path match: tool paths are normalised by stripping "/workspace/" and "./"; run_command uses substring match
(so "cd rich && cat console.py" is not counted as an open; this undercounts opens via run_command only).
Run: uv run python analysis/trace_metrics.py <trace.json> <instance_id> [tasks.jsonl]
"""
import json
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
from gold_core import HUNK_RE, code_file, parse_patch  # noqa: E402

SKILL = "localize-code"
EDIT_TOOLS = ("edit_file", "write_file")
READ_CMD = re.compile(r"\b(cat|sed|head|tail|nl|awk|less|more|grep|rg)\b")
SKILL_TOOLS = ("run_skill_script", "load_skill", "list_skills", "load_skill_resource")


def flatten(atif):
    """ATIF v1.7 dict -> [{'step', 'tool', 'args', 'elapsed_s', 'result'}] in call order.

    The converter puts consecutive calls in one step's tool_calls[] but attaches only the first response to that step;
    later responses become separate system steps. Responses are therefore paired with calls FIFO per tool name.
    """
    calls, pending = [], {}
    for st in atif.get("steps", []):
        for tc in st.get("tool_calls") or []:
            ex = tc.get("extra") or {}
            c = {"step": len(calls) + 1, "tool": tc.get("function_name", ""), "args": tc.get("arguments") or {},
                 "elapsed_s": float(ex.get("elapsed_s", (st.get("extra") or {}).get("elapsed_s", 0.0)) or 0.0),
                 "result": None}
            calls.append(c)
            pending.setdefault(c["tool"], []).append(c)
        obs = st.get("observation")
        if obs:
            name = (obs.get("extra") or {}).get("tool_name", "")
            q = pending.get(name)
            if q:
                q.pop(0)["result"] = str(obs.get("content", ""))
    return calls


def norm(path):
    p = str(path or "").strip()
    for pre in ("/workspace/", "./"):
        if p.startswith(pre):
            p = p[len(pre):]
    return p


def gold_of(patch):
    """{gold file: set of base-snapshot gold lines} for existing code files changed by the patch."""
    files = parse_patch(patch)
    out = {}
    for path, hunks in files.items():
        if not code_file(path) or not hunks:
            continue
        if all(HUNK_RE.match(h["header"]).group(1) == "0" for h in hunks):
            continue  # new file
        lines = set()
        for h in hunks:
            for b in h["blocks"]:
                lines.update(b["removed"])
                if not b["removed"]:
                    lines.update((b["after"], b["after"] + 1))
        out[path] = lines
    return out


def shown_range(result):
    try:
        r = json.loads(result)
    except (TypeError, ValueError):
        return None
    if not isinstance(r, dict) or r.get("status") != "ok" or "start_line" not in r:
        return None
    n = len(str(r.get("content", "")).splitlines())
    return (int(r["start_line"]), int(r["start_line"]) + n - 1) if n else None


def succeeded(result):
    """True when a tool response is JSON with status ok; failed and malformed calls are not counted as events."""
    try:
        d = json.loads(result or "")
    except ValueError:
        return False
    return isinstance(d, dict) and d.get("status") == "ok"


def has_output(result, key):
    """True when a successful tool response has non-blank text under `key` (content of read_file, stdout of run_command)."""
    try:
        return bool(str(json.loads(result or "").get(key) or "").strip())
    except (ValueError, AttributeError):
        return False


def metrics(atif, patch):
    gold = gold_of(patch)
    calls = flatten(atif)
    first = {k: None for k in ("mention", "open", "see_lines", "edit", "skill", "skill_any")}
    edited, by_tool, skill_hit, n_skill = set(), {}, None, 0

    def mark(key, c):
        if first[key] is None:
            first[key] = (c["step"], c["step"] - n_skill, round(c["elapsed_s"], 1))

    for c in calls:
        t, a, res = c["tool"], c["args"], c["result"] or ""
        n_skill += t in SKILL_TOOLS
        by_tool[t] = by_tool.get(t, 0) + 1
        path = norm(a.get("filepath"))
        ok = succeeded(res)
        if gold and any(g in res for g in gold):
            mark("mention", c)
        if t == "read_file" and path in gold and ok and has_output(res, "content"):
            mark("open", c)
            rng = shown_range(res)
            if rng and path not in edited and any(rng[0] <= ln <= rng[1] for ln in gold[path]):
                mark("see_lines", c)
        if t == "run_command" and ok and has_output(res, "stdout") and READ_CMD.search(str(a.get("command", ""))) and any(g in str(a.get("command", "")) for g in gold):
            mark("open", c)
        if t in EDIT_TOOLS and ok:
            if path in gold:
                mark("edit", c)
            edited.add(path)
        if t in SKILL_TOOLS:
            mark("skill_any", c)
        if t == "run_skill_script" and a.get("skill_name") == SKILL:
            if first["skill"] is None:
                skill_hit = bool(gold) and any(g in res for g in gold)
            mark("skill", c)
    row = {"n_gold_files": len(gold), "n_tool_calls": len(calls), "tool_counts": json.dumps(by_tool, sort_keys=True),
           "skill_hit": skill_hit, "edited_files": json.dumps(sorted(edited)),
           "total_tokens": (atif.get("final_metrics") or {}).get("total_tokens")}
    for k, v in first.items():
        row[f"first_{k}_step"], row[f"first_{k}_work_step"], row[f"first_{k}_s"] = v if v else (None, None, None)
    return row


if __name__ == "__main__":
    trace, iid = Path(sys.argv[1]), sys.argv[2]
    tasks = Path(sys.argv[3]) if len(sys.argv) > 3 else Path(__file__).resolve().parents[1] / "data/raw/tasks.jsonl"
    patch = next(json.loads(l)["patch"] for l in open(tasks) if json.loads(l)["instance_id"] == iid)
    print(json.dumps(metrics(json.loads(trace.read_text()), patch), indent=2))


In [ ]:
%%writefile {W}/analysis/c3_report.py
"""C3 report: paired comparison of two agent arms from colab/c3_run.py output.

Input: a run directory laid out like the Colab OUT folder: c3_runs.jsonl and <arm>/rep<k>/traces/trace_<id>.json
(a flat folder of trace_<id>_<arm>.json files, as in work/c3/pilot/, is also accepted). Gold comes from data/raw/tasks.jsonl.
Outcome definitions (per run):
- overflow: error contains "ContextWindowExceeded" (vLLM rejected prompt + max_output_tokens > max_model_len);
- timeout: error contains "session timeout"; tool_budget: tool_calls >= 100 (official budget); other_error: any other error;
- resolved: official verification (Colab environment); empty_patch: patch_chars == 0;
- lost_edit (printed as gold_edit_then_overflow): overflow and the trace shows an edit of a gold file before the failure.
  This is "work in progress was discarded", not "a correct fix was discarded";
- edit_arg_errors: tool responses with "mandatory input parameters are not present" (harness tool-call parsing failures).
Paired analysis over tasks with both arms present: difference in proportions (B - A) with a task bootstrap 95% CI
(10,000 draws, seed 0, as in C1/C2) and an exact McNemar test on discordant pairs; for seconds and tool calls, the median
of per-task differences (B - A) with a task bootstrap 95% CI.
Run: uv run python analysis/c3_report.py <run_dir> [armA armB] [--exclude id,id] [--rep 1]
With a second repetition present, also prints within-arm rep1-vs-rep2 discordance (run-to-run variation).
"""
import argparse
import csv
import json
import math
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from trace_metrics import flatten, metrics  # noqa: E402

ROOT = Path(__file__).resolve().parents[1]
BOOT, SEED = 10_000, 0


def load_csv(path):
    """results/c3_run_rows.csv (written by c3_export_rows.py): same rows as load(), no traces needed."""
    def val(k, v):
        if v == "":
            return None
        if v in ("True", "False"):
            return v == "True"
        return v if k in ("task", "arm") else (float(v) if k == "seconds" else int(v))
    rows = [{k: val(k, v) for k, v in r.items()} for r in csv.DictReader(open(path))]
    return [{k: v for k, v in r.items() if v is not None or k in ("seconds", "tool_calls", "first_open_step", "first_see_lines_step")} for r in rows]


def load(run_dir):
    if run_dir.suffix == ".csv":
        return load_csv(run_dir)
    runs = [json.loads(l) for l in open(run_dir / "c3_runs.jsonl") if l.strip()]
    patches = {json.loads(l)["instance_id"]: json.loads(l)["patch"] for l in open(ROOT / "data/raw/tasks.jsonl")}
    out = []
    for r in runs:
        cands = [run_dir / r["arm"] / f"rep{r['rep']}" / "traces" / f"trace_{r['task']}.json",
                 run_dir / f"trace_{r['task']}_{r['arm']}.json"]
        tp = next((p for p in cands if p.exists()), None)
        err = r.get("error") or r.get("harness_error") or ""
        row = dict(task=r["task"], arm=r["arm"], rep=r["rep"], resolved=bool(r.get("resolved")),
                   overflow="ContextWindowExceeded" in err, timeout="session timeout" in err,
                   tool_budget=(r.get("tool_calls") or 0) >= 100, empty_patch=(r.get("patch_chars") or 0) == 0,
                   seconds=r.get("agent_seconds"), tool_calls=r.get("tool_calls"), has_trace=tp is not None)
        row["other_error"] = bool(err) and not (row["overflow"] or row["timeout"])
        if tp:
            atif = json.loads(tp.read_text())
            m = metrics(atif, patches[r["task"]])
            row["lost_edit"] = row["overflow"] and m["first_edit_step"] is not None
            row["first_open_step"] = m["first_open_step"]
            row["first_see_lines_step"] = m["first_see_lines_step"]
            row["edit_arg_errors"] = sum("mandatory input parameters are not present" in (c["result"] or "")
                                         for c in flatten(atif))
        out.append(row)
    return out


def boot_diff(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    idx = np.random.default_rng(SEED).integers(0, len(a), size=(BOOT, len(a)))
    d = b[idx].mean(1) - a[idx].mean(1)
    return b.mean() - a.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)


def mcnemar_exact(a, b):
    n01 = sum(1 for x, y in zip(a, b) if not x and y)
    n10 = sum(1 for x, y in zip(a, b) if x and not y)
    n = n01 + n10
    if n == 0:
        return n01, n10, 1.0
    k = min(n01, n10)
    p = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return n01, n10, min(1.0, 2 * p)


def discord(x, y):
    """(n, number of discordant pairs, rate, task-bootstrap 95% CI of the rate)."""
    d = np.array([bool(a) != bool(b) for a, b in zip(x, y)], float)
    idx = np.random.default_rng(SEED).integers(0, len(d), size=(BOOT, len(d)))
    bm = d[idx].mean(1)
    return len(d), int(d.sum()), d.mean(), np.percentile(bm, 2.5), np.percentile(bm, 97.5)


def main(argv):
    ap = argparse.ArgumentParser()
    ap.add_argument("run_dir", type=Path)
    ap.add_argument("arms", nargs="*")
    ap.add_argument("--exclude", default="")
    ap.add_argument("--rep", type=int, default=1, help="repetition used for the between-arm comparison")
    a_ = ap.parse_args(argv)
    exclude = set(filter(None, a_.exclude.split(",")))
    all_rows = [r for r in load(a_.run_dir) if r["task"] not in exclude]
    A, B = a_.arms[:2] if len(a_.arms) >= 2 else sorted({r["arm"] for r in all_rows})
    by = {(r["task"], r["rep"], r["arm"]): r for r in all_rows}
    keys = sorted({(t, k) for t, k, _ in by if k == a_.rep and (t, k, A) in by and (t, k, B) in by})
    print(f"run dir {a_.run_dir}; arms A={A} B={B}; rep {a_.rep}; paired tasks = {len(keys)}; "
          f"traces present {sum(r['has_trace'] for r in all_rows)}/{len(all_rows)}; excluded {sorted(exclude)}")
    if not keys:
        return 1
    print(f"\n{'outcome':16} {A:>12} {B:>12}   diff B-A [95% CI]        discordant(A0B1,A1B0)  McNemar p")
    for o in ("resolved", "overflow", "timeout", "tool_budget", "other_error", "empty_patch", "lost_edit"):
        if any(o not in by[(t, k, arm)] for t, k in keys for arm in (A, B)):
            continue
        a = [bool(by[(t, k, A)][o]) for t, k in keys]
        b = [bool(by[(t, k, B)][o]) for t, k in keys]
        d, lo, hi = boot_diff(a, b)
        n01, n10, p = mcnemar_exact(a, b)
        label = "gold_edit_then_overflow" if o == "lost_edit" else o
        print(f"{label:16} {sum(a):>5}/{len(a):<6} {sum(b):>5}/{len(b):<6}   {d:+.3f} [{lo:+.3f}, {hi:+.3f}]"
              f"      ({n01}, {n10})              {p:.3f}")
    for o in ("seconds", "tool_calls", "first_open_step", "first_see_lines_step", "edit_arg_errors"):
        va = [by[(t, k, A)].get(o) for t, k in keys]
        vb = [by[(t, k, B)].get(o) for t, k in keys]
        pa = [x for x in va if x is not None]
        pb = [x for x in vb if x is not None]
        if pa and pb:
            line = f"{o:16} median {np.median(pa):>8.1f} {np.median(pb):>8.1f}   (n {len(pa)}, {len(pb)})"
            d = np.array([y - x for x, y in zip(va, vb) if x is not None and y is not None], float)
            if o in ("seconds", "tool_calls") and len(d):
                idx = np.random.default_rng(SEED).integers(0, len(d), size=(BOOT, len(d)))
                bm = np.median(d[idx], axis=1)
                line += (f"   paired median diff B-A {np.median(d):+.1f} "
                         f"[{np.percentile(bm, 2.5):+.1f}, {np.percentile(bm, 97.5):+.1f}]")
            print(line)

    # Repeatability: same arm, rep 1 vs rep 2, compared with between-arm discordance (rep 1) on the same tasks.
    rep_tasks = sorted({t for t, k, _ in by if k == 2 and all((t, r, arm) in by for r in (1, 2) for arm in (A, B))})
    if rep_tasks:
        print(f"\nrepeatability on {len(rep_tasks)} tasks with both reps of both arms "
              f"(discordant runs / tasks, rate [95% CI])")
        for o in ("resolved", "overflow"):
            cells = [(f"{A} rep1 vs rep2", [by[(t, 1, A)][o] for t in rep_tasks], [by[(t, 2, A)][o] for t in rep_tasks]),
                     (f"{B} rep1 vs rep2", [by[(t, 1, B)][o] for t in rep_tasks], [by[(t, 2, B)][o] for t in rep_tasks]),
                     (f"{A} vs {B} (rep1)", [by[(t, 1, A)][o] for t in rep_tasks], [by[(t, 1, B)][o] for t in rep_tasks]),
                     (f"{A} vs {B} (rep2)", [by[(t, 2, A)][o] for t in rep_tasks], [by[(t, 2, B)][o] for t in rep_tasks])]
            for name, x, y in cells:
                n, k, rate, lo, hi = discord(x, y)
                print(f"  {o:9} {name:24} {k:>3}/{n:<4} {rate:.3f} [{lo:.3f}, {hi:.3f}]")
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv[1:]))


In [ ]:
%%writefile {W}/analysis/c3_fig_prompt.py
"""C3 Figure 1: maximum prompt tokens per run, by arm.

Per run, the maximum `metrics.prompt_tokens` over the agent steps of the ATIF trace. An overflowing base run never records the
rejected request, so its maximum is the largest prompt that succeeded. Prints the summary quoted in reports/c3_results.md and writes the PNG.

    uv run python analysis/c3_fig_prompt.py work/c3/main/g4swe_c3_main paper/fig1_max_prompt.png
"""
import csv
import json
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

ARMS = ["base", "out8k"]
LINES = [(14_336, "compaction\nthreshold", "right"), (16_384, "base prompt\nceiling", "left"),
         (24_576, "out8k prompt\nceiling", "left")]
COLOR = {"base": "#2a78d6", "out8k": "#eb6834"}  # dataviz reference palette, slots 1-2 (light)
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"


def load(run_dir):
    if run_dir.suffix == ".csv":  # results/c3_run_rows.csv, no traces needed
        return [dict(task=r["task"], arm=r["arm"], max_prompt=int(r["max_prompt"]), overflow=r["overflow"] == "True")
                for r in csv.DictReader(open(run_dir)) if r["rep"] == "1"]
    rows = []
    for line in open(run_dir / "c3_runs.jsonl"):
        r = json.loads(line)
        if r["rep"] != 1:  # the figure shows the main run only; repetition 2 is a calibration on 40 tasks
            continue
        trace = json.loads((run_dir / r["arm"] / f"rep{r['rep']}" / "traces" / f"trace_{r['task']}.json").read_text())
        prompts = [s["metrics"]["prompt_tokens"] for s in trace["steps"] if s.get("source") == "agent" and s.get("metrics")]
        rows.append(dict(task=r["task"], arm=r["arm"], max_prompt=max(prompts),
                         overflow="ContextWindowExceeded" in (r.get("error") or "")))
    return rows


def main(run_dir, out_png):
    rows = load(Path(run_dir))
    for arm in ARMS:
        v = np.array([r["max_prompt"] for r in rows if r["arm"] == arm])
        print(f"{arm}: n {len(v)}  median {np.median(v):,.0f}  p95 {np.percentile(v, 95):,.0f}  max {v.max():,}  "
              f"> 16,384: {(v > 16_384).sum()}  > 24,576: {(v > 24_576).sum()}")

    rng = np.random.default_rng(0)
    fig, ax = plt.subplots(figsize=(8, 3.3), dpi=200)
    fig.patch.set_facecolor(SURFACE)
    ax.set_facecolor(SURFACE)
    for x, label, side in LINES:
        ax.axvline(x, color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)
        ax.text(x + (150 if side == "left" else -150), 1.62, label, color=MUTED, fontsize=8, va="top",
                ha="left" if side == "left" else "right")
    for i, arm in enumerate(ARMS):
        sub = [r for r in rows if r["arm"] == arm]
        x = np.array([r["max_prompt"] for r in sub])
        y = i + rng.uniform(-0.22, 0.22, len(sub))
        ovf = np.array([r["overflow"] for r in sub])
        ax.scatter(x[~ovf], y[~ovf], s=22, color=COLOR[arm], edgecolor=SURFACE, linewidth=0.6, zorder=3,
                   label="base, run finished" if arm == "base" else "out8k (no run overflowed)")
        if ovf.any():
            ax.scatter(x[ovf], y[ovf], s=22, facecolor=SURFACE, edgecolor=COLOR[arm], linewidth=1.2, zorder=3,
                       label=f"base, overflow: next request rejected (n = {ovf.sum()})")
    ax.set_yticks(range(len(ARMS)), [f"{a}\n(max_output {m})" for a, m in zip(ARMS, ["16,384", "8,192"])], color=INK, fontsize=9)
    ax.set_ylim(-0.5, 1.7)
    ax.set_xlim(4_000, 26_000)
    ax.set_xticks(range(4_000, 26_001, 4_000), [f"{v // 1000}k" for v in range(4_000, 26_001, 4_000)])
    ax.set_xlabel("Maximum prompt tokens in a run (one dot per run, n = 120 per arm)", color=INK, fontsize=9)
    ax.tick_params(colors=MUTED, labelsize=8, length=0)
    ax.grid(axis="x", color=GRID, lw=0.6, zorder=0)
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.legend(loc="lower left", fontsize=7.5, frameon=False, labelcolor=INK, ncol=3, bbox_to_anchor=(0.0, 1.0),
              handletextpad=0.3, columnspacing=1.5)
    fig.suptitle("Halving max_output_tokens lets prompts grow past the old ceiling, never near the new one",
                 color=INK, fontsize=10, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(out_png, facecolor=SURFACE)
    print(f"wrote {out_png}")


if __name__ == "__main__":
    main(*sys.argv[1:3])


In [ ]:
%%writefile {W}/analysis/c3_overflow_edits.py
"""Overflowing base runs of the C3 main run (rep 1): what 0.2.11's working-tree recovery would have had to submit.

The patch itself was lost in 0.2.7, so this only counts successful edit-tool calls in the trace; shell edits (sed) and later reverts are not tracked,
so it does not measure non-empty recoverable patches.
Run: uv run python analysis/c3_overflow_edits.py [results/c3_run_rows.csv]
"""
import csv
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "analysis"))
from trace_metrics import EDIT_TOOLS, flatten, metrics, succeeded  # noqa: E402

if len(sys.argv) > 1:  # published per-run table instead of traces
    rows = [r for r in csv.DictReader(open(sys.argv[1])) if r["rep"] == "1" and r["arm"] == "base" and r["overflow"] == "True"]
    print(f"overflow runs {len(rows)}; empty patch {sum(r['empty_patch'] == 'True' for r in rows)}; "
          f"successful edit-tool call {sum(r['edit_succeeded'] == 'True' for r in rows)}; "
          f"successfully edited a gold file {sum(r['lost_edit'] == 'True' for r in rows)}")
    sys.exit(0)

RUN = ROOT / "work/c3/main/g4swe_c3_main"
rows = [json.loads(line) for line in open(RUN / "c3_runs.jsonl")]
patches = {t["instance_id"]: t["patch"] for t in map(json.loads, open(ROOT / "data/raw/tasks.jsonl"))}
over = [r for r in rows if str(r["rep"]) == "1" and r["arm"] == "base" and "ContextWindowExceededError" in str(r["error"])]
prefix = sum(str(r["error"]).startswith("Sandbox execution error:") for r in over)
empty = sum(int(r["patch_chars"]) == 0 for r in over)
any_edit = gold_edit = 0
for r in over:
    atif = json.load(open(RUN / f"base/rep1/traces/trace_{r['task']}.json"))
    any_edit += any(c["tool"] in EDIT_TOOLS and succeeded(c["result"]) for c in flatten(atif))
    gold_edit += metrics(atif, patches[r["task"]])["first_edit_step"] is not None
print(f"overflow runs {len(over)}; 'Sandbox execution error:' prefix {prefix}; empty patch {empty}; "
      f"successful edit-tool call {any_edit}; successfully edited a gold file {gold_edit}")


In [ ]:
%%writefile {W}/kaggle/gold_ast/gold_ast.py
"""gold_ast — generated: analysis/gold_core.py + kaggle/gold_ast/main_part.py (do not edit by hand)."""
"""Gold-location core shared by the Kaggle kernel (kaggle/gold_ast) and local regression tests.

attribute(patch_hunks, src) maps every changed old-file line of a code file to the innermost
enclosing def/class of the base snapshot (ast).

Rules (v2, after Codex audit A01/A11):
- A hunk is split into change blocks: maximal runs of '-'/'+' lines between context lines.
- Replacement block (has '-' lines): attributed to the scopes of its removed old lines only.
  Its '+' lines are NOT treated as insertions (A01 fix).
- Pure insertion block (only '+'), inserted between old lines a and a+1 (v3, audit 2 N01/N02):
  * decorator-only block (only '@' lines and continuation, no def/class) placed right before an existing scope T
    whose header column equals the decorator indent -> owner = T (the existing definition gains a decorator);
  * otherwise owner = innermost scope S that lexically encloses the gap and whose body column the new code sits in:
    S.start <= a' <= S.end, where a' = last non-blank old line <= a (trailing blank lines after a scope still belong to it),
    and indent(first non-blank '+' line) > S.col. A new method -> its class; a new top-level def/class -> '<module>'.
- '\\ No newline at end of file' lines are ignored; '+++ /dev/null' (deleted files) are skipped.
- Whitespace-only blocks (every removed old line and every added line is blank) are formatting, not gold: skipped,
  unless they lie inside a multi-line string literal (tokenize), where blank lines are part of the value (N06).
"""
import ast
import io
import re
import tokenize

HUNK_RE = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")
TEST_RE = re.compile(r"(^|/)tests?/|(^|/)test_[^/]*$|_test\.py$")


def code_file(path):
    return path.endswith(".py") and not TEST_RE.search(path) and not path.startswith(("docs/", "docs_src/"))


def parse_patch(patch):
    """{path: [hunk]}; hunk = {'old_start', 'header', 'blocks': [{'removed': [old line nos], 'added': [text],
    'after': old line no. preceding the block}]}"""
    files, cur, hunk, old, block = {}, None, None, 0, None
    for line in patch.splitlines():
        if line.startswith("--- "):
            continue
        if line.startswith("+++ "):
            p = line[4:].strip()
            cur = None if p == "/dev/null" else (p[2:] if p.startswith("b/") else p)
            if cur is not None:
                files.setdefault(cur, [])
            hunk = None
            continue
        if cur is None:
            continue
        m = HUNK_RE.match(line)
        if m:
            old = int(m.group(1)) if (m.group(2) is None or m.group(2) != "0") else int(m.group(1)) + 1
            hunk = {"old_start": int(m.group(1)), "header": line, "blocks": []}
            files[cur].append(hunk)
            block = None
            continue
        if hunk is None or line.startswith("\\"):
            continue
        if line.startswith("-") or line.startswith("+"):
            if block is None:
                block = {"removed": [], "added": [], "after": old - 1}
                hunk["blocks"].append(block)
            if line.startswith("-"):
                block["removed"].append(old)
                old += 1
            else:
                block["added"].append(line[1:])
        else:
            block = None
            old += 1
    return files


def scopes(src):
    """[(start, end, qualname, kind, col)] for every def/class; start includes decorators."""
    tree = ast.parse(src)
    out = []

    def walk(node, prefix):
        for ch in ast.iter_child_nodes(node):
            if isinstance(ch, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                q = f"{prefix}.{ch.name}" if prefix else ch.name
                start = min([ch.lineno] + [d.lineno for d in ch.decorator_list])
                kind = {ast.FunctionDef: "def", ast.AsyncFunctionDef: "async_def", ast.ClassDef: "class"}[type(ch)]
                out.append((start, ch.end_lineno, q, kind, ch.col_offset))
                walk(ch, q)
            else:
                walk(ch, prefix)
    walk(tree, "")
    return out


def innermost(sc, line):
    best = None
    for s in sc:
        if s[0] <= line <= s[1] and (best is None or (s[1] - s[0]) < (best[1] - best[0])):
            best = s
    return best


def _indent(text):
    return len(text) - len(text.lstrip(" \t"))


DEF_RE = re.compile(r"\s*(async\s+def|def|class)\b")


def insertion_owner(sc, a, added, lines):
    nonblank = [t for t in added if t.strip()]
    if not nonblank:
        return None
    ind = _indent(nonblank[0])
    top = [t for t in nonblank if _indent(t) == ind]
    decorator_only = top[0].lstrip().startswith("@") and not any(DEF_RE.match(t) for t in top)
    if decorator_only:
        nxt = [x for x in sc if x[0] == a + 1 and x[4] == ind]
        if nxt:
            return min(nxt, key=lambda x: x[1] - x[0])
    a2 = a
    while a2 > 0 and a2 <= len(lines) and not lines[a2 - 1].strip():
        a2 -= 1
    cands = [x for x in sc if x[0] <= a2 <= x[1] and ind > x[4]]
    return min(cands, key=lambda x: x[1] - x[0]) if cands else None


def string_continuation_lines(src):
    """Old line numbers that lie inside a multi-line string literal (after its first line)."""
    out = set()
    try:
        for tok in tokenize.generate_tokens(io.StringIO(src).readline):
            if tok.type == tokenize.STRING and tok.end[0] > tok.start[0]:
                out.update(range(tok.start[0] + 1, tok.end[0] + 1))
    except (tokenize.TokenError, IndentationError, SyntaxError):
        pass
    return out


def attribute(hunks, src):
    """Yield (hunk_index, qualname | '<module>', kind, line, block_type)."""
    sc = scopes(src)
    lines = src.splitlines()
    in_string = string_continuation_lines(src)
    for hi, h in enumerate(hunks):
        seen = set()
        for b in h["blocks"]:
            removed_text = [lines[ln - 1] if 0 < ln <= len(lines) else "x" for ln in b["removed"]]
            if all(not t.strip() for t in removed_text + b["added"]):
                inside = any(ln in in_string for ln in b["removed"]) or (not b["removed"] and (b["after"] + 1) in in_string)
                if not inside:
                    continue
                s_ = innermost(sc, b["removed"][0] if b["removed"] else b["after"] + 1)
                key = s_[2] if s_ else "<module>"
                if key not in seen:
                    seen.add(key)
                    yield hi, key, (s_[3] if s_ else "module"), (b["removed"] or [b["after"]])[0], "string_ws"
                continue
            if b["removed"]:
                for ln in b["removed"]:
                    s = innermost(sc, ln)
                    key = s[2] if s else "<module>"
                    if key not in seen:
                        seen.add(key)
                        yield hi, key, (s[3] if s else "module"), ln, "replace"
            else:
                s = insertion_owner(sc, b["after"], b["added"], lines)
                key = s[2] if s else "<module>"
                if key not in seen:
                    seen.add(key)
                    yield hi, key, (s[3] if s else "module"), b["after"], "insert"


# ===== Kaggle main (gold_ast, core v3 rules) =====
import csv, glob, json, tarfile
from pathlib import Path

ROOT = Path(sorted(glob.glob("/kaggle/input/**/gemma-4-developer-agent/tasks.jsonl", recursive=True))[0]).parent
SNAP, OUT = ROOT / "snapshots", Path("/kaggle/working")
TASKS = [json.loads(l) for l in open(ROOT / "tasks.jsonl")]
print("root", ROOT, "tasks", len(TASKS), "snapshots", len(list(SNAP.glob("*.tgz"))))


def read_member(names, tf, path):
    cands = sorted([n for n in names if n == path or n.endswith("/" + path)], key=len)
    return tf.extractfile(cands[0]).read().decode("utf-8", errors="replace") if cands else None


rows, filerows = [], []
for t in TASKS:
    tid = t["instance_id"]
    files = parse_patch(t["patch"])
    with tarfile.open(SNAP / f"{tid}.tgz") as tf:
        names = tf.getnames()
        for path, hunks in files.items():
            if not code_file(path):
                continue
            src = read_member(names, tf, path)
            if src is None:
                filerows.append(dict(instance_id=tid, file=path, exists=False, parses=None, n_async=None, n_def=None))
                continue
            try:
                sc = scopes(src)
                ok = True
            except SyntaxError:
                sc, ok = [], False
            filerows.append(dict(instance_id=tid, file=path, exists=True, parses=ok,
                                 n_async=sum(s[3] == "async_def" for s in sc), n_def=len(sc)))
            if not ok:
                continue
            for hi, q, kind, line, btype in attribute(hunks, src):
                rows.append(dict(instance_id=tid, repo=t["repo"], file=path, hunk_index=hi, qualname=q,
                                 kind=kind, line=line, block_type=btype))

for name, data in [("gold_ast.csv", rows), ("gold_files.csv", filerows)]:
    with open(OUT / name, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(data[0].keys()))
        w.writeheader()
        w.writerows(data)
print("gold rows", len(rows), "files", len(filerows),
      "missing files", sum(not r["exists"] for r in filerows),
      "parse failures", sum(r["parses"] is False for r in filerows))


In [ ]:
%%writefile {W}/kaggle/bm25/bm25.py
"""Lexical baseline (v3: .git/HEAD root, decorators in spans, ids merged, library ranking top-200): BM25 from problem_statement to code units on base_commit snapshots.

Units: (1) every def/class span parsed with ast (async included, nested included, innermost spans
are separate units), (2) whole files. Corpus = non-test, non-docs .py files of the snapshot.
Output: ranked top-100 units per task for both granularities (no gold used here).
"""
import ast, csv, glob, json, math, re, tarfile, collections
from pathlib import Path

ROOT = Path(sorted(glob.glob("/kaggle/input/**/gemma-4-developer-agent/tasks.jsonl", recursive=True))[0]).parent
SNAP, OUT = ROOT / "snapshots", Path("/kaggle/working")
TASKS = [json.loads(l) for l in open(ROOT / "tasks.jsonl")]
TEST_RE = re.compile(r"(^|/)tests?/|(^|/)test_[^/]*$|_test\.py$")
PKG = {"fastapi/fastapi": "fastapi", "Textualize/rich": "rich", "psf/requests": "requests", "encode/httpx": "httpx"}
TOK = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
STOP = set("the a an and or of to in for is are be this that with on as it by not if from at we can should when use using".split())


def toks(s):
    out = []
    for w in TOK.findall(s):
        parts = [w] + [p for p in re.split(r"_|(?<=[a-z])(?=[A-Z])", w) if p]
        out += [p.lower() for p in parts if len(p) > 1]
    return [t for t in out if t not in STOP]


def units(path, src):
    mod = (path[4:] if path.startswith("src/") else path)[:-3].replace("/", ".")
    mod = mod[:-9] if mod.endswith(".__init__") else mod
    lines = src.splitlines()
    res = [(f"FILE:{path}", path, src)]
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return res

    def walk(node, prefix):
        for ch in ast.iter_child_nodes(node):
            if isinstance(ch, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                q = f"{prefix}.{ch.name}" if prefix else ch.name
                start = min([ch.lineno] + [d.lineno for d in ch.decorator_list])
                body = "\n".join(lines[start - 1: ch.end_lineno])
                res.append((f"{mod}.{q}", path, body))
                walk(ch, q)
            else:
                walk(ch, prefix)
    walk(tree, "")
    return res


def bm25_rank(query, docs, k1=1.2, b=0.75, top=100):
    tf = [collections.Counter(toks(d)) for d in docs]
    N = len(docs); avgdl = sum(sum(c.values()) for c in tf) / max(N, 1)
    df = collections.Counter(t for c in tf for t in c)
    q = collections.Counter(toks(query))
    scores = []
    for i, c in enumerate(tf):
        dl = sum(c.values()); s = 0.0
        for t, qf in q.items():
            if t in c:
                idf = math.log(1 + (N - df[t] + 0.5) / (df[t] + 0.5))
                s += idf * c[t] * (k1 + 1) / (c[t] + k1 * (1 - b + b * dl / avgdl))
        scores.append(s)
    order = sorted(range(N), key=lambda i: (-scores[i], i))[:top]
    return [(i, scores[i]) for i in order]


rows = []
prefixes = collections.Counter()
for t in TASKS:
    with tarfile.open(SNAP / f"{t['instance_id']}.tgz") as tf:
        mem = [m for m in tf.getmembers() if m.isfile() and m.name.endswith(".py")]
        # repository root = directory holding .git/HEAD (v2: no guessing from a shared first folder, audit A11)
        heads = sorted((m.name for m in tf.getmembers() if m.name == ".git/HEAD" or m.name.endswith("/.git/HEAD")), key=len)
        if not heads:
            raise RuntimeError(f"no .git/HEAD in {t['instance_id']}")
        prefix = heads[0][: -len(".git/HEAD")]
        prefixes[prefix] += 1
        corpus = []
        for m in mem:
            rel = m.name[len(prefix):]
            if rel.startswith(".git/") or TEST_RE.search(rel) or rel.startswith(("docs/", "docs_src/")):
                continue
            src = tf.extractfile(m).read().decode("utf-8", errors="replace")
            corpus += units(rel, src)
    # merge units sharing an id (overloads / redefinitions) so each id occupies one rank (audit A07)
    merged = {}
    for uid, path, text in corpus:
        if uid in merged:
            merged[uid] = (uid, merged[uid][1], merged[uid][2] + "\n" + text)
        else:
            merged[uid] = (uid, path, text)
    corpus = list(merged.values())
    query = t["problem_statement"]
    pkg = PKG[t["repo"]] + "."
    for gran in ("file", "symbol", "symbol_lib"):
        if gran == "symbol_lib":  # library-package units only, filtered BEFORE truncation (audit 2 N04)
            idx = [i for i, u in enumerate(corpus) if not u[0].startswith("FILE:") and u[0].startswith(pkg)]
        else:
            idx = [i for i, u in enumerate(corpus) if (u[0].startswith("FILE:")) == (gran == "file")]
        ranked = bm25_rank(query, [corpus[i][2] for i in idx], top=200 if gran == "symbol_lib" else 100)
        for r, (j, s) in enumerate(ranked, 1):
            u = corpus[idx[j]]
            rows.append(dict(instance_id=t["instance_id"], granularity=gran, rank=r, unit=u[0], file=u[1], score=round(s, 4)))
    print(t["instance_id"], len(corpus), flush=True)

with open(OUT / "bm25_ranks.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print("rows", len(rows), "archive prefixes", dict(prefixes))


In [ ]:
%%writefile {W}/kaggle/gold_mutation/gold_mutation.py
"""gold mutation test — generated: analysis/gold_core.py + kaggle/gold_mutation/main_part.py"""
"""Gold-location core shared by the Kaggle kernel (kaggle/gold_ast) and local regression tests.

attribute(patch_hunks, src) maps every changed old-file line of a code file to the innermost
enclosing def/class of the base snapshot (ast).

Rules (v2, after Codex audit A01/A11):
- A hunk is split into change blocks: maximal runs of '-'/'+' lines between context lines.
- Replacement block (has '-' lines): attributed to the scopes of its removed old lines only.
  Its '+' lines are NOT treated as insertions (A01 fix).
- Pure insertion block (only '+'), inserted between old lines a and a+1 (v3, audit 2 N01/N02):
  * decorator-only block (only '@' lines and continuation, no def/class) placed right before an existing scope T
    whose header column equals the decorator indent -> owner = T (the existing definition gains a decorator);
  * otherwise owner = innermost scope S that lexically encloses the gap and whose body column the new code sits in:
    S.start <= a' <= S.end, where a' = last non-blank old line <= a (trailing blank lines after a scope still belong to it),
    and indent(first non-blank '+' line) > S.col. A new method -> its class; a new top-level def/class -> '<module>'.
- '\\ No newline at end of file' lines are ignored; '+++ /dev/null' (deleted files) are skipped.
- Whitespace-only blocks (every removed old line and every added line is blank) are formatting, not gold: skipped,
  unless they lie inside a multi-line string literal (tokenize), where blank lines are part of the value (N06).
"""
import ast
import io
import re
import tokenize

HUNK_RE = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")
TEST_RE = re.compile(r"(^|/)tests?/|(^|/)test_[^/]*$|_test\.py$")


def code_file(path):
    return path.endswith(".py") and not TEST_RE.search(path) and not path.startswith(("docs/", "docs_src/"))


def parse_patch(patch):
    """{path: [hunk]}; hunk = {'old_start', 'header', 'blocks': [{'removed': [old line nos], 'added': [text],
    'after': old line no. preceding the block}]}"""
    files, cur, hunk, old, block = {}, None, None, 0, None
    for line in patch.splitlines():
        if line.startswith("--- "):
            continue
        if line.startswith("+++ "):
            p = line[4:].strip()
            cur = None if p == "/dev/null" else (p[2:] if p.startswith("b/") else p)
            if cur is not None:
                files.setdefault(cur, [])
            hunk = None
            continue
        if cur is None:
            continue
        m = HUNK_RE.match(line)
        if m:
            old = int(m.group(1)) if (m.group(2) is None or m.group(2) != "0") else int(m.group(1)) + 1
            hunk = {"old_start": int(m.group(1)), "header": line, "blocks": []}
            files[cur].append(hunk)
            block = None
            continue
        if hunk is None or line.startswith("\\"):
            continue
        if line.startswith("-") or line.startswith("+"):
            if block is None:
                block = {"removed": [], "added": [], "after": old - 1}
                hunk["blocks"].append(block)
            if line.startswith("-"):
                block["removed"].append(old)
                old += 1
            else:
                block["added"].append(line[1:])
        else:
            block = None
            old += 1
    return files


def scopes(src):
    """[(start, end, qualname, kind, col)] for every def/class; start includes decorators."""
    tree = ast.parse(src)
    out = []

    def walk(node, prefix):
        for ch in ast.iter_child_nodes(node):
            if isinstance(ch, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                q = f"{prefix}.{ch.name}" if prefix else ch.name
                start = min([ch.lineno] + [d.lineno for d in ch.decorator_list])
                kind = {ast.FunctionDef: "def", ast.AsyncFunctionDef: "async_def", ast.ClassDef: "class"}[type(ch)]
                out.append((start, ch.end_lineno, q, kind, ch.col_offset))
                walk(ch, q)
            else:
                walk(ch, prefix)
    walk(tree, "")
    return out


def innermost(sc, line):
    best = None
    for s in sc:
        if s[0] <= line <= s[1] and (best is None or (s[1] - s[0]) < (best[1] - best[0])):
            best = s
    return best


def _indent(text):
    return len(text) - len(text.lstrip(" \t"))


DEF_RE = re.compile(r"\s*(async\s+def|def|class)\b")


def insertion_owner(sc, a, added, lines):
    nonblank = [t for t in added if t.strip()]
    if not nonblank:
        return None
    ind = _indent(nonblank[0])
    top = [t for t in nonblank if _indent(t) == ind]
    decorator_only = top[0].lstrip().startswith("@") and not any(DEF_RE.match(t) for t in top)
    if decorator_only:
        nxt = [x for x in sc if x[0] == a + 1 and x[4] == ind]
        if nxt:
            return min(nxt, key=lambda x: x[1] - x[0])
    a2 = a
    while a2 > 0 and a2 <= len(lines) and not lines[a2 - 1].strip():
        a2 -= 1
    cands = [x for x in sc if x[0] <= a2 <= x[1] and ind > x[4]]
    return min(cands, key=lambda x: x[1] - x[0]) if cands else None


def string_continuation_lines(src):
    """Old line numbers that lie inside a multi-line string literal (after its first line)."""
    out = set()
    try:
        for tok in tokenize.generate_tokens(io.StringIO(src).readline):
            if tok.type == tokenize.STRING and tok.end[0] > tok.start[0]:
                out.update(range(tok.start[0] + 1, tok.end[0] + 1))
    except (tokenize.TokenError, IndentationError, SyntaxError):
        pass
    return out


def attribute(hunks, src):
    """Yield (hunk_index, qualname | '<module>', kind, line, block_type)."""
    sc = scopes(src)
    lines = src.splitlines()
    in_string = string_continuation_lines(src)
    for hi, h in enumerate(hunks):
        seen = set()
        for b in h["blocks"]:
            removed_text = [lines[ln - 1] if 0 < ln <= len(lines) else "x" for ln in b["removed"]]
            if all(not t.strip() for t in removed_text + b["added"]):
                inside = any(ln in in_string for ln in b["removed"]) or (not b["removed"] and (b["after"] + 1) in in_string)
                if not inside:
                    continue
                s_ = innermost(sc, b["removed"][0] if b["removed"] else b["after"] + 1)
                key = s_[2] if s_ else "<module>"
                if key not in seen:
                    seen.add(key)
                    yield hi, key, (s_[3] if s_ else "module"), (b["removed"] or [b["after"]])[0], "string_ws"
                continue
            if b["removed"]:
                for ln in b["removed"]:
                    s = innermost(sc, ln)
                    key = s[2] if s else "<module>"
                    if key not in seen:
                        seen.add(key)
                        yield hi, key, (s[3] if s else "module"), ln, "replace"
            else:
                s = insertion_owner(sc, b["after"], b["added"], lines)
                key = s[2] if s else "<module>"
                if key not in seen:
                    seen.add(key)
                    yield hi, key, (s[3] if s else "module"), b["after"], "insert"


# ===== Mutation test of gold attribution (model-independent check) =====
# Synthetic patches with KNOWN gold are generated from real base-snapshot library files, diffed with difflib
# (as a unified diff), and fed through parse_patch + attribute. Expected gold is computed with a separate ast
# parent walk (not gold_core.innermost).
import csv, difflib, glob, json, random, tarfile
from pathlib import Path

ROOT = Path(sorted(glob.glob("/kaggle/input/**/gemma-4-developer-agent/tasks.jsonl", recursive=True))[0]).parent
SNAP, OUT = ROOT / "snapshots", Path("/kaggle/working")
TASKS = [json.loads(l) for l in open(ROOT / "tasks.jsonl")]
PKG = {"fastapi/fastapi": "fastapi/", "Textualize/rich": "rich/", "psf/requests": "src/requests/", "encode/httpx": "src/httpx/"}
rng = random.Random(0)


def defs_with_parents(src):
    """[(node, qualname, parent_qualname or None)] via an explicit parent map."""
    tree = ast.parse(src)
    parent = {}
    for p in ast.walk(tree):
        for c in ast.iter_child_nodes(p):
            parent[c] = p
    out = []
    for n in ast.walk(tree):
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            chain, p = [n.name], parent.get(n)
            while p is not None:
                if isinstance(p, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                    chain.append(p.name)
                p = parent.get(p)
            out.append((n, ".".join(reversed(chain))))
    return out


def owner_of_line(src, line):
    """Expected owner of an existing line: the deepest def/class whose [first decorator, end] contains it."""
    best = None
    for n, q in defs_with_parents(src):
        s = min([n.lineno] + [d.lineno for d in n.decorator_list])
        if s <= line <= n.end_lineno and (best is None or q.count(".") > best.count(".")):
            best = q
    return best or "<module>"


def patch_of(path, old, new):
    return "".join(difflib.unified_diff(old.splitlines(True), new.splitlines(True), f"a/{path}", f"b/{path}", n=3))


def run(path, old, new):
    files = parse_patch(patch_of(path, old, new))
    return sorted({q for _, q, *_ in attribute(files.get(path, []), old)}) if files else []


def mutations(path, src):
    L = src.splitlines(True)
    D = defs_with_parents(src)
    funcs = [(n, q) for n, q in D if not isinstance(n, ast.ClassDef) and len(n.body) >= 2]
    classes = [(n, q) for n, q in D if isinstance(n, ast.ClassDef)]
    strings = set()
    try:
        for tok in tokenize.generate_tokens(io.StringIO(src).readline):
            if tok.type == tokenize.STRING:
                strings.update(range(tok.start[0], tok.end[0] + 1))
    except Exception:
        return
    if funcs:
        n, q = rng.choice(funcs)
        body = [ln for ln in range(n.body[0].lineno, n.end_lineno + 1) if L[ln - 1].strip() and ln not in strings
                and not L[ln - 1].lstrip().startswith(("@", "def ", "async def ", "class "))]
        if body:
            ln = rng.choice(body)
            exp = owner_of_line(src, ln)
            new = L[:]; new[ln - 1] = L[ln - 1].rstrip("\n") + "  # mutated\n"
            yield "replace_line", [exp], "".join(new)
            new = L[:ln - 1] + L[ln:]
            yield "delete_line", [exp], "".join(new)
            ind = " " * (len(L[ln - 1]) - len(L[ln - 1].lstrip()))
            new = L[:ln] + [ind + "_mutation_marker = 1\n"] + L[ln:]
            yield "insert_after_line", [owner_of_line(src, ln)], "".join(new)
        # append a statement at the end of the function body (body indentation)
        last = n.end_lineno
        bind = " " * n.body[-1].col_offset
        if last not in strings:
            new = L[:last] + [bind + "_mutation_tail = 1\n"] + L[last:]
            yield "append_body", [q], "".join(new)
        # decorator-only insertion before an existing def
        dec_start = min([n.lineno] + [d.lineno for d in n.decorator_list])
        new = L[:dec_start - 1] + [" " * n.col_offset + "@_mutation_decorator\n"] + L[dec_start - 1:]
        yield "add_decorator", [q], "".join(new)
    if classes:
        c, q = rng.choice(classes)
        mind = " " * (c.col_offset + 4)
        last = c.end_lineno
        for deco in (False, True):
            add = ([mind + "@property\n"] if deco else []) + [mind + "def _mutation_new(self):\n", mind + "    return 1\n"]
            # after the class's last line, then also after one trailing blank line if present
            new = L[:last] + ["\n"] + add + L[last:]
            yield f"new_method{'_decorated' if deco else ''}", [q], "".join(new)
            if last < len(L) and not L[last].strip():
                new = L[:last + 1] + add + ["\n"] + L[last + 1:]
                yield f"new_method_after_blank{'_decorated' if deco else ''}", [q], "".join(new)
    top = [n for n, q in D if "." not in q]
    if top:
        n = rng.choice(top)
        s = min([n.lineno] + [d.lineno for d in n.decorator_list])
        for deco in (False, True):
            add = (["@_mutation_decorator\n"] if deco else []) + ["def _mutation_top():\n", "    return 1\n", "\n", "\n"]
            new = L[:s - 1] + add + L[s - 1:]
            yield f"new_toplevel{'_decorated' if deco else ''}", ["<module>"], "".join(new)
    # blank-line-only insertion outside strings: expect no gold
    cand = [ln for ln in range(1, len(L)) if ln not in strings and (ln + 1) not in strings]
    if cand:
        ln = rng.choice(cand)
        yield "blank_insert", [], "".join(L[:ln] + ["\n"] + L[ln:])


rows = []
for t in TASKS:
    prefix = PKG[t["repo"]]
    with tarfile.open(SNAP / f"{t['instance_id']}.tgz") as tf:
        mem = [m for m in tf.getmembers() if m.isfile() and m.name.endswith(".py")]
        lib = [m for m in mem if m.name.lstrip("./").startswith(prefix) and "/tests/" not in m.name]
        for m in rng.sample(lib, min(2, len(lib))):
            path = m.name.lstrip("./")
            src = tf.extractfile(m).read().decode("utf-8", errors="replace")
            try:
                ast.parse(src)
            except SyntaxError:
                continue
            for kind, exp, new in mutations(path, src):
                try:
                    got = run(path, src, new)
                    err = ""
                except Exception as e:  # noqa: BLE001
                    got, err = None, f"{type(e).__name__}: {e}"[:200]
                rows.append(dict(instance_id=t["instance_id"], file=path, kind=kind, expected="|".join(exp),
                                 got="" if got is None else "|".join(got), ok=(got == sorted(exp)), error=err))

with open(OUT / "mutation_results.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
from collections import Counter
tot, ok = Counter(r["kind"] for r in rows), Counter(r["kind"] for r in rows if r["ok"])
for k in sorted(tot):
    print(f"{k:32s} {ok[k]:4d}/{tot[k]:4d}")
print("ALL", sum(ok.values()), "/", sum(tot.values()))


In [ ]:
%%writefile {W}/kaggle/skill_eval/skill_eval.py
"""skill offline eval — generated: skill/localize-code/scripts/localize.py + kaggle/skill_eval/main_part.py"""
"""Rank the functions/classes of a Python repository by relevance to an issue text (BM25 over AST spans).

Pure standard library (Python >= 3.8). Read-only: never writes inside the repository.
Usage:
    python3 localize.py "issue text ..."                 # query from arguments
    python3 localize.py < issue.txt                      # query from stdin
    python3 localize.py --root /workspace --top 15 --files 5 --max-chars 4000 "..."
    run_skill_script(skill_name="localize-code", file_path="scripts/localize.py", args={"query": "issue text"})
Repository root: --root if it is a directory, else the first git checkout among /workspace (docker sandbox), $PWD and
$PYTHONPATH (subprocess sandbox: the wrapper runs from a temporary directory, and "/workspace" is rewritten only in
shell commands, not inside scripts), else the first of those that exists, else the current directory.
Output (plain text, < 4,000 characters): top files, then top symbols as `path:line  qualified.name`.
"""
import ast
import collections
import math
import os
import re
import sys

SKIP_DIRS = {".git", ".hg", ".svn", "node_modules", "__pycache__", ".venv", "venv", "env", "build", "dist",
             ".tox", ".nox", ".mypy_cache", ".pytest_cache", "site-packages", "docs", "doc", "docs_src",
             "examples", "tests", "test", "testing", "benchmarks"}
TEST_FILE = re.compile(r"(^test_.*|.*_test)\.py$")
TOKEN = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
CAMEL = re.compile(r"_|(?<=[a-z0-9])(?=[A-Z])")
STOP = set("""the a an and or of to in for is are be this that with on as it by not if from at we can should when use
using was were has have had but so do does did into than then there their them they you your our its also only
self none true false return def class import""".split())
MAX_FILE_BYTES = 1_000_000


def tokens(text):
    out = []
    for word in TOKEN.findall(text):
        parts = [p for p in CAMEL.split(word) if p]
        for t in [word] + (parts if len(parts) > 1 else []):
            t = t.lower()
            if len(t) > 1 and t not in STOP:
                out.append(t)
    return out


def python_files(root):
    for d, dirs, files in os.walk(root):
        dirs[:] = sorted(x for x in dirs if x not in SKIP_DIRS and not x.startswith("."))
        for f in sorted(files):
            if f.endswith(".py") and not TEST_FILE.match(f):
                p = os.path.join(d, f)
                try:
                    if os.path.getsize(p) <= MAX_FILE_BYTES:
                        yield p
                except OSError:
                    pass


def units(root):
    """Yield (relative path, line, qualified name, text) for every def/class (decorators included) and each file."""
    for path in python_files(root):
        rel = os.path.relpath(path, root)
        try:
            with open(path, encoding="utf-8", errors="replace") as fh:
                src = fh.read()
            tree = ast.parse(src)
        except (SyntaxError, ValueError, OSError):
            continue
        lines = src.splitlines()
        yield rel, 1, None, src
        stack = [(tree, "")]
        while stack:
            node, prefix = stack.pop()
            for ch in ast.iter_child_nodes(node):
                if isinstance(ch, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                    q = f"{prefix}.{ch.name}" if prefix else ch.name
                    start = min([ch.lineno] + [d.lineno for d in ch.decorator_list])
                    end = getattr(ch, "end_lineno", None) or start
                    yield rel, ch.lineno, q, "\n".join(lines[start - 1:end])
                    stack.append((ch, q))
                else:
                    stack.append((ch, prefix))


def bm25(query_terms, docs, k1=1.2, b=0.75):
    tfs = [collections.Counter(tokens(d)) for d in docs]
    n = len(tfs) or 1
    avg = sum(sum(t.values()) for t in tfs) / n or 1.0
    df = collections.Counter(w for t in tfs for w in t)
    scores = []
    for t in tfs:
        dl = sum(t.values())
        s = 0.0
        for w in query_terms:
            if w in t:
                idf = math.log(1 + (n - df[w] + 0.5) / (df[w] + 0.5))
                s += idf * t[w] * (k1 + 1) / (t[w] + k1 * (1 - b + b * dl / avg))
        scores.append(s)
    return scores


def find_root(given):
    if given and os.path.isdir(given):
        return given
    cands = ["/workspace", os.environ.get("PWD", "")] + os.environ.get("PYTHONPATH", "").split(os.pathsep)
    for c in cands:
        if c and os.path.isdir(os.path.join(c, ".git")):
            return c
    return next((c for c in cands if c and os.path.isdir(c)), os.getcwd())


def main(argv):
    root, top, top_files, max_chars, args = None, 15, 5, 4000, []
    it = iter(argv)
    for a in it:
        if a == "--root":
            root = next(it)
        elif a == "--top":
            top = int(next(it))
        elif a == "--files":
            top_files = int(next(it))
        elif a == "--max-chars":
            max_chars = int(next(it))
        elif a == "--query":
            args.append(next(it))
        elif a == "--":  # run_skill_script puts '--' before positional_args
            continue
        else:
            args.append(a)
    query = " ".join(args).strip() or sys.stdin.read()
    q = set(tokens(query))
    if not q:
        print("localize: empty query")
        return 1
    root = find_root(root)
    files, syms = [], []
    for rel, line, qual, text in units(root):
        (files if qual is None else syms).append((rel, line, qual, text))
    if not syms:
        print(f"localize: no parsable Python files under {root}")
        return 1
    fs = bm25(q, [t for *_, t in files])
    ss = bm25(q, [t for *_, t in syms])
    file_rank = sorted(range(len(files)), key=lambda i: (-fs[i], files[i][0]))[:top_files]
    seen, sym_rank = set(), []
    for i in sorted(range(len(syms)), key=lambda i: (-ss[i], syms[i][0], syms[i][1])):
        key = (syms[i][0], syms[i][2])
        if key not in seen and ss[i] > 0:
            seen.add(key)
            sym_rank.append(i)
        if len(sym_rank) >= top:
            break
    out = [f"Likely files (BM25 over {len(files)} non-test files):"]
    out += [f"  {files[i][0]}" for i in file_rank if fs[i] > 0]
    out.append(f"Likely functions/classes (of {len(syms)}):")
    out += [f"  {syms[i][0]}:{syms[i][1]}  {syms[i][2]}" for i in sym_rank]
    print("\n".join(out)[:max_chars])
    return 0


if False:
    sys.exit(main(sys.argv[1:]))


# ===== Offline evaluation of the localize-code skill script (exact same code as skill/localize-code/scripts/localize.py) =====
import csv, glob, io, json, shutil, tarfile, time, contextlib
from pathlib import Path

ROOT_IN = Path(sorted(glob.glob("/kaggle/input/**/gemma-4-developer-agent/tasks.jsonl", recursive=True))[0]).parent
OUT = Path("/kaggle/working")
TASKS = [json.loads(l) for l in open(ROOT_IN / "tasks.jsonl")]
rows, timing = [], []
for t in TASKS:
    repo = Path("/tmp/repo")
    shutil.rmtree(repo, ignore_errors=True)
    repo.mkdir()
    with tarfile.open(ROOT_IN / "snapshots" / f"{t['instance_id']}.tgz") as tf:
        tf.extractall(repo, filter="data")
    heads = sorted(repo.rglob(".git/HEAD"), key=lambda p: len(p.parts))
    root = heads[0].parent.parent if heads else repo
    buf = io.StringIO()
    t0 = time.time()
    with contextlib.redirect_stdout(buf):
        main(["--root", str(root), "--top", "100", "--files", "20", "--max-chars", "1000000", t["problem_statement"]])
    dt = time.time() - t0
    text = buf.getvalue()
    timing.append(dict(instance_id=t["instance_id"], seconds=round(dt, 2), output_chars=len(text)))
    section, rank_f, rank_s = None, 0, 0
    for line in text.splitlines():
        if line.startswith("Likely files"):
            section = "file"; continue
        if line.startswith("Likely functions"):
            section = "symbol"; continue
        s = line.strip()
        if section == "file" and s:
            rank_f += 1
            rows.append(dict(instance_id=t["instance_id"], granularity="file", rank=rank_f, path=s, line="", qualname=""))
        elif section == "symbol" and s:
            rank_s += 1
            loc, qual = s.split("  ", 1)
            path, ln = loc.rsplit(":", 1)
            rows.append(dict(instance_id=t["instance_id"], granularity="symbol", rank=rank_s, path=path, line=ln, qualname=qual.strip()))
    print(t["instance_id"], round(dt, 2), flush=True)
for name, data in (("skill_ranks.csv", rows), ("skill_timing.csv", timing)):
    with open(OUT / name, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(data[0].keys())); w.writeheader(); w.writerows(data)
print("rows", len(rows), "max seconds", max(x["seconds"] for x in timing), "max chars", max(x["output_chars"] for x in timing))


In [ ]:
%%writefile {W}/results/c3_run_rows.csv
task,arm,rep,resolved,overflow,timeout,tool_budget,other_error,empty_patch,seconds,tool_calls,has_trace,lost_edit,first_open_step,first_see_lines_step,edit_arg_errors,edit_succeeded,max_prompt,max_step_output,second_step_output
fastapi_14266,base,1,False,False,False,True,False,True,563.8,100,True,False,31,32,0,False,16011,745,628
fastapi_14266,out8k,1,False,False,False,True,False,False,638.7,100,True,False,28,29,0,True,14841,2285,909
fastapi_15588,out8k,1,False,False,False,False,False,False,214.7,17,True,False,4,4,0,True,16636,1178,541
fastapi_15588,base,1,False,False,False,False,False,False,166.2,16,True,False,3,3,0,True,14425,1064,579
fastapi_14458,base,1,True,False,False,False,False,False,219.9,13,True,False,2,2,0,True,14438,3390,889
fastapi_14458,out8k,1,True,False,False,False,False,False,229.7,11,True,False,2,2,0,True,15973,3059,1218
fastapi_14459,out8k,1,False,False,False,False,False,False,472.0,46,True,False,7,10,0,True,15986,3304,1981
fastapi_14459,base,1,False,True,False,False,False,True,630.6,46,True,True,6,6,0,True,15761,3221,1787
rich_3278,base,1,False,False,True,False,False,False,1808.9,10,True,False,6,6,25,True,15304,4189,4189
rich_3278,out8k,1,True,False,False,False,False,False,273.7,15,True,False,7,7,8,True,16466,3367,2681
fastapi_14186,out8k,1,False,False,False,False,False,False,470.1,11,True,False,2,2,0,True,15130,5099,3250
fastapi_14186,base,1,False,True,False,False,False,True,449.4,21,True,True,2,2,0,True,14682,2608,2555
rich_3506,base,1,False,True,False,False,False,True,235.2,18,True,False,5,6,0,True,13231,4144,2218
rich_3506,out8k,1,False,False,False,False,False,False,607.4,47,True,False,2,2,0,True,16580,4139,2721
fastapi_13207,out8k,1,False,False,False,False,False,False,561.4,49,True,False,8,9,9,True,15163,1729,1410
fastapi_13207,base,1,False,True,False,False,False,True,554.2,64,True,True,22,,1,True,15165,3391,1541
fastapi_14301,base,1,True,False,False,False,False,False,102.4,14,True,False,2,2,1,True,13374,787,502
fastapi_14301,out8k,1,True,False,False,False,False,False,184.0,21,True,False,2,2,0,True,14469,1713,884
fastapi_14371,out8k,1,False,False,False,False,False,False,728.4,78,True,False,4,4,0,True,15066,1784,1430
fastapi_14371,base,1,False,False,False,False,False,False,228.0,24,True,False,2,3,0,True,15581,1684,448
rich_3064,base,1,False,False,False,False,False,False,172.1,13,True,False,2,4,0,True,14348,1375,937
rich_3064,out8k,1,False,False,False,False,False,False,176.8,26,True,False,2,4,0,True,14742,1315,746
rich_3454,out8k,1,True,False,False,False,False,False,137.9,15,True,False,2,2,6,True,14411,675,462
rich_3454,base,1,True,False,False,False,False,False,97.8,10,True,False,2,2,0,True,14782,553,487
fastapi_14419,base,1,False,False,False,False,False,False,1004.4,63,True,False,4,4,0,True,15688,4951,4136
fastapi_14419,out8k,1,True,False,False,False,False,False,1020.1,67,True,False,5,5,1,True,15973,2887,2558
rich_4079,out8k,1,False,False,False,True,False,False,908.3,100,True,False,12,14,0,True,16366,2119,2034
rich_4079,base,1,False,False,False,False,False,False,486.7,71,True,False,5,9,0,True,14908,1866,1327
fastapi_14851,base,1,True,False,False,True,False,True,681.9,100,True,False,5,10,0,False,15478,1628,1089
fastapi_14851,out8k,1,True,False,False,True,False,True,604.8,100,True,False,6,10,0,False,15435,1501,695
fastapi_13537,out8k,1,True,False,False,False,False,False,162.1,11,True,False,3,6,0,True,15256,1500,1285
fastapi_13537,base,1,True,False,False,False,False,False,238.1,19,True,False,3,7,1,True,14566,1456,1332
fastapi_14361,base,1,False,False,False,False,False,False,331.9,19,True,False,2,4,0,True,15729,3377,1542
fastapi_14361,out8k,1,False,False,False,False,False,False,185.2,18,True,False,3,5,0,True,14351,2009,994
fastapi_12942,out8k,1,False,False,False,False,False,False,433.4,75,True,False,24,28,0,True,14825,1075,1066
fastapi_12942,base,1,False,False,False,False,False,False,325.2,37,True,False,14,20,0,True,14556,1662,986
requests_7505,base,1,False,False,False,False,False,False,590.8,56,True,False,2,2,0,True,15046,2446,1428
requests_7505,out8k,1,False,False,False,False,False,False,483.5,47,True,False,2,2,0,True,16239,2769,2619
rich_3944,out8k,1,False,False,False,False,False,False,193.3,21,True,False,,,0,True,15006,913,681
rich_3944,base,1,False,True,False,False,False,True,72.4,7,True,False,,,0,False,13676,746,614
fastapi_14873,base,1,True,False,False,False,False,False,200.9,21,True,False,2,5,0,True,14377,2306,1015
fastapi_14873,out8k,1,True,False,False,False,False,False,306.0,45,True,False,2,5,0,True,15038,1914,900
fastapi_15030,out8k,1,False,False,False,False,False,False,423.4,44,True,False,4,4,6,True,16018,1632,1452
fastapi_15030,base,1,False,False,False,False,False,False,257.3,20,True,False,3,3,0,True,15243,2221,770
requests_7427,base,1,True,False,False,False,False,False,343.0,30,True,False,2,4,0,True,14525,3090,1660
requests_7427,out8k,1,True,False,False,False,False,False,344.4,24,True,False,2,5,0,True,15613,4395,3248
rich_3718,out8k,1,True,False,False,False,False,False,132.3,16,True,False,4,4,0,True,14557,1526,499
rich_3718,base,1,True,False,False,False,False,False,120.8,15,True,False,3,3,0,True,12649,1596,522
rich_3061,base,1,False,False,False,False,False,False,581.6,71,True,False,6,8,0,True,15315,2183,1524
rich_3061,out8k,1,False,False,False,False,False,False,480.4,39,True,False,3,3,0,True,14652,2968,1731
fastapi_5624,out8k,1,False,False,False,False,False,False,263.2,25,True,False,9,10,0,True,14715,1431,973
fastapi_5624,base,1,False,True,False,False,False,True,133.0,19,True,False,3,11,0,False,15205,1226,596
requests_7328,base,1,False,False,False,False,False,False,1188.3,93,True,False,1,2,12,True,15599,3062,2845
requests_7328,out8k,1,False,False,False,False,False,False,333.1,13,True,False,1,2,0,True,18297,4135,1571
rich_4076,out8k,1,False,False,False,True,False,False,513.9,100,True,False,,,0,True,14460,1834,1474
rich_4076,base,1,False,False,False,False,False,False,247.2,27,True,False,,,3,True,14617,1176,997
rich_3942,base,1,False,True,False,False,False,True,70.5,8,True,False,2,2,0,False,14152,858,667
rich_3942,out8k,1,False,False,False,False,False,False,556.5,99,True,False,2,2,0,True,15385,1946,1058
fastapi_15745,out8k,1,False,False,False,False,False,False,1344.1,98,True,False,3,3,0,True,17325,4291,4126
fastapi_15745,base,1,False,True,False,True,False,True,1446.7,100,True,True,3,3,0,True,15966,4358,4136
rich_3777,base,1,True,False,False,False,False,False,367.3,48,True,False,2,2,0,True,15372,1285,928
rich_3777,out8k,1,True,False,False,False,False,False,729.7,97,True,False,2,2,0,True,16009,1639,1180
rich_3067,out8k,1,False,False,False,False,False,False,383.8,57,True,False,,,0,True,16619,1195,687
rich_3067,base,1,True,False,False,False,False,False,117.8,18,True,False,4,4,1,True,14192,706,296
fastapi_15800,base,1,False,False,False,False,False,False,334.0,36,True,False,3,3,0,True,14984,1389,1017
fastapi_15800,out8k,1,False,False,False,False,False,False,196.9,31,True,False,3,3,0,True,14827,720,673
fastapi_14978,out8k,1,False,False,False,False,False,False,373.0,56,True,False,18,23,2,True,14877,1817,610
fastapi_14978,base,1,False,False,False,False,False,False,399.4,43,True,False,10,11,0,True,15029,2376,1753
fastapi_14605,base,1,False,False,False,False,False,False,351.1,49,True,False,2,4,0,True,14831,4161,663
fastapi_14605,out8k,1,False,False,True,False,False,False,1810.9,16,True,False,2,3,26,True,15014,4172,4172
fastapi_14964,out8k,1,False,False,False,False,False,False,149.4,18,True,False,4,4,0,True,14496,991,694
fastapi_14964,base,1,False,False,False,False,False,False,169.5,12,True,False,4,4,6,True,13879,634,595
rich_3930,base,1,False,False,False,False,False,False,1269.2,33,True,False,2,2,0,True,15215,3273,2782
rich_3930,out8k,1,False,False,True,False,False,False,1809.1,61,True,False,4,4,0,True,15977,3634,2950
fastapi_14262,out8k,1,False,False,False,True,False,False,979.4,100,True,False,3,3,1,True,20289,5402,2428
fastapi_14262,base,1,False,False,False,False,False,False,750.0,73,True,False,4,4,1,True,15649,4107,1692
fastapi_5077,base,1,True,False,False,False,False,False,216.3,20,True,False,1,3,0,True,14796,2257,1836
fastapi_5077,out8k,1,True,False,False,False,False,False,188.9,15,True,False,1,4,0,True,14757,1257,1144
fastapi_14360,out8k,1,False,False,False,False,False,False,346.6,27,True,False,4,6,1,True,14808,2214,1545
fastapi_14360,base,1,False,True,False,False,False,True,965.4,56,True,True,8,15,0,True,15497,5128,2629
rich_4077,base,1,False,False,False,False,False,False,309.0,49,True,False,26,26,0,True,15361,1011,963
rich_4077,out8k,1,False,False,False,False,False,False,178.0,35,True,False,,,0,True,14965,767,593
requests_7502,out8k,1,False,False,False,False,False,False,235.6,39,True,False,2,,0,True,15360,1057,724
requests_7502,base,1,False,False,False,False,False,False,152.6,22,True,False,2,4,0,True,14836,553,469
rich_3105,base,1,False,True,False,False,False,True,295.6,15,True,False,,,0,False,15834,4306,4122
rich_3105,out8k,1,False,False,True,False,False,True,1804.2,79,True,False,,,0,False,20859,8192,4412
rich_3675,out8k,1,False,False,False,False,False,False,167.4,25,True,False,3,,0,True,14396,913,725
rich_3675,base,1,False,False,False,False,False,False,94.2,13,True,False,2,4,0,True,11895,947,589
fastapi_14512,base,1,False,False,False,False,False,False,557.6,67,True,False,22,22,0,True,14689,2511,1516
fastapi_14512,out8k,1,False,False,False,True,False,False,977.2,100,True,False,20,20,3,True,16822,4179,2347
fastapi_14349,out8k,1,False,False,False,False,False,False,206.5,25,True,False,7,7,0,True,14918,2617,425
fastapi_14349,base,1,False,False,False,False,False,False,164.8,16,True,False,2,3,0,True,14868,1644,1001
fastapi_13786,base,1,False,False,False,False,False,False,778.9,63,True,False,3,3,0,True,16174,4779,1649
fastapi_13786,out8k,1,False,False,False,False,False,False,750.7,58,True,False,3,3,0,True,16215,3840,2226
requests_6644,out8k,1,False,False,False,False,False,False,174.3,20,True,False,1,4,0,True,15143,955,874
requests_6644,base,1,False,False,False,False,False,False,207.6,34,True,False,,,0,True,14529,794,781
rich_3935,base,1,False,False,False,False,False,False,577.6,65,True,False,,,0,True,15389,3022,2205
rich_3935,out8k,1,False,False,False,False,False,False,808.3,92,True,False,,,0,True,15358,1729,1623
fastapi_14356,out8k,1,False,False,False,False,False,False,298.1,33,True,False,5,12,0,True,14474,984,946
fastapi_14356,base,1,False,True,False,False,False,True,164.6,12,True,False,3,9,0,False,13942,3590,549
rich_3063,base,1,False,True,False,False,False,True,261.4,15,True,False,3,3,0,True,12500,4362,2211
rich_3063,out8k,1,False,False,True,False,False,False,1809.0,84,True,False,2,2,131,True,16295,4166,4166
rich_4006,out8k,1,False,False,False,False,False,False,628.1,15,True,False,2,3,0,True,18204,4067,2186
rich_4006,base,1,False,True,False,False,False,True,132.0,14,True,True,2,4,0,True,14139,1995,1642
requests_7205,base,1,False,False,False,False,False,False,235.3,27,True,False,2,4,0,True,14744,1537,1096
requests_7205,out8k,1,False,False,False,False,False,False,91.4,15,True,False,2,4,0,True,12194,1506,272
fastapi_14485,out8k,1,False,False,False,False,False,False,690.7,71,True,False,11,18,0,True,15048,4123,2967
fastapi_14485,base,1,False,False,False,False,False,False,342.2,30,True,False,14,19,0,True,15053,5151,1311
rich_3521,base,1,False,True,False,False,False,True,355.6,23,True,False,5,6,0,False,14489,4148,3248
rich_3521,out8k,1,True,False,False,False,False,False,1372.3,68,True,False,4,4,0,True,17405,4619,4137
rich_3468,out8k,1,False,False,False,False,False,False,291.0,24,True,False,3,12,0,True,14719,3724,2208
rich_3468,base,1,False,True,False,False,False,True,285.2,31,True,True,4,18,0,True,14650,3525,2509
rich_3180,base,1,True,False,False,False,False,False,1124.5,42,True,False,5,5,0,True,15899,3989,3653
rich_3180,out8k,1,False,False,False,False,False,False,1455.5,85,True,False,8,8,0,True,17432,4700,4123
fastapi_14297,out8k,1,False,False,False,False,False,False,188.0,8,True,False,3,3,0,True,14466,3202,1326
fastapi_14297,base,1,False,False,False,False,False,False,159.1,14,True,False,3,3,0,True,14338,1297,595
fastapi_14455,base,1,False,False,False,False,False,False,389.2,47,True,False,3,4,0,True,15364,2127,2033
fastapi_14455,out8k,1,False,False,False,False,False,False,410.1,43,True,False,3,3,0,True,15741,1950,1886
fastapi_14372,out8k,1,True,False,False,False,False,False,219.4,37,True,False,4,7,0,True,14527,794,563
fastapi_14372,base,1,True,False,False,False,False,False,278.3,39,True,False,5,9,0,True,14390,1421,1027
fastapi_14077,base,1,False,True,False,False,False,True,205.1,8,True,False,5,5,18,True,13203,4182,429
fastapi_14077,out8k,1,True,False,False,False,False,False,93.3,21,True,False,4,4,0,True,13755,279,227
fastapi_14791,out8k,1,False,False,False,False,False,False,173.2,14,True,False,1,1,0,True,14480,1338,706
fastapi_14791,base,1,False,False,False,False,False,False,145.2,16,True,False,1,1,0,True,14440,848,681
fastapi_14463,base,1,True,False,False,False,False,False,116.6,14,True,False,3,3,0,True,14396,604,533
fastapi_14463,out8k,1,True,False,False,False,False,False,156.4,10,True,False,3,3,0,True,15355,1472,870
rich_3486,out8k,1,False,False,False,False,False,False,1093.5,96,True,False,2,2,0,True,17850,4246,4127
rich_3486,base,1,False,False,False,True,False,False,885.0,100,True,False,2,2,0,True,16044,3837,1604
httpx_3672,base,1,False,True,False,False,False,True,103.3,7,True,False,1,3,0,False,13554,1827,789
httpx_3672,out8k,1,False,False,False,False,False,False,528.4,68,True,False,1,3,0,True,15952,2040,2000
fastapi_11194,out8k,1,True,False,False,False,False,False,207.6,29,True,False,9,17,0,True,14380,1980,1030
fastapi_11194,base,1,True,False,False,False,False,False,343.0,46,True,False,20,26,1,True,15207,1209,1041
rich_3938,base,1,False,False,False,False,False,False,806.8,75,True,False,2,25,0,True,15293,4125,2955
rich_3938,out8k,1,False,False,False,False,False,False,985.9,97,True,False,2,17,0,True,16910,2738,2560
fastapi_14246,out8k,1,False,False,False,False,False,False,782.1,78,True,False,23,37,0,True,16708,3965,1643
fastapi_14246,base,1,False,True,False,False,False,True,432.7,45,True,False,19,,0,True,14407,2666,1736
fastapi_11355,base,1,False,False,False,False,False,False,466.1,57,True,False,19,19,0,True,16161,1850,1562
fastapi_11355,out8k,1,False,False,False,False,False,False,394.8,47,True,False,7,7,0,True,14660,2314,857
rich_3043,out8k,1,True,False,False,False,False,False,227.4,14,True,False,3,3,0,True,15785,2357,1740
rich_3043,base,1,True,False,False,False,False,False,617.1,22,True,False,4,4,0,True,15815,5631,4119
fastapi_13920,base,1,False,False,False,False,False,False,342.9,20,True,False,6,6,0,True,14672,4128,2399
fastapi_13920,out8k,1,False,False,False,False,False,False,277.7,21,True,False,9,9,0,True,15015,3959,1274
rich_3953,out8k,1,False,False,False,False,False,False,225.2,16,True,False,7,7,0,True,14713,2612,1736
rich_3953,base,1,False,True,False,False,False,True,174.8,10,True,False,4,4,0,True,14240,3109,1054
requests_6629,base,1,False,False,False,False,False,False,205.8,35,True,False,1,1,0,True,15095,780,668
requests_6629,out8k,1,False,False,False,False,False,False,468.6,65,True,False,1,1,0,True,14395,2009,1896
fastapi_15785,out8k,1,False,False,False,True,False,False,825.1,100,True,False,4,,11,True,15179,1809,1219
fastapi_15785,base,1,False,False,False,True,False,False,643.5,100,True,False,3,23,0,True,15677,1166,918
rich_3905,base,1,True,False,False,False,False,False,84.3,12,True,False,2,4,1,True,12331,723,477
rich_3905,out8k,1,True,False,False,False,False,False,81.1,11,True,False,2,4,0,True,10594,1298,231
requests_7309,out8k,1,True,False,False,False,False,False,153.7,22,True,False,2,4,0,True,13347,837,730
requests_7309,base,1,True,False,False,False,False,False,164.7,21,True,False,2,4,0,True,16098,716,710
fastapi_14430,base,1,False,False,False,False,False,False,371.4,22,True,False,9,9,0,True,15426,2465,2378
fastapi_14430,out8k,1,False,False,False,False,False,False,374.2,30,True,False,8,8,0,True,14945,2901,1208
rich_3052,out8k,1,False,False,False,False,False,False,141.5,16,True,False,2,2,1,True,14107,1051,679
rich_3052,base,1,False,False,False,False,False,False,210.4,11,True,False,2,2,0,True,14529,3149,1248
rich_2943,base,1,True,False,False,False,False,False,161.3,21,True,False,6,13,0,True,15134,815,541
rich_2943,out8k,1,True,False,False,False,False,False,113.8,16,True,False,4,6,0,True,14828,516,409
fastapi_14786,out8k,1,True,False,False,False,False,False,47.7,9,True,False,1,1,0,True,8370,546,172
fastapi_14786,base,1,True,False,False,False,False,False,55.6,10,True,False,1,1,0,True,8439,610,252
fastapi_15763,base,1,False,False,False,False,False,False,446.5,87,True,False,3,3,0,True,14551,1340,696
fastapi_15763,out8k,1,False,False,False,False,False,False,223.1,29,True,False,2,2,0,True,14496,935,569
rich_3894,out8k,1,True,False,False,False,False,False,133.0,15,True,False,2,2,0,True,14738,649,585
rich_3894,base,1,True,False,False,False,False,False,126.0,11,True,False,2,2,0,True,14050,813,628
rich_3470,base,1,False,False,False,False,False,False,384.4,43,True,False,2,11,0,True,15230,1570,1521
rich_3470,out8k,1,False,False,False,True,False,False,648.5,100,True,False,1,9,0,True,14482,1691,1649
requests_7315,out8k,1,True,False,False,False,False,False,167.3,17,True,False,2,,4,True,13477,809,752
requests_7315,base,1,True,False,False,False,False,False,243.6,42,True,False,2,5,1,True,15185,873,537
fastapi_9425,base,1,False,False,False,False,False,False,774.2,92,True,False,7,11,0,True,16287,2709,2139
fastapi_9425,out8k,1,True,False,False,False,False,False,346.1,50,True,False,6,10,0,True,14577,1601,728
fastapi_14794,out8k,1,True,False,False,False,False,False,110.2,13,True,False,2,4,0,True,13597,2067,312
fastapi_14794,base,1,True,False,False,False,False,False,129.2,13,True,False,5,7,0,True,14413,1844,644
rich_3130,base,1,False,False,False,False,False,False,207.7,21,True,False,3,5,0,True,15458,1143,988
rich_3130,out8k,1,False,False,False,False,False,False,235.6,26,True,False,3,5,0,True,14487,1170,1066
fastapi_14099,out8k,1,False,False,False,False,False,False,994.9,86,True,False,2,2,4,True,17210,4117,3755
fastapi_14099,base,1,False,True,False,False,False,True,600.5,58,True,True,2,2,0,True,16334,4339,2469
rich_3469,base,1,False,True,False,False,False,True,201.4,19,True,False,5,,0,False,13212,3573,708
rich_3469,out8k,1,False,False,False,False,False,False,1238.0,98,True,False,,,0,True,16124,4445,3475
rich_3535,out8k,1,False,False,False,False,False,False,183.5,29,True,False,7,7,0,True,14551,1147,554
rich_3535,base,1,False,False,False,False,False,False,249.7,19,True,False,6,6,0,True,15124,3700,1964
fastapi_14616,base,1,False,False,False,True,False,False,904.2,100,True,False,2,2,0,True,15726,1735,1664
fastapi_14616,out8k,1,False,False,False,True,False,False,812.2,100,True,False,2,2,0,True,15577,1936,1518
fastapi_14306,out8k,1,False,False,False,False,False,False,690.8,67,True,False,2,2,5,True,15375,1694,1612
fastapi_14306,base,1,False,False,False,False,False,False,490.7,81,True,False,2,2,0,True,16190,2181,517
rich_2725,base,1,True,False,False,False,False,False,163.9,20,True,False,3,13,0,True,14968,1536,1152
rich_2725,out8k,1,True,False,False,False,False,False,120.5,18,True,False,3,12,0,True,14514,959,539
rich_3296,out8k,1,False,False,False,False,False,False,156.8,19,True,False,3,8,0,True,14696,1477,505
rich_3296,base,1,False,False,False,False,False,False,179.1,18,True,False,2,7,0,True,15126,1093,826
requests_6757,base,1,False,False,False,False,False,False,139.9,9,True,False,,,0,True,11072,1673,1247
requests_6757,out8k,1,False,False,False,False,False,False,140.9,8,True,False,,,0,True,10986,1540,1052
fastapi_14479,out8k,1,False,False,False,False,False,False,78.1,12,True,False,1,3,0,True,11521,712,384
fastapi_14479,base,1,False,False,False,False,False,False,103.2,15,True,False,1,3,0,True,12292,572,545
rich_3006,base,1,True,False,False,False,False,False,121.9,25,True,False,6,7,0,True,12628,675,475
rich_3006,out8k,1,True,False,False,False,False,False,63.8,15,True,False,6,6,0,True,9204,491,253
fastapi_14258,out8k,1,False,False,False,False,False,False,453.4,44,True,False,2,7,0,True,14387,399,290
fastapi_14258,base,1,False,False,False,False,False,False,475.5,42,True,False,3,8,0,True,14392,595,537
rich_3472,base,1,False,False,False,True,False,True,477.4,100,True,False,2,4,0,False,15150,1819,658
rich_3472,out8k,1,False,False,False,True,False,False,624.3,100,True,False,2,,0,True,15137,1564,1556
rich_4075,out8k,1,True,False,False,False,False,False,288.6,38,True,False,3,4,0,True,14748,1856,1389
rich_4075,base,1,False,False,False,True,False,False,650.1,100,True,False,1,2,6,True,15470,3326,1738
fastapi_14583,base,1,False,True,False,False,False,True,91.4,8,True,False,,,0,False,13788,1467,464
fastapi_14583,out8k,1,False,False,False,False,False,False,292.7,33,True,False,,,0,True,15013,1372,969
fastapi_14303,out8k,1,False,False,False,False,False,False,140.6,15,True,False,3,3,0,True,13206,1827,794
fastapi_14303,base,1,False,False,False,False,False,False,101.5,12,True,False,3,3,0,True,11283,620,582
requests_6589,base,1,False,False,False,False,False,False,220.4,30,True,False,2,2,0,True,15670,1025,845
requests_6589,out8k,1,False,False,False,False,False,False,161.1,19,True,False,2,2,0,True,14708,832,634
fastapi_14448,out8k,1,False,False,False,False,False,False,276.8,12,True,False,3,3,0,True,15066,3590,1199
fastapi_14448,base,1,False,True,False,False,False,True,181.2,8,True,False,3,3,0,False,13688,4135,2047
requests_7433,base,1,False,False,False,True,False,False,788.8,100,True,False,2,4,10,True,15291,1168,1062
requests_7433,out8k,1,False,False,False,True,False,False,763.4,100,True,False,2,4,0,True,14955,1911,1560
fastapi_14953,out8k,1,False,False,False,False,False,False,237.2,27,True,False,2,2,0,True,15819,1056,778
fastapi_14953,base,1,False,False,False,False,False,False,106.1,9,True,False,2,2,0,True,14346,722,372
fastapi_9555,base,1,True,False,False,False,False,False,231.4,22,True,False,12,12,0,True,15821,1454,706
fastapi_9555,out8k,1,True,False,False,False,False,False,143.3,11,True,False,6,6,0,True,16460,1345,684
fastapi_14609,out8k,1,False,False,False,True,False,False,973.9,100,True,False,4,4,0,True,15594,2157,1753
fastapi_14609,base,1,False,True,False,False,False,True,853.4,77,True,True,5,5,27,True,15361,4196,1454
rich_3772,base,1,False,False,False,False,False,False,646.7,19,True,False,2,2,0,True,15533,2637,2186
rich_3772,out8k,1,False,False,False,False,False,False,633.5,21,True,False,2,2,0,True,16347,3443,1340
rich_3518,out8k,1,True,False,False,False,False,False,123.4,16,True,False,2,6,0,True,16163,857,408
rich_3518,base,1,True,False,False,False,False,False,154.4,23,True,False,2,6,0,True,15221,1108,563
fastapi_15280,base,1,False,False,False,False,False,False,608.2,69,True,False,36,37,0,True,15993,7197,544
fastapi_15280,out8k,1,False,False,False,False,False,False,757.5,67,True,False,3,3,0,True,16166,3964,3573
fastapi_9753,out8k,1,True,False,False,False,False,False,285.3,34,True,False,22,22,0,True,15317,3623,936
fastapi_9753,base,1,True,False,False,False,False,False,204.3,14,True,False,8,8,1,True,14350,1585,1291
requests_6592,base,1,False,False,False,False,False,False,59.4,7,True,False,1,1,0,True,8202,525,466
requests_6592,out8k,1,False,False,False,False,False,False,66.7,9,True,False,1,1,0,True,13219,435,429
fastapi_14482,out8k,1,False,False,False,False,False,False,269.2,29,True,False,17,17,0,True,15147,2498,1042
fastapi_14482,base,1,False,True,False,False,False,True,362.3,35,True,True,6,6,0,True,15696,1536,1250
fastapi_13713,base,1,False,False,False,False,False,False,238.2,32,True,False,4,7,0,True,14548,1000,784
fastapi_13713,out8k,1,False,False,False,False,False,False,1086.1,67,True,False,2,10,31,True,16372,4389,4386
fastapi_15589,out8k,1,True,False,False,False,False,False,840.5,73,True,False,6,9,0,True,16006,4062,3337
fastapi_15589,base,1,False,False,False,False,False,False,368.2,38,True,False,8,,0,True,14384,2623,1414
rich_3934,base,1,False,True,False,False,False,True,479.6,19,True,False,6,7,0,True,14850,4905,4391
rich_3934,out8k,1,False,False,False,False,False,False,970.7,68,True,False,2,3,1,True,17272,4344,4344
rich_3480,out8k,1,True,False,False,False,False,False,1095.2,27,True,False,3,5,0,True,15016,1249,990
rich_3480,base,1,True,False,False,False,False,False,1037.3,24,True,False,2,4,0,True,13438,520,500
fastapi_14986,base,1,False,False,False,False,False,False,192.1,30,True,False,2,6,0,True,14756,1640,520
fastapi_14986,out8k,1,False,False,False,False,False,False,224.9,28,True,False,2,7,0,True,14582,1645,1124
rich_3782,out8k,1,False,False,False,False,False,False,194.9,18,True,False,2,2,0,True,16484,873,812
rich_3782,base,1,False,False,False,False,False,False,409.1,51,True,False,2,3,0,True,15440,1562,976
fastapi_14266,out8k,2,False,False,False,True,False,False,602.1,100,False,,,,,,,,
fastapi_14266,base,2,False,True,False,False,False,True,892.0,57,False,,,,,,,,
fastapi_15588,base,2,False,False,False,False,False,False,220.7,24,False,,,,,,,,
fastapi_15588,out8k,2,False,False,False,False,False,False,401.2,26,False,,,,,,,,
fastapi_14458,out8k,2,True,False,False,False,False,False,127.1,10,False,,,,,,,,
fastapi_14458,base,2,True,False,False,False,False,False,171.9,12,False,,,,,,,,
fastapi_14459,base,2,False,False,False,False,False,False,615.2,21,False,,,,,,,,
fastapi_14459,out8k,2,False,False,False,False,False,False,1251.9,83,False,,,,,,,,
rich_3278,out8k,2,False,False,False,False,False,False,646.5,44,False,,,,,,,,
rich_3278,base,2,True,False,False,False,False,False,309.4,33,False,,,,,,,,
fastapi_14186,base,2,False,False,False,False,False,False,413.4,16,False,,,,,,,,
fastapi_14186,out8k,2,False,False,False,False,False,False,449.4,20,False,,,,,,,,
rich_3506,out8k,2,False,False,False,False,False,False,544.2,42,False,,,,,,,,
rich_3506,base,2,True,False,False,False,False,False,195.2,12,False,,,,,,,,
fastapi_13207,base,2,False,False,False,False,False,False,343.1,38,False,,,,,,,,
fastapi_13207,out8k,2,False,False,False,False,False,False,314.3,34,False,,,,,,,,
fastapi_14301,out8k,2,True,False,False,False,False,False,137.6,21,False,,,,,,,,
fastapi_14301,base,2,True,False,False,False,False,False,634.3,74,False,,,,,,,,
fastapi_14371,base,2,False,False,False,False,False,False,338.9,41,False,,,,,,,,
fastapi_14371,out8k,2,False,False,False,False,False,False,351.7,29,False,,,,,,,,
rich_3064,out8k,2,False,False,False,False,False,False,204.8,14,False,,,,,,,,
rich_3064,base,2,False,False,False,False,False,False,460.8,57,False,,,,,,,,
rich_3454,base,2,True,False,False,False,False,False,88.3,12,False,,,,,,,,
rich_3454,out8k,2,True,False,False,False,False,False,110.1,14,False,,,,,,,,
fastapi_14419,out8k,2,False,False,False,False,False,False,1124.7,75,False,,,,,,,,
fastapi_14419,base,2,False,False,False,False,False,False,1627.5,99,False,,,,,,,,
rich_4079,base,2,False,True,False,False,False,True,1357.5,98,False,,,,,,,,
rich_4079,out8k,2,False,False,False,True,False,False,1254.5,100,False,,,,,,,,
fastapi_14851,out8k,2,True,False,False,True,False,True,629.1,100,False,,,,,,,,
fastapi_14851,base,2,True,False,False,True,False,True,555.7,100,False,,,,,,,,
fastapi_13537,base,2,True,False,False,False,False,False,194.7,17,False,,,,,,,,
fastapi_13537,out8k,2,True,False,False,False,False,False,192.5,19,False,,,,,,,,
fastapi_14361,out8k,2,False,False,False,False,False,False,395.2,24,False,,,,,,,,
fastapi_14361,base,2,False,False,False,False,False,False,434.3,47,False,,,,,,,,
fastapi_12942,base,2,False,False,False,False,False,False,248.6,35,False,,,,,,,,
fastapi_12942,out8k,2,False,False,False,False,False,False,415.8,48,False,,,,,,,,
requests_7505,out8k,2,False,False,False,False,False,False,289.7,23,False,,,,,,,,
requests_7505,base,2,False,False,False,False,False,False,238.4,21,False,,,,,,,,
rich_3944,base,2,False,True,False,False,False,True,508.0,35,False,,,,,,,,
rich_3944,out8k,2,False,False,False,False,False,False,450.9,26,False,,,,,,,,
fastapi_14873,out8k,2,True,False,False,True,False,False,711.6,100,False,,,,,,,,
fastapi_14873,base,2,True,False,False,False,False,False,286.5,40,False,,,,,,,,
fastapi_15030,base,2,False,False,False,False,False,False,334.3,34,False,,,,,,,,
fastapi_15030,out8k,2,False,False,False,False,False,False,212.4,22,False,,,,,,,,
requests_7427,out8k,2,True,False,False,False,False,False,279.3,23,False,,,,,,,,
requests_7427,base,2,False,True,False,False,False,True,419.0,39,False,,,,,,,,
rich_3718,base,2,True,False,False,False,False,False,118.6,18,False,,,,,,,,
rich_3718,out8k,2,True,False,False,False,False,False,109.6,13,False,,,,,,,,
rich_3061,out8k,2,False,False,False,False,False,False,268.5,46,False,,,,,,,,
rich_3061,base,2,False,False,False,False,False,False,213.9,24,False,,,,,,,,
fastapi_5624,base,2,False,False,False,True,False,False,701.2,100,False,,,,,,,,
fastapi_5624,out8k,2,True,False,False,False,False,False,197.4,20,False,,,,,,,,
requests_7328,out8k,2,False,False,False,False,False,False,259.0,14,False,,,,,,,,
requests_7328,base,2,False,True,False,False,False,True,583.2,45,False,,,,,,,,
rich_4076,base,2,False,False,False,True,False,False,1081.3,100,False,,,,,,,,
rich_4076,out8k,2,False,False,False,False,False,False,722.0,99,False,,,,,,,,
rich_3942,out8k,2,False,False,False,False,False,False,602.2,59,False,,,,,,,,
rich_3942,base,2,False,True,False,False,False,True,53.1,8,False,,,,,,,,
fastapi_15745,base,2,False,False,False,True,False,False,1260.1,100,False,,,,,,,,
fastapi_15745,out8k,2,False,False,False,True,False,False,906.6,100,False,,,,,,,,
rich_3777,out8k,2,True,False,False,False,False,False,680.3,96,False,,,,,,,,
rich_3777,base,2,True,False,False,False,False,False,661.1,97,False,,,,,,,,
rich_3067,base,2,True,False,False,False,False,False,77.1,16,False,,,,,,,,
rich_3067,out8k,2,False,False,False,False,False,False,620.8,80,False,,,,,,,,
fastapi_15800,out8k,2,False,False,False,False,False,False,268.8,39,False,,,,,,,,
fastapi_15800,base,2,False,False,True,False,False,False,1813.7,35,False,,,,,,,,
fastapi_14978,base,2,False,False,False,False,False,False,288.3,30,False,,,,,,,,
fastapi_14978,out8k,2,False,False,False,True,False,False,541.9,100,False,,,,,,,,
fastapi_14605,out8k,2,False,False,False,False,False,False,231.1,53,False,,,,,,,,
fastapi_14605,base,2,False,True,False,False,False,True,374.5,15,False,,,,,,,,
fastapi_14964,base,2,False,False,False,False,False,False,133.1,16,False,,,,,,,,
fastapi_14964,out8k,2,False,False,False,False,False,False,158.8,18,False,,,,,,,,
rich_3930,out8k,2,False,False,False,False,False,False,699.3,36,False,,,,,,,,
rich_3930,base,2,False,False,False,False,False,False,529.2,31,False,,,,,,,,
fastapi_14262,base,2,False,False,False,True,False,False,907.7,100,False,,,,,,,,
fastapi_14262,out8k,2,False,False,False,False,False,False,657.0,75,False,,,,,,,,
fastapi_5077,out8k,2,True,False,False,False,False,False,328.6,24,False,,,,,,,,
fastapi_5077,base,2,False,False,False,False,False,False,326.2,43,False,,,,,,,,
fastapi_14360,base,2,False,False,False,False,False,False,749.9,68,False,,,,,,,,
fastapi_14360,out8k,2,False,False,False,False,False,False,337.6,16,False,,,,,,,,


In [ ]:
SOURCES = {
 "analysis/gold_core.py": "2219b317ca66de2826d437645410d4a7755485dbfd23d99775cd2c8e4dfecbc2",
 "analysis/c1_report.py": "e2d2c1d8627cef8b62ea7449a941a3b2f4a1dd9fcc327544c64c2ab11478b3fa",
 "analysis/gold_heuristic.py": "bf2e019891d683917c3ed8e07d3132e322ce87300024ba33573d3f9729d8bb39",
 "analysis/embed_geometry.py": "c8dac8d55e7c0140604835ee4a5d2063af875d8814b66ac23a2c99f41929b709",
 "analysis/reach.py": "6ccf19094ebbc67c7e26cd91517b10fd01a2d12fd741f995e20ac27fa398702f",
 "analysis/reach_0211.py": "7201d9bcbf97e8e7a624cf6261e420a95f8b640de4514efdeb15eca8a64cc118",
 "analysis/skill_eval.py": "0f3d76006f481d912ea7c0041d348a30224db3e803d4d685427753bb718b73a2",
 "analysis/trace_metrics.py": "2f1fe81dce86d29c296c7a2737368f83565b2df34c3a51d062aed023f2ab31d1",
 "analysis/c3_report.py": "627ecf3f9158106f65bae2498a716dd68e64070bf0a2d4453cd9522448493227",
 "analysis/c3_fig_prompt.py": "235f2a300f0bd999eff2c253d8bc720089d8955cd231ef0b38a73a4d055987aa",
 "analysis/c3_overflow_edits.py": "b96aae82baa8fe9407e175ad6f977d7463463bd2e11c719339c80ac2314342c4",
 "kaggle/gold_ast/gold_ast.py": "a59e9de6d974905a9a3c3a95d0989aa4552765b3692343d4641277c2441bc5b4",
 "kaggle/bm25/bm25.py": "19f37abe34f81c59ab2d6b7eecd5afac07575c01ef672067c944d2c529929c29",
 "kaggle/gold_mutation/gold_mutation.py": "d07cfa6940ccf3afbfe02dc1256b4d3179dd5586beaddbc1ab87594242bb2d09",
 "kaggle/skill_eval/skill_eval.py": "e4a789f8ac62e701e267721d46b91ef47f71360d83fc5b24a34ef8aa48d6847b",
 "results/c3_run_rows.csv": "9997cd3ad03615c04e7098099217c48d31746ac197ee82b2a0ce3db79a392a99"
}
# The scripts above were written byte for byte from the repository; these digests identify them.
for rel, want in SOURCES.items():
    got = sha(WORK / rel)
    assert got == want, rel
    print(got[:16], rel)
check("swegemma wheel sha256", sha(WHEEL), "9ad876581f3acde8ec3c369858f0834e2b9f41a3cffce3ca42f09cb090aea9d3")
zipfile.ZipFile(WHEEL).extractall(WORK / "wheel_0_2_11")
check("tasks.jsonl sha256", sha(INPUT / "tasks.jsonl"), "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6")

## C1. The provided tools and data

The tool behaviour itself was read from the wheels' source (`reports/tool_mechanics.md`). Here we recount the data facts over all 127 graphs. The file listing shows the same data under 0.2.11; the all-graphs hash below confirms it byte for byte.


In [ ]:
import collections

import numpy as np
import pandas as pd

KIND = re.compile(r"^\s*(async\s+def|def|class)\b", re.M)
nodes = edges = isolated = 0
edge_types, kinds = collections.Counter(), collections.Counter()
for p in sorted(INPUT.glob("graphs/*.json")):
    g = json.load(open(p))
    ids = {n["id"] for n in g["nodes"]}
    touched = {e["source"] for e in g["edges"]} | {e["target"] for e in g["edges"]}
    nodes += len(ids)
    edges += len(g["edges"])
    isolated += len(ids - touched)
    edge_types.update(e["type"] for e in g["edges"])
    for n in g["nodes"]:
        m = KIND.search(n.get("text") or "")
        kinds[re.sub(r"\s+", " ", m.group(1)) if m else "other"] += 1
check("graph nodes", nodes, 367985)
check("graph edges", edges, 452588)
check("edge types", dict(edge_types), {"calls": 452588})
check("async function nodes", kinds["async def"], 0)
print(f"nodes without any edge: {isolated:,} = {isolated / nodes * 100:.1f}%")
print("node kinds:", dict(kinds))

### Gold fix locations and the BM25 baseline

These cells extract gold symbols from every snapshot and rank spans with BM25, then compare the output digests with those of the runs behind the paper. They take several minutes.


In [ ]:
kernel("gold_ast/gold_ast.py", "work/c1/gold_ast_v4", ["gold_ast.csv", "gold_files.csv"])
kernel("bm25/bm25.py", "work/c1/bm25_v3", ["bm25_ranks.csv"])
check("gold_ast.csv sha256", sha(WORK / "work/c1/gold_ast_v4/gold_ast.csv"), "75a03c85496548e86c8cb0acf08dcbb8b5e4b611cd6bbe84ec9e47a5a8993b23")
check("bm25_ranks.csv sha256", sha(WORK / "work/c1/bm25_v3/bm25_ranks.csv"), "00575f4d5f52fe9c04b4322a6f4c886f611d9e603b1797c5474ccdde4590cec2")
run(["analysis/gold_heuristic.py"], tail=0)
check("heuristic gold.csv sha256", sha(WORK / "work/c1/gold_coverage/gold.csv"), "2ed7d354465e5a1c4391007a42d13be3ed187fc9d058037e0b943f43d02743b6")

### Mutation test of the gold rule

Synthetic patches of known location on all 129 snapshots: the attribution rule must recover each location.


In [ ]:
kernel("gold_mutation/gold_mutation.py", "work/c1/gold_mutation", ["mutation_results.csv"], tail=1)
mut = pd.read_csv(WORK / "work/c1/gold_mutation/mutation_results.csv", keep_default_na=False)
check("mutation patches attributed correctly", (int((mut.ok.astype(str) == "True").sum()), len(mut)), (1821, 1821))

### Gold coverage, BM25 reach and the gate test

The `§3` block gives gold coverage, including the async definitions missing from the graphs and the agreement with the text heuristic. The `§4` block gives BM25 reach. The gate block tests graph expansion on top of a library-only BM25 ranking (85 of 115 at k = 50, printed as 73.9%) at equal budget.


In [ ]:
run(["analysis/c1_report.py"])
manifest = json.load(open(WORK / "reports/c1_manifest.json"))
check("hash over all 127 graphs", manifest["graphs_sha256_of_sorted_hashes"], "06d0752a022bbddeb7f1500bbff3f51867d1f18a0fbf7c2d9861ee14533584a7")

### Embedding geometry

Effective rank and first-component share, as medians over the 127 archives (first numeric column).


In [ ]:
run(["analysis/embed_geometry.py"], tail=0)
summary = (WORK / "work/c1/embed_geometry_v2/summary.md").read_text()
medians = {}
for label in ("effective rank", "PC1 variance share"):
    line = next(ln for ln in summary.splitlines() if ln.startswith(f"| {label}"))
    print(line)
    medians[label] = line.split("|")[2].strip()
check("median effective rank, PC1 share", (medians["effective rank"], medians["PC1 variance share"]), ("4.0559", "0.6308"))

### Table 1: reach from the issue text

The 0.2.11 rows call the wheel's own `resolve_node_name`, `get_neighbor` and `get_similar_nodes`.


In [ ]:
run(["analysis/reach.py"], tail=0)
run(["analysis/reach_0211.py", "--wheel-src", "wheel_0_2_11", "--out", "work/c1/reach_0211"])

In [ ]:
ids = pd.read_csv(WORK / "work/c1/reach_0211/seeds_per_task.csv").task.tolist()
old = pd.read_csv(WORK / "work/c1/reach_v3/per_task.csv")
old = old[(old.view == "e2e") & old.strategy.isin(["nb1_impl", "emb_impl"])]
new = pd.read_csv(WORK / "work/c1/reach_0211/per_task.csv")
hits = pd.read_csv(WORK / "work/c1/report/hits.csv").set_index("task").loc[ids]
KS1 = (10, 50, 100)
rows = {}
for label, strat in (("0.2.7 tools, issue identifiers as seeds (neighbours)", "nb1_impl"), ("0.2.7 tools, issue identifiers as seeds (similarity)", "emb_impl")):
    d = old[old.strategy == strat]
    rows[label] = [int(d[d.k == k].hit.sum()) for k in KS1]
for label, strat in (("0.2.11 resolver, 1-hop neighbours", "neighbors_e2e"), ("0.2.11 resolver, similarity search with lexical fallback", "similar_e2e"),
                     ("0.2.11 search_similar_code on the whole issue text (lexical fallback only)", "issue_text_query")):
    d = new[new.strategy == strat]
    rows[label] = [int(d[f"hit{k}"].sum()) for k in KS1]
rows["BM25 over AST function/class spans"] = [int(hits[f"bm25_snapshot@{k}"].sum()) for k in KS1]
table1 = pd.DataFrame(rows, index=[f"hit@{k}" for k in KS1]).T
display(table1)
check("tasks with an in-graph gold symbol", len(ids), 115)
check("0.2.7 neighbours", rows["0.2.7 tools, issue identifiers as seeds (neighbours)"], [2, 2, 2])
check("0.2.11 neighbours", rows["0.2.11 resolver, 1-hop neighbours"], [30, 35, 35])
check("0.2.11 similarity", rows["0.2.11 resolver, similarity search with lexical fallback"], [42, 55, 62])
check("0.2.11 whole issue", rows["0.2.11 search_similar_code on the whole issue text (lexical fallback only)"], [32, 64, 84])
check("BM25", rows["BM25 over AST function/class spans"], [51, 83, 98])

**Reading.** The resolver matters: a policy that reached almost nothing now reaches about a third of the tasks through neighbours, and about half through similarity search. A large part of that comes from text matching, because the whole-issue query uses no embeddings at all. Each shipped policy still trails BM25, and each paired confidence interval printed above excludes zero.

These are static policies ("start from identifiers as written in the issue"). An agent that searches first could do better, so this does not measure what an agent would reach. fastapi and rich also make up 89% of the sample (see the leave-one-repo-out lines above).


### Gate test at k = 50

Does expanding BM25's top m units along `calls` edges help at equal budget? The confidence intervals below are recomputed from the per-task table. This tests one family of expansion policies, not whether structure can help localisation in general.


In [ ]:
gate = pd.read_csv(WORK / "work/c1/report/gate.csv")
g50 = gate[gate.k == 50]
B = np.random.default_rng(0).integers(0, len(ids), size=(10_000, len(ids)))
out = []
for (m, order), d in g50.groupby(["m", "order"]):
    d = d.set_index("task").loc[ids]
    x = d.hit.astype(int).to_numpy() - d.base.astype(int).to_numpy()
    bs = x[B].mean(axis=1) * 100
    out.append(dict(m=m, order=order, diff_pp=round(x.mean() * 100, 1), ci_low=round(np.percentile(bs, 2.5), 1), ci_high=round(np.percentile(bs, 97.5), 1), gained=int((x > 0).sum()), lost=int((x < 0).sum())))
gate_table = pd.DataFrame(out)
display(gate_table)
check("gate settings at k = 50 whose CI excludes zero", int((gate_table.ci_high < 0).sum()), 6)

## C2. The offline skill evaluation

The skill script is embedded in the kernel below, unchanged from `skill/localize-code/scripts/localize.py`. It is a separate BM25 implementation from the C1 baseline. Hit rates are for gold symbols in the graph (first column) and for any gold symbol (second column). Run times are left out of this output because they vary between runs.


In [ ]:
kernel("skill_eval/skill_eval.py", "work/c2/skill_eval", ["skill_ranks.csv", "skill_timing.csv"], tail=0)
skill_out = run(["analysis/skill_eval.py"], drop=r"^runtime per task", ret=True)
for k, want in (("5", 46), ("10", 53), ("15", 59), ("50", 89), ("100", 100)):
    check(f"skill in-graph gold hit@{k}", int(re.search(rf"symbol hit@ *{k}: in-graph gold (\d+)/115", skill_out).group(1)), want)
check("skill file hit@10", int(re.search(r"file hit@10: (\d+)/115", skill_out).group(1)), 95)

## C3. Context overflow

vLLM rejects a request when prompt tokens plus `max_output_tokens` exceed 32,768. With 16,384 reserved, the prompt ceiling is 16,384, and the compaction threshold of 14,336 leaves 2,048 tokens of headroom. One large tool response can jump over it.

The first block below is Table 3 of the paper; the second block is the repetition on 40 tasks.


In [ ]:
run(["analysis/c3_report.py", "results/c3_run_rows.csv", "base", "out8k"])
c3 = pd.read_csv(WORK / "results/c3_run_rows.csv")
main = c3[c3.rep == 1]
wide = {a: main[main.arm == a].set_index("task") for a in ("base", "out8k")}
check("C3 rows in total, repetition 1", (len(c3), len(main)), (320, 240))
check("overflow runs base, out8k", (int(wide["base"].overflow.sum()), int(wide["out8k"].overflow.sum())), (26, 0))
check("resolved base, out8k", (int(wide["base"].resolved.sum()), int(wide["out8k"].resolved.sum())), (29, 34))
for col, want in (("empty_patch", (29, 2)), ("timeout", (1, 4)), ("tool_budget", (9, 12))):
    check(f"{col} base, out8k", (int(wide["base"][col].sum()), int(wide["out8k"][col].sum())), want)
check("gold file edited before overflow, base", int((wide["base"].lost_edit == True).sum()), 10)  # noqa: E712
check("runs with a first successful gold read, base, out8k", (int(wide["base"].first_open_step.notna().sum()), int(wide["out8k"].first_open_step.notna().sum())), (113, 111))
check("median first gold read, base, out8k", (float(wide["base"].first_open_step.median()), float(wide["out8k"].first_open_step.median())), (3.0, 3.0))
check("runs with gold lines displayed, base, out8k", (int(wide["base"].first_see_lines_step.notna().sum()), int(wide["out8k"].first_see_lines_step.notna().sum())), (109, 106))
over = wide["base"][wide["base"].overflow].index
o8 = wide["out8k"].loc[over]
print("out8k on the 26 tasks where base overflowed: resolved", int(o8.resolved.sum()), "| timeout", int(o8.timeout.sum()), "| tool budget", int(o8.tool_budget.sum()),
      "| failing patch", int((~o8.resolved & ~o8.timeout & ~o8.tool_budget & ~o8.empty_patch).sum()))
lost = wide["base"][wide["base"].lost_edit == True].index  # noqa: E712
print("base runs with a successful gold-file edit before overflowing:", len(lost), "| out8k resolved on those tasks:", int(wide["out8k"].loc[lost].resolved.sum()))
print("largest single-step output: base", int(wide["base"].max_step_output.max()), "| out8k", int(wide["out8k"].max_step_output.max()),
      "| next largest in out8k", int(sorted(list(wide["out8k"].max_step_output) + list(wide["out8k"].second_step_output))[-2]))
repo = main[main.arm == "base"].assign(repo=lambda d: d.task.str.split("_").str[0]).groupby("repo").overflow.agg(["sum", "count"])
print(repo.to_string())
rep2 = c3[(c3.rep == 2)].groupby("arm").overflow.sum().to_dict()
twice = set(c3[(c3.rep == 1) & (c3.arm == "base") & c3.overflow].task) & set(c3[(c3.rep == 2) & (c3.arm == "base") & c3.overflow].task)
print("repetition 2 overflows:", rep2, "| tasks overflowing in both base runs:", len(twice))

**Reading.** The fix removes the failure it targets, but the effect on resolved tasks is small. Its confidence interval includes zero, and the arms disagree with each other about as often as an arm disagrees with itself on a second run. On most tasks whose paired base run overflowed, the separate out8k run ended in a failing patch.

That is a task-level observation. It does not show whether the discarded base edits were correct, and the data cannot separate it from task difficulty.


### Figure 1

Maximum prompt tokens per run. An overflowing base run never records the rejected request, so its dot is the largest prompt that succeeded.


In [ ]:
from IPython.display import Image

fig = FINAL / "fig1_max_prompt.png"
run(["analysis/c3_fig_prompt.py", "results/c3_run_rows.csv", str(fig)], tail=3, drop="^wrote")
display(Image(filename=str(fig), width=800))

### What 0.2.11 changes

On 0.2.7 an overflowing run submits nothing. In a mock-model test of the real `run_agent_sandbox`, 0.2.11 instead submits the source edits, without test files. We did not re-run the experiment on 0.2.11, so its overflow rate and resolved count are unmeasured. The line below counts successful edit calls in the overflowing runs. Shell edits and later reverts are not tracked, so it does not measure recoverable patches, and whether any would pass is unknown.


In [ ]:
run(["analysis/c3_overflow_edits.py", "results/c3_run_rows.csv"])

In [ ]:
# Keep intermediate files outside /kaggle/working: only the figure remains as an output file.
if ON_KAGGLE:
    for p in Path("/kaggle/working").iterdir():
        if p.name != fig.name and p.suffix != ".ipynb":
            shutil.rmtree(p) if p.is_dir() else p.unlink()
    check("files left in /kaggle/working", sorted(p.name for p in Path("/kaggle/working").iterdir() if p.suffix != ".ipynb"), [fig.name])
print(len(CHECKS), "checks passed")

## Sources, licenses and limits

- **Competition data.** [Google - The Gemma 4 Developer Agent](https://www.kaggle.com/competitions/gemma-4-developer-agent): tasks, graphs, embeddings and snapshots, read in place under the competition rules. Section 4.b restricts redistribution of competition data, so the only output file is Figure 1. Aggregate results appear in the cell outputs; issue text, code and patches do not.
- **Harness wheel.** `swegemma` 0.2.11 from the [wheelhouse dataset](https://www.kaggle.com/datasets/metric/gemma-4-developer-agent-wheelhouse), checked by digest.
- **This notebook's code.** Apache 2.0.
- **Environment limits.** C3 ran on one A100 (tensor parallel 1), not the scoring setup, with a 30-minute budget. "Resolved" means the official `swegemma` verification in that environment, so official scores can differ on individual tasks.
- **Scope.** Four public repositories, with fastapi and rich dominating; the hidden test set uses private repositories. Results describe associations in this setup. They do not establish why particular runs fail.

Independent work; not affiliated with or endorsed by Google or Kaggle.

### Appendix: changes to expected values

| Check | Before | After | Reason |
|---|---|---|---|
| Gold file edited before overflow, base (no check existed; now in `c3_tables`) | 12, out8k resolved 1 on those tasks | 10, out8k resolved 0 | Count successful tool calls only, after a read-only audit |
